# ARES-X · Rover autónomo de exploración planetaria
**IA híbrida: RNA + lógica difusa + algoritmo genético, con supervisor de seguridad.**

Simulación ficticia con parámetros astronómicos de referencia. Todo valor operacional es un *supuesto de simulación*.

**Cómo usar:** `Entorno de ejecución → Ejecutar todas`. CPU gratuita, sin GPU ni claves. El panel de control (sección 5)
ocupa una sola pantalla; cada botón ejecuta el motor Python de este cuaderno.

Flujo por paso: sensores con ruido → preprocesamiento → RNA (probabilidades) → difuso (velocidad/cautela) →
AG (rutas, solo ante objetivo/bloqueo/evento grave) → supervisor de seguridad → acción → física → bitácora.

In [ ]:
# 0. Dependencias (todas gratuitas y preinstaladas en Colab)
import importlib, subprocess, sys
for pkg in ('numpy', 'pandas', 'sklearn', 'matplotlib'):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn' if pkg == 'sklearn' else pkg], check=True)
import os
for folder in ('ares_x/src/ares', 'ares_x/web', 'ares_x/tests'):
    os.makedirs(folder, exist_ok=True)
print('Python', sys.version.split()[0])

## 1. Código modular
Clases `PlanetProfile`, `World`, `SensorSuite`, `NeuralPerception`, `FuzzyController`, `GeneticPlanner`, `SafetySupervisor`, `MissionLogger`, `HybridMission` y `Dashboard`, escritas como módulos dentro del cuaderno.

In [ ]:
%%writefile ares_x/src/ares/__init__.py
"""ARES-X deterministic physical simulation, independent of AI modules."""
from .planet import PlanetProfile
from .world import World
from .sensors import SensorSuite
from .events import Event, EVENTS
from .logger import MissionLogger
from .state_machine import Mission, MissionState, StateMachine

__all__ = ['PlanetProfile', 'World', 'SensorSuite', 'Event', 'EVENTS', 'MissionLogger', 'Mission', 'MissionState', 'StateMachine']


In [ ]:
%%writefile ares_x/src/ares/planet.py
"""Planetary reference profiles and explicit vehicle compatibility."""
from dataclasses import dataclass
import unicodedata

ASSUMPTION = 'supuesto de simulación'


def normalize(value):
    return ''.join(c for c in unicodedata.normalize('NFD', value.lower()) if unicodedata.category(c) != 'Mn')


@dataclass(frozen=True)
class PlanetProfile:
    name: str
    rocky: bool
    gravity: float
    temperature: float
    pressure: float
    oxygen: float
    hazards: tuple
    mode: str = 'rover'
    operational_label: str = ASSUMPTION

    @classmethod
    def for_body(cls, name='Marte', mode='rover'):
        key = normalize(name)
        if key not in PROFILES:
            raise ValueError('Planeta desconocido')
        mode_key = normalize(mode)
        modes = {'rover': 'rover', 'ruedas': 'rover', 'sonda': 'probe', 'probe': 'probe'}
        if mode_key not in modes:
            raise ValueError('Modo inválido: use rover o sonda')
        profile = cls(*PROFILES[key], mode=modes[mode_key])
        profile.validate()
        return profile

    def validate(self):
        if self.mode not in ('rover', 'probe'):
            raise ValueError('Modo inválido: use rover o sonda')
        if not self.rocky and self.mode == 'rover':
            raise ValueError(f'{self.name} no tiene suelo sólido accesible y estable para ruedas; seleccione modo sonda atmosférica virtual.')
        return self


PROFILES = {
    'mercurio': ('Mercurio', True, 3.7, 167., 0., 0., ('radiación', 'contraste térmico')),
    'venus': ('Venus', True, 8.87, 464., 9200000., 0., ('corrosión', 'presión', 'temperatura')),
    'tierra': ('Tierra', True, 9.81, 15., 101325., 20.95, ('sismo', 'lluvia')),
    'marte': ('Marte', True, 3.71, -63., 610., .13, ('polvo', 'barranco', 'noche')),
    'jupiter': ('Júpiter', False, 24.79, -110., 100000., 0., ('turbulencia', 'presión')),
    'saturno': ('Saturno', False, 10.44, -140., 100000., 0., ('viento', 'enlace')),
    'urano': ('Urano', False, 8.69, -195., 100000., 0., ('frío', 'energía')),
    'neptuno': ('Neptuno', False, 11.15, -200., 100000., 0., ('viento', 'latencia')),
}


In [ ]:
%%writefile ares_x/src/ares/world.py
"""Seeded two-dimensional physical terrain; coordinates are (row, column)."""
import numpy as np
from .planet import ASSUMPTION


class World:
    operational_label = ASSUMPTION

    def __init__(self, width=16, height=12, seed=0, difficulty='medium', science_count=5):
        if not all(isinstance(v, int) and not isinstance(v, bool) and v > 0 for v in (width, height)):
            raise ValueError('Las dimensiones deben ser enteros positivos')
        if difficulty not in ('low', 'medium', 'high'):
            raise ValueError('Dificultad inválida: low, medium o high')
        if not isinstance(science_count, int) or science_count < 0:
            raise ValueError('Cantidad de sitios inválida')
        self.width, self.height, self.seed, self.difficulty = width, height, seed, difficulty
        self.rng = np.random.default_rng(seed)
        self.elevation = self.rng.uniform(0, 3, (height, width))
        self.elevation[0, 0] = 0
        cells = [(r, c) for r in range(height) for c in range(width) if (r, c) != (0, 0)]
        self.rng.shuffle(cells)
        count = int(len(cells) * {'low': .05, 'medium': .12, 'high': .22}[difficulty])
        self.obstacles = set(cells[:count])
        free = cells[count:]
        self.science_sites = set(free[:science_count])
        self.resources = set(free[science_count:science_count + 3])
        self.refuges = {(0, 0)}
        self.cliffs = set()
        self.visited_science = set()

    def in_bounds(self, position):
        try:
            r, c = position
            return isinstance(r, (int, np.integer)) and isinstance(c, (int, np.integer)) and 0 <= r < self.height and 0 <= c < self.width
        except (TypeError, ValueError):
            return False

    def add_obstacle(self, position):
        if not self.in_bounds(position):
            raise ValueError('Obstáculo fuera del mapa')
        self.obstacles.add(tuple(position))

    def traversable(self, position, mode='rover'):
        return self.in_bounds(position) and tuple(position) not in self.obstacles and (mode == 'probe' or tuple(position) not in self.cliffs)

    @property
    def science_progress(self):
        return len(self.visited_science & self.science_sites) / len(self.science_sites) if self.science_sites else 1.0

    def refuge_distance(self, position, mode='rover'):
        """Shortest currently feasible return distance, without promising rescue."""
        from collections import deque
        queue = deque([(tuple(position), 0)])
        seen = {tuple(position)}
        while queue:
            cell, distance = queue.popleft()
            if cell in self.refuges:
                return distance
            for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                neighbor = (cell[0] + dr, cell[1] + dc)
                if neighbor not in seen and self.traversable(neighbor, mode):
                    seen.add(neighbor)
                    queue.append((neighbor, distance + 1))
        return float('inf')


In [ ]:
%%writefile ares_x/src/ares/sensors.py
"""Imperfect measurements, with explicit critical-reading validation."""
import math
import numpy as np
from .planet import ASSUMPTION


class SensorSuite:
    operational_label = ASSUMPTION
    ranges = {'distance': (0, 20), 'slope': (0, 45), 'battery': (0, 100),
              'elevation_delta': (-1e6, 1e6), 'temperature_external': (-273.15, 1e4)}

    def __init__(self, seed=0, noise=.05):
        if not math.isfinite(noise) or noise < 0:
            raise ValueError('Ruido inválido')
        self.rng = np.random.default_rng(seed)
        self.noise = noise
        self.failures = {}
        self.link_available = True

    @staticmethod
    def validate(readings):
        for key in ('distance',):
            value = readings.get(key)
            if not isinstance(value, (int, float, np.number)) or not math.isfinite(value) or not 0 < value <= 20:
                return False
        for key in ('elevation_delta', 'temperature_external', 'temperature_internal'):
            if key in readings:
                value = readings[key]
                if not isinstance(value, (int, float, np.number)) or not math.isfinite(value):
                    return False
        for key, bounds in SensorSuite.ranges.items():
            if key in readings:
                value = readings[key]
                if not isinstance(value, (int, float, np.number)) or not math.isfinite(value) or not bounds[0] <= value <= bounds[1]:
                    return False
        return True

    def fail(self, sensor='distance', failure='nan'):
        if failure not in ('nan', 'saturation', 'missing'):
            raise ValueError('Tipo de falla desconocido')
        self.failures[sensor] = failure

    def observe(self, world, position, target, battery, profile, daylight=True, visibility=1.):
        valid = world.in_bounds(target)
        delta = float(world.elevation[target] - world.elevation[position]) if valid else 0.
        blocked = not world.traversable(target, profile.mode)
        readings = {
            'distance': 0. if blocked else float(np.clip(10 + self.rng.normal(0, self.noise), .001, 20)),
            'elevation_delta': delta,
            'cliff': valid and (tuple(target) in world.cliffs or delta < -5),
            'slope': float(min(45, abs(delta) * 3)),
            'battery': battery,
            'temperature_external': profile.temperature + float(self.rng.normal(0, self.noise)),
            'temperature_internal': 20., 'radiation': .1, 'pressure': profile.pressure,
            'oxygen_ambient': profile.oxygen, 'oxygen_stored': 100., 'co2': 95. if profile.name == 'Marte' else .04,
            'so2': 0., 'visibility': visibility, 'particles': 1 - visibility,
            'solar_power': 10. * visibility if daylight else 0., 'position': tuple(position),
            'heading': 0., 'vibration': 0., 'texture': float(self.rng.random()),
            'link_available': self.link_available,
        }
        for key, failure in self.failures.items():
            readings[key] = {'nan': float('nan'), 'saturation': float('inf'), 'missing': None}[failure]
        return readings


In [ ]:
%%writefile ares_x/src/ares/events.py
"""Explicit, ordered event injection; narrative signals are not scientific proof."""
from dataclasses import dataclass, field
from .planet import normalize

EVENTS = ('barranco', 'bateria_baja', 'dia', 'noche', 'sismo', 'meteorito', 'tormenta_polvo', 'corrosion', 'biosenal_anomala', 'alienigena', 'falla_sensor', 'perdida_enlace', 'viento')
ALIASES = {'low_battery': 'bateria_baja', 'day': 'dia', 'night': 'noche', 'quake': 'sismo', 'dust_storm': 'tormenta_polvo', 'sensor_failure': 'falla_sensor', 'link_loss': 'perdida_enlace', 'wind': 'viento'}


@dataclass
class Event:
    kind: str
    data: dict = field(default_factory=dict)

    def __post_init__(self):
        self.kind = normalize(self.kind).replace(' ', '_')
        self.kind = ALIASES.get(self.kind, self.kind)
        if self.kind not in EVENTS:
            raise ValueError('Evento desconocido')

    def apply(self, mission):
        world = mission.world
        position = tuple(self.data.get('position', mission.position))
        if self.kind in ('barranco', 'meteorito', 'alienigena'):
            if not world.in_bounds(position):
                raise ValueError('Evento fuera del mapa')
            if self.kind == 'barranco':
                world.cliffs.add(position)
                world.elevation[position] -= 20
            else:
                world.add_obstacle(position)
        elif self.kind == 'bateria_baja':
            mission.battery = min(mission.battery, 10.)
        elif self.kind in ('dia', 'noche'):
            mission.daylight = self.kind == 'dia'
        elif self.kind == 'sismo':
            mission.wait_steps = max(mission.wait_steps, 1)
        elif self.kind == 'tormenta_polvo':
            mission.visibility = .2
        elif self.kind == 'corrosion':
            mission.health = max(0., mission.health - 20.)
        elif self.kind == 'falla_sensor':
            mission.sensors.fail(self.data.get('sensor', 'distance'), self.data.get('failure', 'nan'))
        elif self.kind == 'perdida_enlace':
            mission.sensors.link_available = False
        elif self.kind == 'viento' and mission.profile.mode == 'probe':
            mission.wait_steps = max(mission.wait_steps, 1)
        return {'kind': self.kind, 'message': {
            'biosenal_anomala': 'Bioseñal anómala simulada; no confirma vida.',
            'alienigena': 'Evento narrativo completamente ficticio; evitar objeto.',
            'viento': 'Viento simulado; límites específicos del modo sonda.',
        }.get(self.kind, 'Evento aplicado: ' + self.kind)}


In [ ]:
%%writefile ares_x/src/ares/logger.py
"""Independent step snapshots with portable JSON-valued CSV columns."""
import copy
import csv
import io
import json
import math
from pathlib import Path


def clean(value):
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, dict):
        return {str(k): clean(v) for k, v in value.items()}
    if isinstance(value, (tuple, list)):
        return [clean(v) for v in value]
    return value


class MissionLogger:
    def __init__(self):
        self.trace = []

    def record(self, **record):
        snapshot = clean(copy.deepcopy(record))
        self.trace.append(snapshot)
        return copy.deepcopy(snapshot)

    def export_csv(self, path=None):
        output = io.StringIO(newline='')
        fields = list(dict.fromkeys(key for row in self.trace for key in row))
        writer = csv.DictWriter(output, fieldnames=fields)
        if fields:
            writer.writeheader()
            for row in self.trace:
                writer.writerow({k: json.dumps(v, ensure_ascii=False, sort_keys=True) if isinstance(v, (list, dict)) else v for k, v in row.items()})
        result = output.getvalue()
        if path is not None:
            Path(path).write_text(result, encoding='utf-8')
        return result


In [ ]:
%%writefile ares_x/src/ares/state_machine.py
"""Physical-only mission loop. No AI policy is implied by this controller."""
from collections import deque
from enum import Enum
import math
from .planet import PlanetProfile, ASSUMPTION
from .sensors import SensorSuite
from .logger import MissionLogger
from .events import Event


class MissionState(str, Enum):
    CONFIGURACION = 'CONFIGURACION'
    EXPLORACION = 'EXPLORACION'
    ALERTA = 'ALERTA'
    REPLANIFICACION = 'REPLANIFICACION'
    REFUGIO = 'REFUGIO'
    MODO_SEGURO = 'MODO_SEGURO'
    EXITO = 'EXITO'
    FRACASO = 'FRACASO'


class Mission:
    operational_label = ASSUMPTION

    def __init__(self, world, profile=None, battery=100., step_duration=1., sensors=None, logger=None, thermal_limit=500.):
        if not math.isfinite(battery) or not 0 <= battery <= 100:
            raise ValueError('Batería inválida: debe estar entre 0 y 100')
        if not math.isfinite(step_duration) or step_duration <= 0:
            raise ValueError('Duración de paso inválida')
        if not math.isfinite(thermal_limit) or thermal_limit <= -273.15:
            raise ValueError('Límite térmico inválido')
        self.thermal_limit = thermal_limit
        self.world = world
        self.profile = (profile or PlanetProfile.for_body()).validate()
        self.battery, self.step_duration = float(battery), float(step_duration)
        self.sensors = sensors or SensorSuite(world.seed)
        self.logger = logger or MissionLogger()
        self.position = (0, 0)
        self.state = MissionState.CONFIGURACION
        self.health = 100.
        self.daylight, self.visibility = True, 1.
        self.wait_steps = 0
        self.elapsed = 0.
        self.pending_events = []
        self.paused = False
        self.path = [self.position]

    @property
    def science_progress(self):
        return self.world.science_progress

    def refuge_distance(self, start):
        """Physical reachability only; this is not an AI route planner."""
        queue = deque([(start, 0)])
        seen = {start}
        while queue:
            position, distance = queue.popleft()
            if position in self.world.refuges:
                return distance
            for dr, dc in ((-1, 0), (0, 1), (1, 0), (0, -1)):
                neighbor = (position[0] + dr, position[1] + dc)
                if neighbor in seen or not self.world.traversable(neighbor, self.profile.mode):
                    continue
                if self.profile.mode == 'rover' and self.world.elevation[neighbor] - self.world.elevation[position] < -5:
                    continue
                seen.add(neighbor)
                queue.append((neighbor, distance + 1))
        return math.inf

    def inject_event(self, kind, **data):
        event = Event(kind, data)
        self.pending_events.append(event.apply(self))

    def step(self, target=None):
        if self.paused or self.state in (MissionState.EXITO, MissionState.FRACASO):
            return None
        target = self.position if target is None else tuple(target)
        readings = self.sensors.observe(self.world, self.position, target, self.battery, self.profile, self.daylight, self.visibility)
        action, reason = 'detener', 'ESPERA'
        if self.battery <= 0 or self.health <= 0:
            self.state, reason = MissionState.FRACASO, 'RECURSOS_AGOTADOS'
        elif readings['temperature_external'] is not None and math.isfinite(readings['temperature_external']) and readings['temperature_external'] > self.thermal_limit:
            self.state, reason = MissionState.MODO_SEGURO, 'LIMITE_TERMICO'
        elif self.health < 30:
            self.state, reason = MissionState.MODO_SEGURO, 'DANO_CRITICO'
        elif readings['cliff'] and self.profile.mode == 'rover':
            if self.world.in_bounds(target):
                self.world.cliffs.add(target)
            self.state, reason = MissionState.REPLANIFICACION, 'BARRANCO_CONFIRMADO'
        elif not self.world.traversable(target, self.profile.mode):
            self.state, reason = MissionState.REPLANIFICACION, 'RUTA_BLOQUEADA'
        elif not self.sensors.validate(readings):
            self.state, reason = MissionState.MODO_SEGURO, 'SENSOR_CRITICO_INVALIDO'
        elif self.wait_steps:
            self.wait_steps -= 1
            self.state, reason = MissionState.ALERTA, 'ESPERAR_REMEDIR'
        elif abs(target[0] - self.position[0]) + abs(target[1] - self.position[1]) > 1:
            self.state, reason = MissionState.REPLANIFICACION, 'MOVIMIENTO_NO_CONTIGUO'
        elif target != self.position:
            cost = self.step_duration * (1 + (1-self.visibility))
            return_distance = self.refuge_distance(self.position)
            target_distance = self.refuge_distance(target)
            if cost > self.battery:
                self.state, reason = MissionState.MODO_SEGURO, 'ENERGIA_INSUFICIENTE'
            elif self.battery <= 10 and self.position not in self.world.refuges and (return_distance * cost > self.battery or target_distance >= return_distance):
                self.state, reason = MissionState.MODO_SEGURO, 'REFUGIO_INALCANZABLE'
                if return_distance * cost <= self.battery:
                    self.state, reason = MissionState.REFUGIO, 'RECURSOS_CRITICOS'
            else:
                self.battery = max(0., self.battery - cost)
                self.position = target
                self.path.append(target)
                self.world.visited_science.update({target} & self.world.science_sites)
                self.state, action, reason = MissionState.EXPLORACION, 'avanzar', 'MOVIMIENTO_VALIDO'
                if self.battery == 0:
                    self.state, reason = MissionState.FRACASO, 'RECURSOS_AGOTADOS'
        else:
            self.state = MissionState.EXPLORACION
        if self.science_progress == 1 and self.position in self.world.refuges and self.state == MissionState.EXPLORACION:
            self.state, reason = MissionState.EXITO, 'OBJETIVOS_Y_RETORNO_COMPLETADOS'
        self.elapsed += self.step_duration
        events, self.pending_events = self.pending_events, []
        return self.logger.record(step=len(self.logger.trace), elapsed=self.elapsed, state=self.state.value,
                                  position=self.position, battery=self.battery, health=self.health,
                                  sensors=readings, events=events, action=action, reason=reason,
                                  science_progress=self.science_progress, seed=self.world.seed,
                                  operational_label=ASSUMPTION)


StateMachine = Mission


In [ ]:
%%writefile ares_x/src/ares/neural.py
"""Inspectable CPU perception trained on noisy, map-separated synthetic data.

Ground truth is a simulator property: cliff membership or a physical drop below
-5 m takes precedence over obstacle membership. Neither membership nor the
deterministic edge flag is passed to the MLP. Observations are noisy proxies;
confidence is not a safety guarantee and never replaces the edge-sensor veto.
"""
from collections.abc import Mapping
import warnings

import numpy as np
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

from .planet import ASSUMPTION, PlanetProfile
from .sensors import SensorSuite
from .world import World


FEATURES = ('distance', 'slope', 'elevation_delta', 'texture', 'visibility', 'vibration')
LABELS = ('libre', 'obstaculo', 'barranco')
BOUNDS = ((.000001, 20.), (0., 45.), (-1e6, 1e6), (0., 1.), (0., 1.), (0., 20.))


def _integer(value, minimum, message):
    if isinstance(value, bool) or not isinstance(value, (int, np.integer)) or value < minimum:
        raise ValueError(message)
    return int(value)


def _matrix(values):
    try:
        array = np.asarray(values, dtype=float)
    except (TypeError, ValueError, OverflowError) as error:
        raise ValueError('Entrada de sensores inválida: se requieren números finitos') from error
    if array.ndim != 2 or array.shape[1] != len(FEATURES) or not len(array) or not np.isfinite(array).all():
        raise ValueError('Entrada de sensores inválida: se requieren seis valores finitos')
    lower, upper = np.asarray(BOUNDS).T
    if np.any(array < lower) or np.any(array > upper):
        raise ValueError('Entrada de sensores fuera de rango físico; distancia cero requiere frenado')
    return array


def generate_training_data(seed=0, map_count=18, samples_per_map=120):
    """Generate map-grouped observations with simulator-owned terrain labels.

    Distance, slope, elevation and texture include observation noise and map
    biases. Cliffs are added as synthetic simulator terrain (not sensor labels).
    Distances below sensor resolution are represented as a small positive
    return here; a literal zero from the live sensor is separately invalid.
    """
    seed = _integer(seed, 0, 'Semilla inválida')
    map_count = _integer(map_count, 5, 'Se requieren al menos cinco mapas')
    samples_per_map = _integer(samples_per_map, 10, 'Se requieren al menos diez muestras por mapa')
    rng = np.random.default_rng(seed)
    profile = PlanetProfile.for_body('Marte')
    rows, labels, groups = [], [], []
    for group in range(map_count):
        map_seed = int(rng.integers(0, 2**31 - 1))
        world = World(width=12, height=10, seed=map_seed, difficulty=('low', 'medium', 'high')[group % 3])
        sensors = SensorSuite(seed=map_seed, noise=.12)
        cells = [(r, c) for r in range(world.height) for c in range(world.width) if (r, c) != (0, 0)]
        cliff_indices = rng.choice(len(cells), size=12, replace=False)
        world.cliffs.update(cells[index] for index in cliff_indices)
        for cell in world.cliffs:
            world.elevation[cell] = rng.uniform(-12, -7)
        edges = [(position, (position[0] + dr, position[1] + dc))
                 for position in cells for dr, dc in ((0, 1), (0, -1), (1, 0), (-1, 0))
                 if world.in_bounds((position[0] + dr, position[1] + dc))]
        distance_bias = rng.normal(0, .3)
        for index in rng.integers(0, len(edges), size=samples_per_map):
            position, target = edges[index]
            delta = float(world.elevation[target] - world.elevation[position])
            label = ('barranco' if target in world.cliffs or delta < -5 else
                     'obstaculo' if target in world.obstacles else 'libre')
            observation = sensors.observe(world, position, target, 80., profile,
                                          visibility=float(rng.uniform(.35, 1.)))
            observation['distance'] = float(np.clip(observation['distance'] + distance_bias + rng.normal(0, .8), .01, 20))
            observation['elevation_delta'] = delta + rng.normal(0, 1.3)
            observation['slope'] = float(np.clip(observation['slope'] + rng.normal(0, 2.), 0, 45))
            observation['texture'] = float(np.clip(.25 + .3 * (target in world.obstacles) + rng.normal(0, .22), 0, 1))
            observation['vibration'] = float(np.clip(abs(delta) * .15 + rng.normal(0, .15), 0, 20))
            rows.append([observation[name] for name in FEATURES])
            labels.append(label)
            groups.append(group)
    return {'X': np.asarray(rows), 'y': np.asarray(labels), 'groups': np.asarray(groups),
            'operational_label': ASSUMPTION}


def _calibration_error(truth, probabilities, predictions):
    """Ten-bin expected calibration error, measured only on the named split."""
    confidence = probabilities.max(axis=1)
    correct = truth == predictions
    bins = np.minimum((confidence * 10).astype(int), 9)
    return float(sum(np.mean(bins == index) * abs(correct[bins == index].mean() - confidence[bins == index].mean())
                     for index in range(10) if np.any(bins == index)))


class NeuralPerception:
    """MLP (32, 16), training-only scaling, and transparent held-out metrics.

    ``train`` returns validation/test metrics and split provenance. Validation
    is reserved for inspection, not mixed into fitting. ``predict`` accepts a
    sensor mapping or a row ordered by FEATURES; malformed critical input
    raises ValueError so the caller must brake/degrade rather than infer blind.
    """
    operational_label = ASSUMPTION

    def __init__(self, seed=0, confidence_threshold=.65):
        self.seed = _integer(seed, 0, 'Semilla inválida')
        try:
            threshold = float(confidence_threshold)
        except (TypeError, ValueError, OverflowError) as error:
            raise ValueError('Umbral de confianza inválido') from error
        if not np.isfinite(threshold) or not 0 < threshold <= 1:
            raise ValueError('Umbral de confianza inválido')
        self.confidence_threshold = threshold
        self.model = None
        self.scaler = None

    def train(self, dataset=None, map_count=18, samples_per_map=120, max_iter=200):
        max_iter = _integer(max_iter, 1, 'Cantidad de iteraciones inválida')
        data = generate_training_data(self.seed, map_count, samples_per_map) if dataset is None else dataset
        try:
            rows = _matrix(data['X'])
            truth = np.asarray(data['y'], dtype=str)
            groups = np.asarray(data['groups'])
        except (KeyError, TypeError, ValueError) as error:
            raise ValueError('Entrada de datos de entrenamiento inválida') from error
        if (truth.shape != (len(rows),) or groups.shape != (len(rows),)
                or set(truth) != set(LABELS) or groups.dtype.kind not in 'iu'):
            raise ValueError('Los datos requieren tres clases y grupos de mapa enteros')
        unique = np.unique(groups)
        if len(unique) < 5:
            raise ValueError('Se requieren al menos cinco mapas independientes')
        ordered = np.random.default_rng(self.seed).permutation(unique)
        train_end, valid_end = int(.6 * len(ordered)), int(.8 * len(ordered))
        map_splits = dict(zip(('train', 'validation', 'test'), np.split(ordered, (train_end, valid_end))))
        indices = {key: np.flatnonzero(np.isin(groups, maps)) for key, maps in map_splits.items()}
        if set(truth[indices['train']]) != set(LABELS):
            raise ValueError('Los mapas de entrenamiento deben contener las tres clases')
        scaler = StandardScaler().fit(rows[indices['train']])
        model = MLPClassifier(hidden_layer_sizes=(32, 16), activation='relu',
                              early_stopping=False, random_state=self.seed,
                              max_iter=max_iter, learning_rate_init=.003)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always', ConvergenceWarning)
            model.fit(scaler.transform(rows[indices['train']]), truth[indices['train']])
        majority = max(LABELS, key=lambda label: np.sum(truth[indices['train']] == label))
        report = {'operational_label': ASSUMPTION,
                  'split_maps': {key: maps.tolist() for key, maps in map_splits.items()},
                  'training_samples': len(indices['train']),
                  'converged': not any(issubclass(item.category, ConvergenceWarning) for item in caught),
                  'iterations': int(model.n_iter_), 'majority_label': majority,
                  'confidence_note': 'Probabilidad del modelo; no es garantía de seguridad'}
        for name in ('validation', 'test'):
            selection = indices[name]
            expected = truth[selection]
            probabilities = model.predict_proba(scaler.transform(rows[selection]))
            predicted = model.classes_[probabilities.argmax(axis=1)]
            report[name] = {
                'samples': len(selection), 'labels': list(LABELS),
                'accuracy': float(accuracy_score(expected, predicted)),
                'majority_baseline': float(np.mean(expected == majority)),
                'confusion_matrix': confusion_matrix(expected, predicted, labels=LABELS).tolist(),
                'macro_f1': float(f1_score(expected, predicted, labels=LABELS, average='macro', zero_division=0)),
                'barranco_recall': float(recall_score(expected, predicted, labels=['barranco'], average='macro', zero_division=0)),
                'calibration_error': _calibration_error(expected, probabilities, predicted),
            }
        self.model, self.scaler = model, scaler
        self.split_indices_, self.report_ = indices, report
        return report

    def predict(self, readings):
        if self.model is None:
            raise ValueError('La red debe entrenarse antes de predecir')
        if isinstance(readings, Mapping):
            try:
                values = [readings[key] for key in FEATURES]
            except KeyError as error:
                raise ValueError(f'Entrada de sensor incompleta: falta {error.args[0]}') from error
        else:
            values = readings
        row = _matrix([values])
        normalized = self.scaler.transform(row)
        probabilities = dict(zip(self.model.classes_, self.model.predict_proba(normalized)[0].tolist()))
        probabilities = {label: float(probabilities[label]) for label in LABELS}
        activation = normalized
        summaries = []
        for weights, bias in zip(self.model.coefs_[:-1], self.model.intercepts_[:-1]):
            activation = np.maximum(0., activation @ weights + bias)
            summaries.append({'neurons': activation.shape[1], 'mean': float(activation.mean()),
                              'max': float(activation.max()), 'min': float(activation.min()),
                              'active_fraction': float(np.mean(activation > 0))})
        label = max(probabilities, key=probabilities.get)
        confidence = probabilities[label]
        return {'label': label, 'probabilities': probabilities, 'confidence': confidence,
                'low_confidence': confidence < self.confidence_threshold,
                'raw_input': row[0].tolist(), 'normalized_input': normalized[0].tolist(),
                'feature_names': list(FEATURES), 'activation_summaries': summaries,
                'risk': .7 * probabilities['obstaculo'] + probabilities['barranco'],
                'operational_label': ASSUMPTION}


In [ ]:
%%writefile ares_x/src/ares/fuzzy.py
"""Inspectable Mamdani controller; all operational thresholds are assumptions."""
import math

import numpy as np

from .planet import ASSUMPTION


def triangular(value, a, b, c):
    """Triangular membership, including degenerate shoulder endpoints."""
    if value < a or value > c:
        return 0.0
    if value == b:
        return 1.0
    return float((value - a) / (b - a) if value < b else (c - value) / (c - b))


def trapezoidal(value, a, b, c, d):
    """Trapezoidal membership with closed left/right shoulders."""
    if value < a or value > d:
        return 0.0
    if b <= value <= c:
        return 1.0
    return float((value - a) / (b - a) if value < b else (d - value) / (d - c))


class FuzzyController:
    """Speed is advisory, not a physical veto; H3 supplies hard safety limits.

    Invalid input or zero aggregate area produces speed 0 and caution 1.
    Temperature uses hypothetical hardware-relative thresholds, not planetary
    engineering limits. Min implements AND, max implements union of rules.
    """

    operational_label = ASSUMPTION
    membership_sets = {
        'distance': {'cerca': ('trap', 0, 0, 2, 6),
                     'media': ('tri', 2, 9, 15), 'lejos': ('trap', 10, 15, 20, 20)},
        'battery': {'critica': ('trap', 0, 0, 8, 18),
                    'baja': ('tri', 10, 30, 55), 'alta': ('trap', 40, 65, 100, 100)},
        'slope': {'plana': ('trap', 0, 0, 4, 12),
                  'media': ('tri', 5, 20, 32), 'fuerte': ('trap', 25, 35, 45, 45)},
        'temperature': {'fria': ('trap', -273.15, -273.15, -60, 0),
                        'templada': ('tri', -30, 20, 70),
                        'caliente': ('trap', 40, 90, 1000, 1000)},
    }
    output_sets = {
        'parar': ('trap', 0, 0, .01, .06),
        'lenta': ('tri', .03, .18, .35),
        'moderada': ('tri', .25, .5, .75),
        'rapida': ('trap', .6, .85, 1, 1),
    }
    # Each tuple is an AND antecedent and its speed consequent.
    rule_specs = (
        ('Barranco o riesgo cercano: parar.', (('distance', 'cerca'),), 'parar'),
        ('Batería crítica: avance lento.', (('battery', 'critica'),), 'lenta'),
        ('Pendiente fuerte y batería alta: avance lento.', (('slope', 'fuerte'), ('battery', 'alta')), 'lenta'),
        ('Pendiente media y distancia media: avance lento.', (('slope', 'media'), ('distance', 'media')), 'lenta'),
        ('Batería alta, terreno plano y riesgo lejano: rápido.', (('battery', 'alta'), ('slope', 'plana'), ('distance', 'lejos')), 'rapida'),
        ('Batería baja y distancia media: lento.', (('battery', 'baja'), ('distance', 'media')), 'lenta'),
        ('Distancia media y terreno plano: moderado.', (('distance', 'media'), ('slope', 'plana')), 'moderada'),
        ('Temperatura caliente: parar y evaluar protección.', (('temperature', 'caliente'),), 'parar'),
        ('Temperatura fría: lento para proteger equipos.', (('temperature', 'fria'),), 'lenta'),
        ('Batería alta, distancia media y temperatura templada: moderado.', (('battery', 'alta'), ('distance', 'media'), ('temperature', 'templada')), 'moderada'),
    )

    def __init__(self, rules=None):
        self.rules = [
            {'id': index + 1, 'description': description,
             'antecedents': antecedents, 'consequent': consequent}
            for index, (description, antecedents, consequent) in enumerate(self.rule_specs)
        ] if rules is None else list(rules)
        self.universe = np.linspace(0, 1, 201)

    @staticmethod
    def membership(value, specification):
        kind, *points = specification
        return triangular(value, *points) if kind == 'tri' else trapezoidal(value, *points)

    def infer(self, distance, battery, slope, temperature=20):
        values = dict(distance=distance, battery=battery, slope=slope, temperature=temperature)
        limits = {'distance': (0, 20), 'battery': (0, 100),
                  'slope': (0, 45), 'temperature': (-273.15, 1000)}
        aggregate = np.zeros_like(self.universe)
        result = {'speed': 0.0, 'caution': 1.0, 'centroid': 0.0,
                  'membership_degrees': {}, 'activated_rules': [],
                  'universe': self.universe.tolist(), 'aggregated_curve': aggregate.tolist(),
                  'reason': 'ENTRADA_INVALIDA', 'operational_label': ASSUMPTION}
        if any(not isinstance(value, (int, float, np.number)) or not math.isfinite(value)
               or not limits[name][0] <= value <= limits[name][1]
               for name, value in values.items()):
            return result
        degrees = {name: {label: self.membership(values[name], spec)
                          for label, spec in sets.items()}
                   for name, sets in self.membership_sets.items()}
        result['membership_degrees'] = degrees
        for rule in self.rules:
            strength = min(degrees[name][label] for name, label in rule['antecedents'])
            if strength > 0:
                curve = np.array([self.membership(x, self.output_sets[rule['consequent']])
                                  for x in self.universe])
                aggregate = np.maximum(aggregate, np.minimum(strength, curve))
                result['activated_rules'].append({**rule, 'strength': strength})
        integrate = getattr(np, 'trapezoid', None) or np.trapz  # numpy < 2 compatibility
        area = integrate(aggregate, self.universe)
        centroid = float(integrate(self.universe * aggregate, self.universe) / area) if area > 0 else 0.0
        result.update(speed=centroid, centroid=centroid, caution=1.0 - centroid,
                      aggregated_curve=aggregate.tolist(),
                      reason='INFERENCIA_MAMDANI' if area > 0 else 'SIN_REGLAS_ACTIVAS')
        return result


In [ ]:
%%writefile ares_x/src/ares/genetic.py
"""Seeded waypoint evolution over a known map, never a future-event oracle."""
from collections import deque
from math import isfinite
import random

from .planet import ASSUMPTION


class GeneticPlanner:
    """Science-visit ordering and maze feasibility with conservative safe return.

    All operational defaults and objective weights are simulation assumptions.
    Positive science competes with normalized distance, energy, risk and time.
    Every infeasible chromosome receives a penalty above any possible reward.
    """

    operational_label = ASSUMPTION
    default_weights = dict(science=3., distance=1., energy=1., predicted_risk=2., time=1.)

    def __init__(self, population_size=30, generations=30, waypoints=4, seed=0,
                 mutation_rate=.25, crossover_rate=.85, elitism=2,
                 tournament_size=3, weights=None, early_stop=True):
        for name, value, minimum in [('population_size', population_size, 30),
                                     ('generations', generations, 30),
                                     ('waypoints', waypoints, 1),
                                     ('elitism', elitism, 1),
                                     ('tournament_size', tournament_size, 2)]:
            if not isinstance(value, int) or isinstance(value, bool) or value < minimum:
                raise ValueError(f'{name}: valor inválido; mínimo {minimum}')
        if elitism >= population_size or tournament_size > population_size:
            raise ValueError('Elitismo o torneo incompatible con la población')
        if any(not isinstance(v, (int, float)) or not isfinite(v) or not 0 <= v <= 1
               for v in (mutation_rate, crossover_rate)):
            raise ValueError('Las tasas deben estar entre 0 y 1')
        self.weights = self.default_weights.copy()
        if weights is not None:
            if set(weights) - set(self.weights):
                raise ValueError('Componente de fitness desconocido')
            self.weights.update(weights)
        if any(not isinstance(v, (int, float)) or not isfinite(v) or v < 0
               for v in self.weights.values()) or not any(self.weights.values()):
            raise ValueError('Pesos inválidos: deben ser finitos, no negativos y no todos cero')
        self.population_size, self.generations, self.waypoints = population_size, generations, waypoints
        self.seed, self.mutation_rate, self.crossover_rate = seed, mutation_rate, crossover_rate
        self.elitism, self.tournament_size, self.early_stop = elitism, tournament_size, early_stop
        self.rng = random.Random(seed)
        self.history = []

    @staticmethod
    def _free(world):
        return [(r, c) for r in range(world.height) for c in range(world.width)
                if world.traversable((r, c))]

    def repair(self, chromosome, world):
        """Clamp invalid coordinates and replace blocked genes with nearest free cells."""
        free = self._free(world)
        if not free:
            return [(0, 0)] * self.waypoints
        genes = list(chromosome)[:self.waypoints]
        genes += [free[0]] * (self.waypoints - len(genes))
        repaired = []
        for gene in genes:
            try:
                row, col = gene
                row = min(world.height - 1, max(0, round(row))) if isfinite(row) else 0
                col = min(world.width - 1, max(0, round(col))) if isfinite(col) else 0
            except (TypeError, ValueError, OverflowError):
                row, col = free[0]
            point = (row, col)
            if not world.traversable(point):
                point = min(free, key=lambda p: (abs(p[0] - row) + abs(p[1] - col), p))
            repaired.append(point)
        return repaired

    def mutate(self, chromosome, world):
        """Mutate positions or scientific visit order, then repair every gene."""
        genes = self.repair(chromosome, world)
        free = self._free(world)
        if free:
            for i in range(self.waypoints):
                if self.rng.random() < self.mutation_rate:
                    genes[i] = self.rng.choice(free)
            if self.waypoints > 1 and self.rng.random() < self.mutation_rate:
                a, b = self.rng.sample(range(self.waypoints), 2)
                genes[a], genes[b] = genes[b], genes[a]
        return self.repair(genes, world)

    @staticmethod
    def _path(world, start, goal):
        if not world.traversable(start) or not world.traversable(goal):
            return None
        queue, previous = deque([start]), {start: None}
        while queue:
            cell = queue.popleft()
            if cell == goal:
                route = []
                while cell is not None:
                    route.append(cell)
                    cell = previous[cell]
                return route[::-1]
            for dr, dc in ((-1, 0), (0, 1), (1, 0), (0, -1)):
                neighbor = (cell[0] + dr, cell[1] + dc)
                if (neighbor not in previous and world.traversable(neighbor)
                        and world.elevation[neighbor] - world.elevation[cell] >= -5):
                    # Match the deterministic cliff rule used by SensorSuite.
                    previous[neighbor] = cell
                    queue.append(neighbor)
        return None

    def _return_distance(self, world, position):
        """Return paths obey the same directed cliff edges as outbound travel."""
        distances = []
        for refuge in sorted(world.refuges):
            path = self._path(world, position, refuge)
            if path is not None:
                distances.append(len(path) - 1)
        return min(distances, default=float('inf'))

    @staticmethod
    def _risk_at(estimated_risk, cell):
        if estimated_risk is None:
            return 0.
        try:
            value = estimated_risk.get(cell, 0.) if hasattr(estimated_risk, 'get') else estimated_risk[cell]
            value = float(value)
        except (TypeError, ValueError, IndexError, KeyError):
            raise ValueError('Mapa de riesgo estimado inválido') from None
        if not isfinite(value) or not 0 <= value <= 1:
            raise ValueError('Riesgo estimado fuera de [0, 1]')
        return value

    def evaluate(self, chromosome, world, start, goal, battery=100, estimated_risk=None,
                 temperature=20, thermal_limit=80, energy_per_step=1, time_per_step=1):
        """Decode a chromosome; normalized costs use fixed map-wide scales.

        Battery reserves cover the route AND a shortest known return from its end.
        Thermal feasibility is a supplied current hardware estimate, not weather.
        """
        numeric = (battery, temperature, thermal_limit, energy_per_step, time_per_step)
        if any(not isinstance(v, (int, float)) or not isfinite(v) for v in numeric):
            raise ValueError('Parámetros de planificación no finitos')
        if battery < 0 or energy_per_step <= 0 or time_per_step <= 0:
            raise ValueError('Batería o costo de paso inválido')
        genes = self.repair(chromosome, world)
        repaired = list(chromosome) != genes
        violations, route = [], []
        if not world.in_bounds(start) or not world.in_bounds(goal):
            violations.append('bounds')
        elif not world.traversable(start) or not world.traversable(goal):
            violations.append('collision_or_cliff')
        else:
            route = [tuple(start)]
            for index, point in enumerate(genes + [tuple(goal)]):
                segment = self._path(world, route[-1], point)
                if segment is None:
                    if index < self.waypoints:
                        # A disconnected optional waypoint cannot invalidate a reachable goal.
                        genes[index] = route[-1]
                        repaired = True
                        continue
                    violations.append('unreachable')
                    break
                route.extend(segment[1:])
        if temperature > thermal_limit:
            violations.append('thermal')
        distance = max(0, len(route) - 1)
        return_distance = self._return_distance(world, route[-1]) if route else float('inf')
        energy_required = (distance + return_distance) * energy_per_step
        if energy_required > battery or (distance > 0 and battery == 0):
            violations.append('battery')
        remaining_sites = world.science_sites - world.visited_science
        science_order = []
        for cell in route:
            if cell in remaining_sites and cell not in science_order:
                science_order.append(cell)
        max_distance = max(1, (world.width * world.height - 1) * (self.waypoints + 1))
        components = {
            'science': len(science_order) / len(remaining_sites) if remaining_sites else 0.,
            'distance': min(1., distance / max_distance),
            'energy': min(1., distance * energy_per_step / 100.),
            'predicted_risk': sum(self._risk_at(estimated_risk, p) for p in route) / max(1, len(route)),
            'time': min(1., distance * time_per_step / max_distance),
        }
        penalty = (1 + sum(self.weights.values())) * len(violations)
        fitness = self.weights['science'] * components['science'] - sum(
            self.weights[k] * v for k, v in components.items() if k != 'science') - penalty
        return dict(chromosome=genes, route=route, feasible=not violations, fitness=fitness,
                    components=components, penalty=penalty, violations=violations,
                    science_order=science_order, energy_required=energy_required,
                    distance=distance, repaired=repaired)

    def plan(self, world, start, goal, battery=100, estimated_risk=None, **kwargs):
        """Return at least three candidates plus inspectable evolutionary history.

        A planning run resets its seed for identical inputs/configuration. Early
        stopping can end before the configured >=30-generation budget, after ten
        stagnant generations; the global best feasible route is always retained.
        """
        self.rng = random.Random(self.seed)
        free = self._free(world)
        fallback = tuple(start) if world.in_bounds(start) else (0, 0)
        population = [[fallback] * self.waypoints]
        sites = sorted(world.science_sites - world.visited_science)
        for _ in range(self.population_size - 1):
            pool = sites if sites and self.rng.random() < .5 else free
            population.append([self.rng.choice(pool) if pool else fallback for _ in range(self.waypoints)])
        self.history, archive = [], {}
        stale, best_score, best_feasible = 0, float('-inf'), None
        counters = dict(tournaments=0, crossovers=0, mutations=0)
        for generation in range(self.generations + 1):
            scored = [self.evaluate(g, world, start, goal, battery, estimated_risk, **kwargs) for g in population]
            scored.sort(key=lambda c: (c['feasible'], c['fitness']), reverse=True)
            for candidate in scored:
                key = tuple(candidate['route'])
                if key not in archive or candidate['fitness'] > archive[key]['fitness']:
                    archive[key] = candidate
                if candidate['feasible'] and (best_feasible is None or candidate['fitness'] > best_feasible['fitness']):
                    best_feasible = candidate
            score = scored[0]['fitness']
            stale = 0 if score > best_score else stale + 1
            best_score = max(best_score, score)
            self.history.append(dict(generation=generation, chromosomes=[c['chromosome'][:] for c in scored],
                                     best_chromosome=scored[0]['chromosome'][:], best_fitness=score,
                                     best_feasible_fitness=best_feasible['fitness'] if best_feasible else None,
                                     stale_generations=stale, **counters))
            if generation == self.generations or (self.early_stop and stale >= 10):
                break
            population = [c['chromosome'][:] for c in scored[:self.elitism]]
            counters = dict(tournaments=0, crossovers=0, mutations=0)
            while len(population) < self.population_size:
                parents = []
                for _ in range(2):
                    contestants = self.rng.sample(scored, self.tournament_size)
                    parents.append(max(contestants, key=lambda c: (c['feasible'], c['fitness']))['chromosome'])
                    counters['tournaments'] += 1
                child = parents[0][:]
                if self.waypoints > 1 and self.rng.random() < self.crossover_rate:
                    split = self.rng.randrange(1, self.waypoints)
                    child = parents[0][:split] + parents[1][split:]
                    counters['crossovers'] += 1
                mutated = self.mutate(child, world)
                counters['mutations'] += sum(a != b for a, b in zip(child, mutated))
                population.append(mutated)
        candidates = sorted(archive.values(), key=lambda c: (c['feasible'], c['fitness']), reverse=True)[:3]
        while len(candidates) < 3:
            candidates.append(scored[len(candidates) % len(scored)])
        feasible = best_feasible is not None
        return dict(route=best_feasible['route'] if feasible else [], feasible=feasible,
                    fitness=best_feasible['fitness'] if feasible else scored[0]['fitness'],
                    candidates=candidates, history=self.history, generations_run=generation,
                    stopped_early=generation < self.generations,
                    message='Ruta factible (supuesto de simulación)' if feasible else
                    'Sin solución: retornar solo si es factible; de lo contrario, esperar en modo seguro.')


In [ ]:
%%writefile ares_x/src/ares/supervisor.py
"""Hybrid safety supervisor: physical prohibitions first, then AI proposals.

Every decision carries a reproducible reason code. Rules are evaluated in a
fixed priority order, so simultaneous events always resolve the same way.
"""
from dataclasses import dataclass, field

from .planet import ASSUMPTION

# Ordered by priority (lower index wins). Codes are stable for audits/tests.
REASONS = {
    'R00_RECURSOS_AGOTADOS': 'Batería o integridad agotadas: misión terminada.',
    'R01_LIMITE_TERMICO': 'Temperatura fuera de la tolerancia del hardware: se aísla el sistema.',
    'R02_SENSORES_NAVEGACION_FALLIDOS': 'Fallan todos los sensores de navegación: no se avanza a ciegas.',
    'R03_DANO_CRITICO': 'Integridad mecánica crítica: modo seguro.',
    'R04_VETO_BARRANCO_CONFIRMADO': 'El sensor de borde confirma un barranco en la siguiente celda: se detiene y replanifica.',
    'R05_RNA_PELIGRO': 'La RNA predice peligro en la siguiente celda con confianza suficiente: se detiene y replanifica.',
    'R06_REFUGIO_INALCANZABLE': 'La energía no alcanza para llegar a ningún refugio: se declara inalcanzable y se espera en modo seguro.',
    'R07_RETORNO_ENERGIA': 'La energía restante solo alcanza para volver: se prioriza el refugio.',
    'R08_ESPERA_SISMO': 'Sismo detectado: frenar, esperar y volver a medir.',
    'R09_RECARGA': 'En refugio con batería incompleta: se recarga.',
    'R10_SIN_RUTA': 'No existe ruta factible: se informa sin solución y se espera.',
    'R11_REPLANIFICAR': 'Ruta bloqueada o evento grave: se ejecuta el AG.',
    'R12_DIFUSO_PARAR': 'El controlador difuso recomienda velocidad casi nula: se detiene.',
    'R13_AVANCE_DEGRADADO': 'Sensor degradado o RNA con baja confianza: avance conservador fusionado con sensor determinista.',
    'R14_AVANCE': 'Ruta segura y sensores válidos: se avanza.',
    'R15_MISION_COMPLETA': 'Objetivos alcanzables completados y retorno seguro.',
}
PRIORITY = list(REASONS)


@dataclass
class Decision:
    action: str            # avanzar | detener | recargar | volver | replanificar
    reason: str
    state: str
    speed_cap: float = 1.0
    vetoes: list = field(default_factory=list)
    permitted_by: list = field(default_factory=list)

    @property
    def explanation(self):
        return REASONS[self.reason]

    def to_dict(self):
        return {'action': self.action, 'reason': self.reason, 'state': self.state,
                'explanation': self.explanation, 'speed_cap': self.speed_cap,
                'vetoes': list(self.vetoes), 'permitted_by': list(self.permitted_by),
                'operational_label': ASSUMPTION}


class SafetySupervisor:
    """Arbitrates RNA, fuzzy and GA recommendations under hard constraints.

    ``context`` keys: battery, health (required); temperature_internal,
    thermal_limit, nav_valid (sensor -> bool), edge_cliff, rna (prediction or
    None), rna_used, fuzzy_speed, has_route, needs_replan, quake_wait,
    at_refuge, energy_to_refuge, recharge_target, returning, mission_complete.
    """

    operational_label = ASSUMPTION

    def __init__(self, confidence_threshold=.65, hazard_threshold=.5, return_margin=1.25,
                 stop_speed=.08, degraded_speed=.3):
        self.confidence_threshold = confidence_threshold
        self.hazard_threshold = hazard_threshold
        self.return_margin = return_margin
        self.stop_speed = stop_speed
        self.degraded_speed = degraded_speed

    def decide(self, ctx):
        vetoes, permits = [], []
        battery, health = ctx['battery'], ctx['health']
        if battery <= 0 or health <= 0:
            return Decision('detener', 'R00_RECURSOS_AGOTADOS', 'FRACASO')
        if ctx.get('temperature_internal', 20) > ctx.get('thermal_limit', 1e9):
            return Decision('detener', 'R01_LIMITE_TERMICO', 'MODO_SEGURO')
        nav = ctx.get('nav_valid', {})
        if nav and not any(nav.values()):
            return Decision('detener', 'R02_SENSORES_NAVEGACION_FALLIDOS', 'MODO_SEGURO')
        if health < 30:
            return Decision('detener', 'R03_DANO_CRITICO', 'MODO_SEGURO')
        if ctx.get('mission_complete'):
            return Decision('detener', 'R15_MISION_COMPLETA', 'EXITO')
        # Energy feasibility is a physical constraint and precedes AI proposals.
        energy_to_refuge = ctx.get('energy_to_refuge', 0.)
        if not ctx.get('at_refuge') and (energy_to_refuge == float('inf') or energy_to_refuge > battery):
            return Decision('detener', 'R06_REFUGIO_INALCANZABLE', 'MODO_SEGURO')
        if ctx.get('edge_cliff'):
            vetoes.append('Barranco confirmado por sensor de borde')
            return Decision('replanificar', 'R04_VETO_BARRANCO_CONFIRMADO', 'REPLANIFICACION', 0., vetoes)
        rna = ctx.get('rna') if ctx.get('rna_used', True) else None
        low_confidence = False
        if rna is not None:
            hazard = 1 - rna['probabilities']['libre']
            if rna['confidence'] >= self.confidence_threshold and rna['label'] != 'libre':
                vetoes.append(f"RNA: {rna['label']} p={rna['confidence']:.2f}")
                return Decision('replanificar', 'R05_RNA_PELIGRO', 'ALERTA', 0., vetoes)
            low_confidence = rna['low_confidence'] or hazard >= self.hazard_threshold
            permits.append(f"RNA: libre p={rna['probabilities']['libre']:.2f}")
        if ctx.get('quake_wait', 0) > 0:
            return Decision('detener', 'R08_ESPERA_SISMO', 'ALERTA', 0., vetoes, permits)
        if ctx.get('at_refuge') and battery < ctx.get('recharge_target', 0):
            return Decision('recargar', 'R09_RECARGA', 'REFUGIO', 0., vetoes, permits)
        if not ctx.get('at_refuge') and not ctx.get('returning') and energy_to_refuge * self.return_margin >= battery:
            permits.append('Energía justa para el retorno')
            return Decision('volver', 'R07_RETORNO_ENERGIA', 'REFUGIO', self.degraded_speed, vetoes, permits)
        if ctx.get('needs_replan'):
            return Decision('replanificar', 'R11_REPLANIFICAR', 'REPLANIFICACION', 0., vetoes, permits)
        if not ctx.get('has_route'):
            return Decision('detener', 'R10_SIN_RUTA', 'ALERTA', 0., vetoes, permits)
        speed = ctx.get('fuzzy_speed')
        if speed is not None and speed < self.stop_speed:
            return Decision('detener', 'R12_DIFUSO_PARAR', 'ALERTA', 0., vetoes, permits)
        if low_confidence or (nav and not all(nav.values())):
            permits.append('Avance limitado por degradación')
            return Decision('avanzar', 'R13_AVANCE_DEGRADADO', 'ALERTA', self.degraded_speed, vetoes, permits)
        state = 'REFUGIO' if ctx.get('returning') else 'EXPLORACION'
        return Decision('avanzar', 'R14_AVANCE', state, 1.0, vetoes, permits)

    def filter_routes(self, candidates, known_cliffs, known_obstacles):
        """Reject GA candidates crossing confirmed hazards; document each veto."""
        accepted, rejected = [], []
        for index, candidate in enumerate(candidates):
            route = candidate.get('route') or []
            bad = [cell for cell in route if cell in known_cliffs or cell in known_obstacles]
            if bad:
                rejected.append({'index': index, 'reason': 'cruza peligro confirmado', 'cells': [list(c) for c in bad[:3]]})
            elif not candidate.get('feasible'):
                rejected.append({'index': index, 'reason': 'infactible: ' + ', '.join(candidate.get('violations', [])), 'cells': []})
            elif not route:
                rejected.append({'index': index, 'reason': 'ruta vacía', 'cells': []})
            else:
                accepted.append(index)
        return accepted, rejected


In [ ]:
%%writefile ares_x/src/ares/mission.py
"""Hybrid mission loop.

observar -> validar_sensores -> predecir_riesgo_rna -> inferir_control_difuso ->
replanificar_ag_si_corresponde -> aplicar_restricciones -> ejecutar_accion ->
actualizar_mundo -> registrar_traza

The rover is not omniscient: it plans over a *belief* map that misses hidden
cliffs and part of the obstacles. Those are revealed through sensors (RNA
camera/LIDAR lookahead, deterministic edge sensor) or, worst case, by contact.
"""
from collections import deque
import math
import time

import numpy as np

from .events import Event
from .fuzzy import FuzzyController
from .genetic import GeneticPlanner
from .logger import MissionLogger, clean
from .neural import NeuralPerception
from .planet import ASSUMPTION, PROFILES, PlanetProfile, normalize
from .sensors import SensorSuite
from .supervisor import SafetySupervisor
from .world import World

POLICIES = {
    'hibrido': dict(rna=True, fuzzy=True, ga=True, fixed_risk=False, label='Híbrido completo'),
    'sin_rna': dict(rna=False, fuzzy=True, ga=True, fixed_risk=False, label='Ablación: sin RNA'),
    'sin_difuso': dict(rna=True, fuzzy=False, ga=True, fixed_risk=False, label='Ablación: sin difuso'),
    'sin_ag': dict(rna=True, fuzzy=True, ga=False, fixed_risk=False, label='Ablación: sin AG'),
    'control': dict(rna=False, fuzzy=False, ga=False, fixed_risk=False, label='Control: ruta más corta sin riesgos'),
    'ag_puro': dict(rna=False, fuzzy=False, ga=True, fixed_risk=True, label='AG puro con riesgos fijos'),
}
FIXED_SPEED = .5          # speed used when the fuzzy controller is disabled
RISK_PRIOR = .1           # belief risk of a never-observed cell (supuesto)
FIXED_RISK = .15          # constant estimated risk for the 'ag_puro' baseline
DAY_LENGTH = 30           # steps per half day/night cycle (supuesto)
MAX_STEPS = 400
EVENT_LABELS = {
    'barranco': 'Barranco', 'bateria_baja': 'Batería baja', 'dia': 'Día', 'noche': 'Noche',
    'sismo': 'Sismo', 'meteorito': 'Meteorito', 'tormenta_polvo': 'Tormenta de polvo',
    'corrosion': 'Corrosión', 'biosenal_anomala': 'Señal biológica anómala',
    'alienigena': 'Objeto/ser alienígena (ficción)', 'falla_sensor': 'Falla de sensor',
    'perdida_enlace': 'Pérdida de enlace', 'viento': 'Ráfaga de viento',
}

_NN_CACHE = {}


def shared_perception(seed=0):
    """Train the MLP once per process (offline phase) and reuse it online."""
    if seed not in _NN_CACHE:
        started = time.perf_counter()
        perception = NeuralPerception(seed=seed)
        perception.train()
        perception.train_seconds_ = time.perf_counter() - started
        _NN_CACHE[seed] = perception
    return _NN_CACHE[seed]


def _finite(value):
    return isinstance(value, (int, float, np.number)) and not isinstance(value, bool) and math.isfinite(value)


def validate_config(config):
    """Return a normalized config or raise ValueError with a readable message."""
    cfg = dict(planet='Marte', mode='rover', seed=42, difficulty='medium', battery=100.,
               policy='hibrido', step_duration=1., science_count=5, thermal_limit=70.,
               weights=None, hidden_cliffs=4)
    cfg.update({k: v for k, v in (config or {}).items() if v is not None})
    if normalize(str(cfg['planet'])) not in PROFILES:
        raise ValueError(f"Planeta desconocido: {cfg['planet']}")
    if normalize(str(cfg['mode'])) not in ('rover', 'ruedas', 'sonda', 'probe'):
        raise ValueError('Modo inválido: use rover (ruedas) o sonda')
    for key, low, high in (('battery', 0, 100), ('step_duration', .01, 100), ('thermal_limit', -100, 1000)):
        try:
            cfg[key] = float(cfg[key])
        except (TypeError, ValueError):
            raise ValueError(f'{key}: debe ser un número') from None
        if not math.isfinite(cfg[key]) or not low <= cfg[key] <= high:
            raise ValueError(f'{key}: valor imposible ({cfg[key]}); rango permitido {low}–{high}')
    for key, low, high in (('seed', 0, 2**31 - 1), ('science_count', 0, 12), ('hidden_cliffs', 0, 12)):
        value = cfg[key]
        if isinstance(value, float) and value.is_integer():
            value = int(value)
        if isinstance(value, bool) or not isinstance(value, (int, np.integer)) or not low <= value <= high:
            raise ValueError(f'{key}: debe ser entero entre {low} y {high}')
        cfg[key] = int(value)
    if cfg['difficulty'] not in ('low', 'medium', 'high'):
        raise ValueError('Dificultad inválida: low, medium o high')
    if cfg['policy'] not in POLICIES:
        raise ValueError(f"Política desconocida: {cfg['policy']}")
    if cfg['weights'] is not None:
        weights = {}
        for key, value in dict(cfg['weights']).items():
            try:
                value = float(value)
            except (TypeError, ValueError):
                raise ValueError(f'Peso {key}: debe ser un número') from None
            if not math.isfinite(value) or value < 0:
                raise ValueError(f'Peso {key}: debe ser finito y no negativo')
            weights[key] = value
        if set(weights) - set(GeneticPlanner.default_weights):
            raise ValueError('Componente de fitness desconocido')
        if not any(weights.values()):
            raise ValueError('Los pesos del fitness no pueden ser todos cero')
        cfg['weights'] = weights
    # Raises a scientific explanation for gas giants + wheels (CA-07).
    PlanetProfile.for_body(cfg['planet'], cfg['mode'])
    return cfg


class CachedPlanner(GeneticPlanner):
    """Same GA, with per-run memoized BFS segments (speed only, same results)."""

    def plan(self, world, start, goal, battery=100, estimated_risk=None, **kwargs):
        self._cache = {}
        try:
            return super().plan(world, start, goal, battery, estimated_risk, **kwargs)
        finally:
            self._cache = None

    def _path(self, world, start, goal):
        cache = getattr(self, '_cache', None)
        if cache is None:
            return GeneticPlanner._path(world, start, goal)
        key = (tuple(start), tuple(goal))
        if key not in cache:
            cache[key] = GeneticPlanner._path(world, start, goal)
        return cache[key]


class HybridMission:
    operational_label = ASSUMPTION

    def __init__(self, **config):
        self.config = validate_config(config)
        cfg = self.config
        self.policy = POLICIES[cfg['policy']]
        self.profile = PlanetProfile.for_body(cfg['planet'], cfg['mode'])
        self.seed = cfg['seed']
        self.rng = np.random.default_rng(self.seed + 7)
        self.world = World(16, 12, seed=self.seed, difficulty=cfg['difficulty'], science_count=cfg['science_count'])
        self.home = (0, 0)
        self._add_second_refuge()
        self.hidden_cliffs, self.hidden_obstacles = self._hide_hazards(cfg['hidden_cliffs'])
        self.belief = self._make_belief()
        self.sensors = SensorSuite(self.seed, noise=.05)
        self.perception = shared_perception(0)
        self.fuzzy = FuzzyController()
        self.supervisor = SafetySupervisor()
        self.logger = MissionLogger()
        self.risk = {}
        self.position = self.home
        self.battery = cfg['battery']
        self.health = 100.
        self.daylight, self.visibility = True, 1.
        self.wait_steps = 0
        self.elapsed = 0.
        self.step_index = 0
        self.state = 'CONFIGURACION'
        self.route = None
        self.path = [self.position]
        self.planned_history = []
        self.ga = None
        self.ga_context = None
        self.ga_history_raw = []
        self.ga_runs = []
        self.returning = False
        self.finished = False
        self.outcome = None
        self.replan_reason = 'INICIO'
        self.timers = {'storm': 0, 'link': 0, 'vibration': 0}
        self.sensor_timers = {}
        self.day_counter = 0
        self.no_route_steps = 0
        self.nothing_left = False
        self.pending_events = []
        self.event_log = []
        self.last = {}
        self.metrics = dict(collisions=0, cliffs_avoided=0, hazards_avoided=0, replans=0, ga_runs=0,
                            science_samples=0, safety_violations=0, energy_used=0., distance=0,
                            decision_seconds=0., planning_seconds=0., anomalies=0)

    # ------------------------------------------------------------------ setup
    def _free_cells(self):
        world = self.world
        return [(r, c) for r in range(world.height) for c in range(world.width)
                if world.traversable((r, c)) and (r, c) not in world.refuges]

    def _add_second_refuge(self):
        far = [cell for cell in self._free_cells() if cell[0] + cell[1] >= 18
               and cell not in self.world.science_sites]
        if far:
            self.world.refuges.add(far[int(self.rng.integers(len(far)))])

    def _hide_hazards(self, count):
        world = self.world
        reserved = world.refuges | world.science_sites | {self.home, (0, 1), (1, 0), (1, 1)}
        candidates = [cell for cell in self._free_cells() if cell not in reserved]
        cliffs = set()
        if self.profile.mode == 'rover' and candidates:
            for index in self.rng.permutation(len(candidates))[:count]:
                cell = candidates[int(index)]
                cliffs.add(cell)
                world.cliffs.add(cell)
                world.elevation[cell] -= 10.
        obstacles = sorted(world.obstacles)
        hidden = {obstacles[int(i)] for i in self.rng.permutation(len(obstacles))[:int(len(obstacles) * .3)]}
        return cliffs, hidden

    def _make_belief(self):
        belief = World.__new__(World)
        world = self.world
        belief.width, belief.height, belief.seed, belief.difficulty = world.width, world.height, world.seed, world.difficulty
        belief.rng = None
        belief.elevation = world.elevation.copy()
        for cell in self.hidden_cliffs:
            belief.elevation[cell] += 10.  # orbital map does not resolve the drop
        belief.obstacles = set(world.obstacles) - self.hidden_obstacles
        belief.cliffs = set()
        belief.science_sites = world.science_sites
        belief.visited_science = world.visited_science
        belief.resources = world.resources
        belief.refuges = world.refuges
        return belief

    # -------------------------------------------------------------- helpers
    @property
    def temperature_internal(self):
        return 20. + (self.profile.temperature - 20.) * .1  # thermal control model (supuesto)

    def cell_cost(self, speed=FIXED_SPEED, slope=0.):
        base = .45 * (.7 + .6 * speed) * (1 + .5 * (1 - self.visibility)) + .01 * slope
        return base * self.config['step_duration']

    def _bfs(self, start, goal=None, avoid=frozenset(), goals=None):
        goals = goals or {goal}
        world = self.belief
        queue, previous = deque([start]), {start: None}
        while queue:
            cell = queue.popleft()
            if cell in goals:
                route = []
                while cell is not None:
                    route.append(cell)
                    cell = previous[cell]
                return route[::-1]
            for dr, dc in ((-1, 0), (0, 1), (1, 0), (0, -1)):
                nxt = (cell[0] + dr, cell[1] + dc)
                if nxt in previous or not world.traversable(nxt, self.profile.mode) or (nxt in avoid and nxt not in goals):
                    continue
                if self.profile.mode == 'rover' and world.elevation[nxt] - world.elevation[cell] < -5:
                    continue
                previous[nxt] = cell
                queue.append(nxt)
        return None

    def refuge_route(self):
        return self._bfs(self.position, goals=set(self.world.refuges))

    def energy_to_refuge(self):
        if self.position in self.world.refuges:
            return 0.
        route = self.refuge_route()
        return math.inf if route is None else (len(route) - 1) * self.cell_cost()

    def _risk_avoid(self):
        return {cell for cell, value in self.risk.items() if value >= .5}

    def _hazard_distance(self):
        """Meters to the first known/predicted hazard along the planned route."""
        for k, cell in enumerate((self.route or [])[1:5], start=1):
            known = cell in self.belief.cliffs or cell in self.belief.obstacles
            if known or self.risk.get(cell, 0.) >= .5:
                return float((k - 1) * 5 + 1)
        return 20.

    # ------------------------------------------------------------ perception
    def _features(self, cell, readings):
        distance = readings.get('distance')
        delta = readings.get('elevation_delta')
        if not _finite(distance) or not _finite(delta):
            raise ValueError('Lectura inválida para la RNA')
        noise = 1 + 2 * (1 - self.visibility)
        rng = self.rng
        return {
            'distance': float(np.clip(distance + rng.normal(0, .8 * noise), .01, 20)),
            'slope': float(np.clip(readings['slope'] + rng.normal(0, 2.), 0, 45)),
            'elevation_delta': float(delta + rng.normal(0, 1.3 * noise)),
            'texture': float(np.clip(.25 + .3 * (cell in self.world.obstacles) + rng.normal(0, .22 * noise), 0, 1)),
            'visibility': float(np.clip(self.visibility, .35, 1.)),
            'vibration': float(np.clip(abs(delta) * .15 + rng.normal(0, .15) + (1.5 if self.timers['vibration'] else 0), 0, 20)),
        }

    def _observe(self, cell):
        return self.sensors.observe(self.world, self.position, cell, self.battery, self.profile,
                                    self.daylight, self.visibility)

    # ------------------------------------------------------------- planning
    def _plan(self, reason):
        started = time.perf_counter()
        self.metrics['replans'] += 1
        self.replan_reason = None
        if self.returning:
            route = self.refuge_route()
            self.route = route if route else [self.position]
            self.ga = dict(reason=reason, mode='retorno', candidates=[], rejected=[], history=[],
                           message='Ruta directa al refugio más cercano.')
        elif self.policy['ga']:
            self._plan_ga(reason)
        else:
            self._plan_greedy(reason)
        self.ga['step'] = self.step_index
        self.planned_history.append(list(self.route))
        self.metrics['planning_seconds'] += time.perf_counter() - started

    def risk_map(self):
        cells = [(r, c) for r in range(self.belief.height) for c in range(self.belief.width)]
        if self.policy['fixed_risk']:
            base = {cell: FIXED_RISK for cell in cells}
        else:
            base = {cell: RISK_PRIOR for cell in cells}
            base.update({cell: float(np.clip(v, 0, 1)) for cell, v in self.risk.items()})
        for cell in self.belief.cliffs | self.belief.obstacles:
            base[cell] = 1.
        return base

    def _plan_ga(self, reason):
        self.metrics['ga_runs'] += 1
        planner = CachedPlanner(population_size=30, generations=30, waypoints=3,
                                seed=self.seed + self.metrics['ga_runs'], weights=self.config['weights'])
        goal = self.home
        risk = self.risk_map()
        kwargs = dict(temperature=self.temperature_internal, thermal_limit=self.config['thermal_limit'],
                      energy_per_step=self.cell_cost())
        result = planner.plan(self.belief, self.position, goal, battery=self.battery, estimated_risk=risk, **kwargs)
        accepted, rejected = self.supervisor.filter_routes(result['candidates'], self.belief.cliffs, self.belief.obstacles)
        chosen = accepted[0] if accepted else None
        self.route = list(result['candidates'][chosen]['route']) if chosen is not None else [self.position]
        if chosen is None or (not result['candidates'][chosen]['science_order'] and self.position in self.world.refuges):
            self.nothing_left = self.position in self.world.refuges
        self.ga_context = dict(planner=planner, goal=goal, battery=self.battery, risk=risk, kwargs=kwargs,
                               start=self.position)
        self.ga_history_raw = result['history']
        self.ga = dict(
            reason=reason, mode='ag', chosen=chosen, rejected=rejected, feasible=result['feasible'],
            message=result['message'], generations_run=result['generations_run'], stopped_early=result['stopped_early'],
            candidates=[dict(index=i, route=[list(p) for p in c['route']], fitness=round(c['fitness'], 4),
                             feasible=c['feasible'], components={k: round(v, 4) for k, v in c['components'].items()},
                             penalty=round(c['penalty'], 3), violations=c['violations'],
                             science_order=[list(p) for p in c['science_order']],
                             chromosome=[list(p) for p in c['chromosome']], distance=c['distance'])
                        for i, c in enumerate(result['candidates'])],
            history=[dict(generation=h['generation'], best=round(h['best_fitness'], 4),
                          best_feasible=None if h['best_feasible_fitness'] is None else round(h['best_feasible_fitness'], 4),
                          tournaments=h['tournaments'], crossovers=h['crossovers'], mutations=h['mutations'])
                     for h in result['history']],
            weights=dict(planner.weights))
        self.ga_runs.append(dict(ga=self.ga, raw=self.ga_history_raw, ctx=self.ga_context,
                                 belief=(set(self.belief.obstacles), set(self.belief.cliffs)),
                                 step=self.step_index))

    def ga_run(self, run=None):
        if not self.ga_runs:
            raise ValueError('Aún no hay ejecuciones del AG')
        run = len(self.ga_runs) - 1 if run is None else max(0, min(int(run), len(self.ga_runs) - 1))
        return run, self.ga_runs[run]

    def _plan_greedy(self, reason):
        """Nearest-neighbour tour over remaining sites with a battery check (no GA)."""
        avoid = self._risk_avoid() if self.policy['rna'] else set()
        remaining = sorted(self.world.science_sites - self.world.visited_science)
        position, route, order = self.position, [self.position], []
        cost = self.cell_cost()
        while remaining:
            options = [(self._bfs(position, site, avoid), site) for site in remaining]
            options = [(p, s) for p, s in options if p]
            if not options:
                break
            path, site = min(options, key=lambda o: (len(o[0]), o[1]))
            back = self._bfs(site, goals=set(self.world.refuges), avoid=avoid)
            if back is None or (len(route) - 1 + len(path) - 1 + len(back) - 1) * cost > self.battery:
                break
            route.extend(path[1:])
            order.append(site)
            remaining.remove(site)
            position = site
        back = self._bfs(position, self.home, avoid) or self._bfs(position, goals=set(self.world.refuges), avoid=avoid)
        if back:
            route.extend(back[1:])
        self.route = route
        if not order and self.position in self.world.refuges:
            self.nothing_left = True
        self.ga = dict(reason=reason, mode='voraz', chosen=None, rejected=[], candidates=[], history=[],
                       message='Planificador voraz (sin AG): vecino más cercano.',
                       science_order=[list(p) for p in order])

    def ga_generation(self, generation, run=None):
        """Chromosomes of one generation re-evaluated with that run's planning context."""
        run, record = self.ga_run(run)
        raw, ctx = record['raw'], record['ctx']
        generation = max(0, min(int(generation), len(raw) - 1))
        entry = raw[generation]
        # Re-evaluate against the belief known when that plan was made.
        snapshot = World.__new__(World)
        snapshot.__dict__.update(self.belief.__dict__)
        snapshot.obstacles, snapshot.cliffs = set(record['belief'][0]), set(record['belief'][1])
        rows = []
        for chromosome in entry['chromosomes'][:8]:
            result = ctx['planner'].evaluate(chromosome, snapshot, ctx['start'], ctx['goal'], ctx['battery'],
                                             ctx['risk'], **ctx['kwargs'])
            rows.append(dict(chromosome=[list(p) for p in chromosome], fitness=round(result['fitness'], 4),
                             feasible=result['feasible'], violations=result['violations'],
                             components={k: round(v, 4) for k, v in result['components'].items()}))
        return dict(generation=generation, total=len(raw), rows=rows, run=run, runs=len(self.ga_runs),
                    tournaments=entry['tournaments'], crossovers=entry['crossovers'], mutations=entry['mutations'],
                    best=round(entry['best_fitness'], 4))

    # ---------------------------------------------------------------- events
    def inject(self, kind, **data):
        position = data.pop('position', None)
        event = Event(kind, data)
        kind = event.kind
        if position is not None:
            position = tuple(int(v) for v in position)
        if kind in ('barranco', 'meteorito', 'alienigena'):
            if kind == 'barranco' and self.profile.mode != 'rover':
                raise ValueError('Barranco no aplica al modo sonda')
            if position is None:
                route = self.route or [self.position]
                offset = {'barranco': 1, 'meteorito': 3, 'alienigena': 2}[kind]
                position = route[min(offset, len(route) - 1)] if len(route) > 1 else None
                if position is None or position in self.world.refuges:
                    neighbors = [(self.position[0] + dr, self.position[1] + dc) for dr, dc in ((0, 1), (1, 0), (0, -1), (-1, 0))]
                    neighbors = [n for n in neighbors if self.world.traversable(n) and n not in self.world.refuges]
                    if not neighbors:
                        raise ValueError('No hay celda libre adyacente para el evento')
                    position = neighbors[0]
            if not self.world.in_bounds(position):
                raise ValueError('Evento fuera del mapa')
            if position == self.position or position in self.world.refuges:
                raise ValueError('No se puede colocar un peligro sobre el rover o un refugio')
            event.data['position'] = position
        if kind == 'falla_sensor':
            sensor = data.get('sensor', 'distance')
            names = ['distance', 'elevation_delta'] if sensor == 'todos' else [sensor]
            for name in names:
                self.sensors.fail(name, data.get('failure', 'nan'))
                self.sensor_timers[name] = 8
            pretty = {'distance': 'proximidad', 'elevation_delta': 'LIDAR'}
            result = {'kind': kind, 'message': 'Falla simulada de sensor ' + ', '.join(pretty.get(n, n) for n in names)
                      + ' (NaN) durante 8 pasos: degradación segura.'}
        else:
            result = event.apply(self)
        if kind == 'barranco':
            self.world.elevation[position] += 10.  # Event lowers 20 m; keep a 10 m drop
        elif kind == 'meteorito':
            self.belief.add_obstacle(position)
            for dr in (-1, 0, 1):
                for dc in (-1, 0, 1):
                    cell = (position[0] + dr, position[1] + dc)
                    if self.world.in_bounds(cell) and cell != self.position:
                        self.risk[cell] = max(self.risk.get(cell, 0.), .6)
            self.timers['vibration'] = 2
            self.replan_reason = 'METEORITO'
        elif kind == 'sismo':
            self.wait_steps = 2
            self.timers['vibration'] = 3
            self.replan_reason = 'SISMO'
        elif kind == 'tormenta_polvo':
            self.timers['storm'] = 15
        elif kind in ('dia', 'noche'):
            self.day_counter = 0
        elif kind == 'perdida_enlace':
            self.timers['link'] = 10
        elif kind == 'biosenal_anomala':
            self.metrics['anomalies'] += 1
        elif kind == 'viento' and self.profile.mode == 'probe':
            self.replan_reason = 'VIENTO'
        entry = dict(step=self.step_index, kind=kind, label=EVENT_LABELS.get(kind, kind),
                     message=result.get('message', kind), position=list(position) if position else None)
        self.event_log.append(entry)
        self.pending_events.append(entry)
        return entry

    def _tick_environment(self):
        self.day_counter += 1
        if self.day_counter >= DAY_LENGTH:
            self.day_counter = 0
            self.daylight = not self.daylight
        if self.timers['storm']:
            self.timers['storm'] -= 1
            self.visibility = .2
        else:
            self.visibility = 1. if self.daylight else .6
        for key in ('link', 'vibration'):
            if self.timers[key]:
                self.timers[key] -= 1
        for name in list(self.sensor_timers):
            self.sensor_timers[name] -= 1
            if self.sensor_timers[name] <= 0:
                del self.sensor_timers[name]
                self.sensors.failures.pop(name, None)
        if self.daylight and self.profile.mode == 'rover':
            self.battery = min(100., self.battery + .12 * self.visibility)  # solar (supuesto)
        if self.profile.name == 'Venus':
            self.health = max(0., self.health - .4)  # corrosion (supuesto)

    # ------------------------------------------------------------------ step
    def step(self):
        if self.finished:
            return self.last.get('record')
        started = time.perf_counter()
        self.step_index += 1
        self.elapsed += self.config['step_duration']
        depleted = self.battery <= 0  # 0 % means end of mission: solar cannot revive it
        self._tick_environment()
        if depleted:
            self.battery = 0.
        complete = self.position in self.world.refuges and self.step_index > 1 and not self.returning and (
            self.world.science_progress == 1 or self.nothing_left)
        if not complete:
            if self.route is None or (self.replan_reason and self.replan_reason not in ('RECARGA_COMPLETA',)):
                self._plan(self.replan_reason or 'INICIO')
            elif self.replan_reason == 'RECARGA_COMPLETA':
                self._plan('RECARGA_COMPLETA')
            if self.route and len(self.route) <= 1 and not (self.returning and self.position in self.world.refuges):
                self._plan('OBJETIVO_ALCANZADO')
                complete = self.position in self.world.refuges and self.nothing_left and self.step_index > 1
        next_cell = self.route[1] if self.route and len(self.route) > 1 else None

        # 1-2. observe + validate
        readings = self._observe(next_cell or self.position)
        nav_valid = {'proximidad': _finite(readings.get('distance')) and 0 <= readings['distance'] <= 20,
                     'lidar': _finite(readings.get('elevation_delta'))}
        edge = bool(readings.get('cliff')) if next_cell and nav_valid['lidar'] and self.profile.mode == 'rover' else None

        # 3. RNA lookahead (next two cells)
        rna, rna_error = None, None
        if self.policy['rna'] and next_cell:
            for k, cell in enumerate(self.route[1:3]):
                obs = readings if k == 0 else self._observe(cell)
                try:
                    features = self._features(cell, obs)
                    prediction = self.perception.predict(features)
                except ValueError as error:
                    rna_error = str(error)
                    break
                self.risk[cell] = float(np.clip(prediction['risk'], 0, 1))
                if k == 0:
                    rna = dict(prediction, features=features, cell=list(cell))
                elif prediction['label'] != 'libre' and prediction['confidence'] >= self.supervisor.confidence_threshold:
                    # Early warning two cells ahead: mark the cell and replan next step.
                    (self.belief.cliffs if prediction['label'] == 'barranco' else self.belief.obstacles).add(cell)
                    self.replan_reason = 'RNA_ANTICIPA_PELIGRO'
                    self.metrics['hazards_avoided'] += 1

        # 4. fuzzy control
        slope = readings['slope'] if _finite(readings.get('slope')) else 20.
        hazard = self._hazard_distance() if nav_valid['proximidad'] else 2.
        if edge:
            hazard = 1.
        fuzzy = None
        if self.policy['fuzzy']:
            fuzzy = self.fuzzy.infer(hazard, float(np.clip(self.battery, 0, 100)), float(np.clip(slope, 0, 45)),
                                     float(np.clip(self.temperature_internal, -273.15, 1000)))
            fuzzy['inputs'] = dict(distance=hazard, battery=round(self.battery, 2), slope=round(slope, 2),
                                   temperature=round(self.temperature_internal, 2))
            speed = fuzzy['speed']
        else:
            speed = FIXED_SPEED

        # 5. supervisor
        at_refuge = self.position in self.world.refuges
        ctx = dict(battery=self.battery, health=self.health, temperature_internal=self.temperature_internal,
                   thermal_limit=self.config['thermal_limit'], nav_valid=nav_valid, edge_cliff=edge,
                   rna=rna, rna_used=self.policy['rna'], fuzzy_speed=speed if self.policy['fuzzy'] else None,
                   has_route=next_cell is not None,
                   needs_replan=self.replan_reason is not None and self.replan_reason != 'RECARGA_COMPLETA',
                   quake_wait=self.wait_steps, at_refuge=at_refuge, energy_to_refuge=self.energy_to_refuge(),
                   recharge_target=90. if self.returning else 0., returning=self.returning,
                   mission_complete=complete)
        decision = self.supervisor.decide(ctx)
        moved, collision, cost_total = [], None, 0.

        # 6. act
        action = decision.action
        if action == 'replanificar':
            if decision.reason == 'R04_VETO_BARRANCO_CONFIRMADO':
                self.belief.cliffs.add(next_cell)
                self.metrics['cliffs_avoided'] += 1
            elif decision.reason == 'R05_RNA_PELIGRO':
                (self.belief.cliffs if rna['label'] == 'barranco' else self.belief.obstacles).add(next_cell)
                self.metrics['hazards_avoided'] += 1
            self._plan(decision.reason)
        elif action == 'volver':
            self.returning = True
            self._plan('RETORNO_ENERGIA')
        elif action == 'recargar':
            self.battery = min(100., self.battery + 6.)
            if self.battery >= 90.:
                self.returning = False
                self.replan_reason = 'RECARGA_COMPLETA'
        elif action == 'avanzar':
            effective = min(speed, decision.speed_cap)
            cells = 2 if (effective >= .65 and len(self.route) > 2
                          and self.risk.get(self.route[2], RISK_PRIOR) < .3) else 1
            for _ in range(cells):
                if len(self.route) < 2:
                    break
                cell = self.route[1]
                cost = self.cell_cost(effective, slope)
                if cell in self.world.cliffs and self.profile.mode == 'rover':
                    self.health = 0.
                    self.metrics['safety_violations'] += 1
                    collision = 'caida_barranco'
                    break
                if cell in self.world.obstacles:
                    self.health = max(0., self.health - 25.)
                    self.metrics['collisions'] += 1
                    self.belief.obstacles.add(cell)
                    self.battery = max(0., self.battery - cost / 2)
                    collision = 'colision'
                    self.replan_reason = 'COLISION'
                    break
                if cost > self.battery:
                    break
                self.battery = max(0., self.battery - cost)
                cost_total += cost
                self.position = cell
                self.route.pop(0)
                self.path.append(cell)
                moved.append(cell)
                self.metrics['distance'] += 1
                if cell in self.world.science_sites and cell not in self.world.visited_science:
                    self.world.visited_science.add(cell)
                    self.metrics['science_samples'] += 1
        if action not in ('avanzar', 'recargar'):
            idle = .05 * self.config['step_duration']
            self.battery = max(0., self.battery - idle)
            cost_total += idle
        self.metrics['energy_used'] += cost_total
        if self.wait_steps and decision.reason == 'R08_ESPERA_SISMO':
            self.wait_steps -= 1
        self.no_route_steps = self.no_route_steps + 1 if decision.reason == 'R10_SIN_RUTA' else 0

        # 7. update mission state
        self.state = decision.state
        if collision == 'caida_barranco' or self.health <= 0 or (self.battery <= 0 and self.position not in self.world.refuges):
            self.state, self.finished = 'FRACASO', True
            self.outcome = 'Rover perdido: ' + ('caída en barranco' if collision == 'caida_barranco' else 'recursos agotados')
        elif decision.reason == 'R15_MISION_COMPLETA':
            self.finished, self.outcome = True, 'Misión cumplida: objetivos alcanzables y retorno seguro'
        elif decision.reason in ('R06_REFUGIO_INALCANZABLE', 'R01_LIMITE_TERMICO', 'R00_RECURSOS_AGOTADOS'):
            self.finished, self.outcome = True, decision.explanation
            self.state = 'FRACASO' if decision.reason == 'R00_RECURSOS_AGOTADOS' else 'MODO_SEGURO'
        elif self.no_route_steps >= 10:
            self.state, self.finished, self.outcome = 'FRACASO', True, 'Sin solución: no existe ruta factible'
        elif self.step_index >= MAX_STEPS:
            self.state, self.finished, self.outcome = 'FRACASO', True, 'Tiempo extremo agotado'
        elif collision:
            self.state = 'ALERTA'
        self.metrics['decision_seconds'] += time.perf_counter() - started

        # 8. trace
        events, self.pending_events = self.pending_events, []
        explanation = decision.explanation
        if collision == 'colision':
            explanation += ' Contacto con obstáculo no detectado: daño -25 %, se replanifica.'
        if self.timers['link']:
            explanation += ' Enlace perdido: autonomía local, órdenes en cola.'
        ga_now = self.ga if self.ga is not None and self.ga.get('step') == self.step_index else None
        record = self.logger.record(
            step=self.step_index, t=round(self.elapsed, 2), policy=self.config['policy'], state=self.state,
            position=list(self.position), moved=[list(c) for c in moved], battery=round(self.battery, 3),
            health=round(self.health, 2), action=action, reason=decision.reason, explanation=explanation,
            vetoes=decision.vetoes, permitted_by=decision.permitted_by,
            sensors={k: readings.get(k) for k in ('distance', 'elevation_delta', 'slope', 'temperature_external',
                                                     'visibility', 'solar_power', 'pressure', 'oxygen_ambient',
                                                     'radiation', 'link_available')},
            nav_valid=nav_valid, edge_cliff=edge,
            rna=None if rna is None else dict(label=rna['label'], confidence=round(rna['confidence'], 4),
                                              probabilities={k: round(v, 4) for k, v in rna['probabilities'].items()},
                                              low_confidence=rna['low_confidence']),
            rna_error=rna_error,
            fuzzy=None if fuzzy is None else dict(speed=round(fuzzy['speed'], 4), caution=round(fuzzy['caution'], 4),
                                                  rules=[[r['id'], round(r['strength'], 3)] for r in fuzzy['activated_rules']]),
            crisp_speed=round(speed, 4), collision=collision,
            ga=None if ga_now is None else dict(
                mode=ga_now['mode'], reason=ga_now['reason'], chosen=ga_now.get('chosen'),
                best_fitness=(ga_now['candidates'][ga_now['chosen']]['fitness']
                              if ga_now.get('chosen') is not None else None),
                rejected=ga_now.get('rejected')),
            route=[list(c) for c in (self.route or [])[:12]], events=events,
            science=round(self.world.science_progress, 3), seed=self.seed, operational_label=ASSUMPTION)
        self.last = dict(record=record, rna=rna, fuzzy=fuzzy, decision=decision.to_dict(), readings=readings,
                         nav_valid=nav_valid, speed=speed)
        return record

    def run(self, max_steps=MAX_STEPS):
        while not self.finished and self.step_index < max_steps:
            self.step()
        return self.summary()

    # ------------------------------------------------------------- reporting
    def summary(self):
        m = self.metrics
        return dict(policy=self.config['policy'], seed=self.seed, planet=self.profile.name,
                    success=self.state == 'EXITO', state=self.state, outcome=self.outcome,
                    steps=self.step_index, collisions=m['collisions'], cliffs_avoided=m['cliffs_avoided'],
                    hazards_avoided=m['hazards_avoided'], battery_left=round(self.battery, 3),
                    energy_used=round(m['energy_used'], 3), distance=m['distance'],
                    science=round(self.world.science_progress, 3), samples=m['science_samples'],
                    replans=m['replans'], ga_runs=m['ga_runs'], safety_violations=m['safety_violations'],
                    decision_ms=round(1000 * m['decision_seconds'] / max(1, self.step_index), 3),
                    planning_s=round(m['planning_seconds'], 3), health=round(self.health, 2))

    def export_json(self):
        return clean(dict(version='1.0', operational_label=ASSUMPTION, config=self.config,
                          summary=self.summary(), events=self.event_log, trace=self.logger.trace))

    def export_csv(self):
        return self.logger.export_csv()


In [ ]:
%%writefile ares_x/src/ares/experiments.py
"""Repeated experiments: >= 20 seeds per scenario, ablations and baselines (RF-14, RF-15)."""
import time

import numpy as np
import pandas as pd

from .mission import POLICIES, HybridMission

SCENARIOS = {
    'estandar': dict(label='Estándar (peligros ocultos)', events=[]),
    'eventos': dict(label='Con eventos (barranco, tormenta, sismo, falla de sensor)',
                    events=[(3, 'barranco', {}), (8, 'tormenta_polvo', {}), (14, 'sismo', {}),
                            (20, 'falla_sensor', {'sensor': 'distance'}), (26, 'alienigena', {})]),
    'bateria_baja': dict(label='Batería inicial 25 %', events=[], battery=25.),
}
METRICS = ['success', 'collisions', 'cliffs_avoided', 'hazards_avoided', 'battery_left', 'distance',
           'science', 'replans', 'safety_violations', 'decision_ms', 'steps']


def run_one(policy, seed, scenario='estandar', **overrides):
    spec = SCENARIOS[scenario]
    config = dict(policy=policy, seed=seed, battery=spec.get('battery', 100.), hidden_cliffs=6, difficulty='medium')
    config.update(overrides)
    mission = HybridMission(**config)
    schedule = sorted(spec['events'], key=lambda e: e[0])
    started = time.perf_counter()
    while not mission.finished and mission.step_index < 400:
        for at, kind, data in schedule:
            if mission.step_index == at:
                try:
                    mission.inject(kind, **dict(data))
                except ValueError:
                    pass  # event not applicable in this state (e.g. no free cell)
        mission.step()
    row = mission.summary()
    row.update(scenario=scenario, wall_s=round(time.perf_counter() - started, 3))
    return row


def run_experiments(policies=None, seeds=20, scenario='estandar', first_seed=0, progress=None, **overrides):
    policies = list(policies or POLICIES)
    unknown = set(policies) - set(POLICIES)
    if unknown:
        raise ValueError(f'Políticas desconocidas: {sorted(unknown)}')
    if not isinstance(seeds, int) or not 1 <= seeds <= 200:
        raise ValueError('Cantidad de semillas inválida (1–200)')
    rows = []
    for policy in policies:
        for seed in range(first_seed, first_seed + seeds):
            rows.append(run_one(policy, seed, scenario, **overrides))
            if progress:
                progress(len(rows), len(policies) * seeds)
    return pd.DataFrame(rows)


def summarize(frame):
    """Mean, median and standard deviation per policy (dispersion is reported)."""
    frame = frame.copy()
    frame['success'] = frame['success'].astype(float)
    grouped = frame.groupby('policy')[METRICS]
    table = pd.concat({'media': grouped.mean(), 'mediana': grouped.median(), 'desv': grouped.std(ddof=1).fillna(0.)}, axis=1)
    order = [p for p in POLICIES if p in table.index]
    return table.loc[order]


def summary_records(frame):
    table = summarize(frame)
    records = []
    for policy, row in table.iterrows():
        entry = {'policy': policy, 'label': POLICIES[policy]['label'], 'n': int((frame['policy'] == policy).sum())}
        for metric in METRICS:
            entry[metric] = {stat: float(np.round(row[(stat, metric)], 4)) for stat in ('media', 'mediana', 'desv')}
        records.append(entry)
    return records


In [ ]:
%%writefile ares_x/src/ares/api.py
"""JSON command surface shared by the Colab bridge, the local server and Pyodide.

Every UI action goes through ``Session.dispatch(fn, args)`` so the dashboard is
just a view: the AI engine and the simulation live in Python.
"""
import json
import math

import numpy as np

from .experiments import SCENARIOS, run_experiments, summary_records
from .fuzzy import FuzzyController
from .logger import clean
from .mission import EVENT_LABELS, POLICIES, HybridMission, shared_perception, validate_config
from .neural import FEATURES
from .planet import ASSUMPTION, PROFILES, PlanetProfile
from .supervisor import REASONS

VERSION = '1.0.0'


def _r(value, digits=3):
    if isinstance(value, (float, np.floating)):
        return round(float(value), digits) if math.isfinite(value) else None
    return value


def _cells(cells):
    return sorted([int(r), int(c)] for r, c in cells)


def fuzzy_sets():
    """Sampled membership curves for every linguistic variable (for plots)."""
    controller = FuzzyController()
    universes = {'distance': (0, 20), 'battery': (0, 100), 'slope': (0, 45), 'temperature': (-80, 120)}
    out = {}
    for name, sets in controller.membership_sets.items():
        low, high = universes[name]
        xs = np.linspace(low, high, 61)
        out[name] = {'x': [_r(x, 2) for x in xs],
                     'sets': {label: [_r(controller.membership(float(x), spec), 3) for x in xs]
                              for label, spec in sets.items()}}
    xs = np.linspace(0, 1, 61)
    out['speed'] = {'x': [_r(x, 3) for x in xs],
                    'sets': {label: [_r(controller.membership(float(x), spec), 3) for x in xs]
                             for label, spec in controller.output_sets.items()}}
    return out


def fuzzy_payload(result, controller=None):
    controller = controller or FuzzyController()
    descriptions = {rule['id']: rule for rule in controller.rules}
    curve = result.get('aggregated_curve') or []
    return dict(
        speed=_r(result['speed'], 4), caution=_r(result['caution'], 4), centroid=_r(result['centroid'], 4),
        reason=result['reason'], inputs=result.get('inputs'),
        degrees={k: {s: _r(v, 3) for s, v in d.items()} for k, d in result.get('membership_degrees', {}).items()},
        rules=[dict(id=r['id'], strength=_r(r['strength'], 3), description=r['description'], consequent=r['consequent'])
               for r in result.get('activated_rules', [])],
        all_rules=[dict(id=r['id'], description=r['description'], consequent=r['consequent'],
                        antecedents=[list(a) for a in r['antecedents']]) for r in descriptions.values()],
        curve=[_r(v, 3) for v in curve[::4]], fallback='Sin reglas activas: velocidad 0 (parar), cautela 1.')


def ga_view(ga):
    ga = ga or {}
    return dict(mode=ga.get('mode'), reason=ga.get('reason'), chosen=ga.get('chosen'), message=ga.get('message'),
                rejected=ga.get('rejected', []), generations_run=ga.get('generations_run'),
                stopped_early=ga.get('stopped_early'), weights=ga.get('weights'), step=ga.get('step'),
                history=ga.get('history', []),
                candidates=[{k: v for k, v in c.items() if k != 'route'} for c in ga.get('candidates', [])])


class Session:
    def __init__(self, **config):
        self.config = validate_config(config)
        self.mission = HybridMission(**self.config)
        self.comparison = None

    # ---------------------------------------------------------------- views
    def snapshot(self):
        m = self.mission
        world, belief = m.world, m.belief
        hidden_obstacles = world.obstacles - belief.obstacles
        hidden_cliffs = world.cliffs - belief.cliffs
        ga = m.ga or {}
        last = m.last
        trace = m.logger.trace
        perception = m.perception
        rna = last.get('rna')
        decision = last.get('decision')
        readings = last.get('readings') or {}
        fuzzy = last.get('fuzzy')
        log = [dict(step=t['step'], state=t['state'], action=t['action'], reason=t['reason'],
                    battery=t['battery'], explanation=t['explanation'],
                    events=[e['label'] for e in t['events']],
                    rna=(t['rna'] or {}).get('label'), conf=(t['rna'] or {}).get('confidence'),
                    speed=t['crisp_speed'], ga=(t['ga'] or {}).get('best_fitness')) for t in trace[-60:]]
        return clean(dict(
            version=VERSION, operational_label=ASSUMPTION, config=self.config,
            policy_label=POLICIES[self.config['policy']]['label'],
            planet=dict(name=m.profile.name, mode=m.profile.mode, gravity=m.profile.gravity,
                        temperature=m.profile.temperature, pressure=m.profile.pressure,
                        oxygen=m.profile.oxygen, hazards=list(m.profile.hazards)),
            status=dict(step=m.step_index, t=m.elapsed, state=m.state, finished=m.finished, outcome=m.outcome,
                        position=list(m.position), battery=_r(m.battery, 2), health=_r(m.health, 2),
                        daylight=m.daylight, visibility=m.visibility, returning=m.returning,
                        science=_r(world.science_progress, 3), link=m.timers['link'] == 0,
                        storm=m.timers['storm'], quake=m.wait_steps,
                        sensor_failures=sorted(m.sensors.failures),
                        temperature_internal=_r(m.temperature_internal, 1)),
            map=dict(width=world.width, height=world.height,
                     elevation=[[_r(v, 2) for v in row] for row in world.elevation.tolist()],
                     obstacles=_cells(belief.obstacles), hidden_obstacles=_cells(hidden_obstacles),
                     cliffs=_cells(belief.cliffs), hidden_cliffs=_cells(hidden_cliffs),
                     science=[[int(r), int(c), (r, c) in world.visited_science] for r, c in sorted(world.science_sites)],
                     refuges=_cells(world.refuges), resources=_cells(world.resources),
                     risk=[[int(r), int(c), _r(v, 2)] for (r, c), v in sorted(m.risk.items()) if v >= .15],
                     path=[list(p) for p in m.path], route=[list(p) for p in (m.route or [])],
                     candidates=[c['route'] for c in ga.get('candidates', [])][:3],
                     chosen=ga.get('chosen'),
                     events=[dict(kind=e['kind'], position=e['position']) for e in m.event_log if e.get('position')]),
            telemetry=dict(
                distance=_r(readings.get('distance')), elevation_delta=_r(readings.get('elevation_delta')),
                slope=_r(readings.get('slope')), temperature_external=_r(readings.get('temperature_external'), 1),
                radiation=_r(readings.get('radiation')), pressure=_r(readings.get('pressure'), 1),
                oxygen_ambient=_r(readings.get('oxygen_ambient')), oxygen_stored=_r(readings.get('oxygen_stored')),
                co2=_r(readings.get('co2')), solar_power=_r(readings.get('solar_power')),
                nav_valid=last.get('nav_valid'), edge_cliff=(trace[-1]['edge_cliff'] if trace else None)),
            decision=decision,
            rna=None if rna is None else dict(
                label=rna['label'], confidence=_r(rna['confidence'], 4), low_confidence=rna['low_confidence'],
                probabilities={k: _r(v, 4) for k, v in rna['probabilities'].items()}, cell=rna.get('cell'),
                features={k: _r(v, 3) for k, v in rna['features'].items()},
                normalized=[_r(v, 3) for v in rna['normalized_input']],
                layers=[{k: _r(v, 3) for k, v in s.items()} for s in rna['activation_summaries']]),
            rna_error=(trace[-1]['rna_error'] if trace else None),
            fuzzy=None if fuzzy is None else fuzzy_payload(fuzzy, m.fuzzy),
            speed=_r(last.get('speed'), 4),
            ga=ga_view(ga),
            ga_runs=[dict(index=i, reason=r['ga']['reason'], step=r['step'],
                          generations=r['ga'].get('generations_run')) for i, r in enumerate(m.ga_runs)],
            metrics=m.summary(),
            series=dict(battery=[t['battery'] for t in trace][-200:],
                        risk=[_r(1 - (t['rna'] or {}).get('probabilities', {}).get('libre', 1.), 3) for t in trace][-200:],
                        speed=[t['crisp_speed'] for t in trace][-200:]),
            log=log,
            nn=dict(test=perception.report_['test'], validation=perception.report_['validation'],
                    converged=perception.report_['converged'], iterations=perception.report_['iterations'],
                    training_samples=perception.report_['training_samples'],
                    split_maps=perception.report_['split_maps'],
                    train_seconds=_r(getattr(perception, 'train_seconds_', None), 2)),
            comparison=self.comparison))

    def meta(self):
        return clean(dict(version=VERSION, planets=[dict(key=k, name=v[0], rocky=v[1]) for k, v in PROFILES.items()],
                          policies={k: v['label'] for k, v in POLICIES.items()}, events=EVENT_LABELS,
                          scenarios={k: v['label'] for k, v in SCENARIOS.items()}, reasons=REASONS,
                          features=list(FEATURES), fuzzy_sets=fuzzy_sets(), operational_label=ASSUMPTION))

    # ------------------------------------------------------------- commands
    def dispatch(self, fn, args=None):
        args = dict(args or {})
        try:
            handler = getattr(self, 'cmd_' + str(fn))
        except AttributeError:
            return {'error': f'Comando desconocido: {fn}'}
        try:
            return handler(**args)
        except (ValueError, TypeError, KeyError) as error:
            message = error.args[0] if error.args else str(error)
            return {'error': str(message)}

    def cmd_meta(self):
        return self.meta()

    def cmd_state(self):
        return self.snapshot()

    def cmd_configure(self, **config):
        merged = dict(self.config)
        merged.update(config)
        config = validate_config(merged)  # raises readable ValueError (CA-07, NaN sliders)
        self.config = config
        self.mission = HybridMission(**config)
        return self.snapshot()

    def cmd_validate(self, **config):
        merged = dict(self.config)
        merged.update(config)
        validate_config(merged)
        return {'ok': True}

    def cmd_reset(self):
        self.mission = HybridMission(**self.config)
        return self.snapshot()

    def cmd_step(self, n=1):
        n = int(n)
        if not 1 <= n <= 400:
            raise ValueError('Pasos inválidos (1–400)')
        for _ in range(n):
            if self.mission.finished:
                break
            self.mission.step()
        return self.snapshot()

    def cmd_inject(self, kind, **data):
        self.mission.inject(kind, **data)
        return self.snapshot()

    def cmd_place(self, position, kind='obstaculo'):
        """Operator edits the real terrain (RF-02); the rover must discover it."""
        m = self.mission
        cell = tuple(int(v) for v in position)
        if not m.world.in_bounds(cell):
            raise ValueError('Celda fuera del mapa')
        if cell == m.position or cell in m.world.refuges:
            raise ValueError('No se puede colocar sobre el rover o un refugio')
        if kind == 'barranco':
            m.inject('barranco', position=list(cell))
        elif kind == 'obstaculo':
            m.world.add_obstacle(cell)
            m.event_log.append(dict(step=m.step_index, kind='obstaculo_manual', label='Obstáculo manual',
                                    message='Obstáculo colocado por el operador (oculto al rover).', position=list(cell)))
        elif kind == 'limpiar':
            m.world.obstacles.discard(cell)
            m.world.cliffs.discard(cell)
            m.belief.obstacles.discard(cell)
            m.belief.cliffs.discard(cell)
        else:
            raise ValueError('Tipo de celda desconocido')
        return self.snapshot()

    def cmd_fuzzy_eval(self, distance, battery, slope, temperature=20.):
        controller = FuzzyController()
        values = [float(v) for v in (distance, battery, slope, temperature)]
        result = controller.infer(*values)
        result['inputs'] = dict(distance=values[0], battery=values[1], slope=values[2], temperature=values[3])
        return clean(fuzzy_payload(result, controller))

    def cmd_rna_eval(self, **features):
        perception = shared_perception(0)
        values = {name: float(features[name]) for name in FEATURES}
        prediction = perception.predict(values)
        return clean(dict(label=prediction['label'], confidence=_r(prediction['confidence'], 4),
                          low_confidence=prediction['low_confidence'],
                          probabilities={k: _r(v, 4) for k, v in prediction['probabilities'].items()},
                          normalized=[_r(v, 3) for v in prediction['normalized_input']],
                          layers=[{k: _r(v, 3) for k, v in s.items()} for s in prediction['activation_summaries']],
                          risk=_r(prediction['risk'], 4)))

    def cmd_ga_generation(self, generation=0, run=None):
        return clean(self.mission.ga_generation(generation, run))

    def cmd_ga_run(self, run=None):
        index, record = self.mission.ga_run(run)
        return clean(dict(run=index, ga=ga_view(record['ga'])))

    def cmd_compare(self, seeds=5, scenario='estandar', policies=None):
        if scenario not in SCENARIOS:
            raise ValueError('Escenario desconocido')
        frame = run_experiments(policies, int(seeds), scenario, planet=self.config['planet'], mode=self.config['mode'])
        self.comparison = dict(scenario=scenario, label=SCENARIOS[scenario]['label'], seeds=int(seeds),
                               summary=summary_records(frame))
        self._comparison_frame = frame
        return clean(self.comparison)

    def cmd_export(self, format='json'):
        if format == 'csv':
            return {'filename': f"ares_x_traza_semilla{self.config['seed']}.csv", 'content': self.mission.export_csv()}
        if format == 'comparison_csv':
            frame = getattr(self, '_comparison_frame', None)
            if frame is None:
                raise ValueError('Primero ejecute una comparación de políticas')
            return {'filename': 'ares_x_comparacion.csv', 'content': frame.to_csv(index=False)}
        payload = json.dumps(self.mission.export_json(), ensure_ascii=False, indent=1)
        return {'filename': f"ares_x_mision_semilla{self.config['seed']}.json", 'content': payload}


def handle_json(session, fn, args_json):
    """String-in/string-out entry point (Colab callback and Pyodide)."""
    try:
        args = json.loads(args_json) if args_json else {}
    except json.JSONDecodeError:
        return json.dumps({'error': 'Argumentos JSON inválidos'})
    return json.dumps(session.dispatch(fn, args), ensure_ascii=False, allow_nan=False)


In [ ]:
%%writefile ares_x/src/ares/dashboard.py
"""Single-screen dashboard bridge for Google Colab (and plain Jupyter).

In Colab the HTML panel calls ``google.colab.kernel.invokeFunction('ares_api')``
and every action is executed by the Python engine in this kernel. Outside Colab
the same panel is served by a local HTTP server and embedded in an IFrame.
"""
import json
import threading
from pathlib import Path

from .api import Session, handle_json

ROOT = Path(__file__).resolve().parents[2]


class Dashboard:
    def __init__(self, session=None, height=880, **config):
        self.session = session or Session(**config)
        self.height = int(height)

    def html(self):
        page = (ROOT / 'web' / 'index.html').read_text(encoding='utf-8')
        # Inside a notebook output the viewport is the output frame: fix its height.
        return page.replace('--app-h:100vh;', f'--app-h:{self.height}px;', 1)

    def _callback(self, fn, args_json):
        from IPython.display import JSON
        return JSON(json.loads(handle_json(self.session, fn, args_json)))

    def show(self):
        from IPython.display import HTML, IFrame, display
        try:
            from google.colab import output
        except ImportError:
            port = self.serve()
            display(IFrame(f'http://127.0.0.1:{port}/', width='100%', height=self.height + 10))
            return
        output.register_callback('ares_api', self._callback)
        display(HTML(self.html()))

    def serve(self, port=0):
        """Serve this session locally (non-Colab notebooks)."""
        from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
        session, page, lock = self.session, (ROOT / 'web' / 'index.html').read_text(encoding='utf-8'), threading.Lock()

        class Handler(BaseHTTPRequestHandler):
            def _send(self, body, content_type):
                data = body.encode('utf-8')
                self.send_response(200)
                self.send_header('Content-Type', content_type)
                self.send_header('Content-Length', str(len(data)))
                self.end_headers()
                self.wfile.write(data)

            def do_GET(self):
                self._send(page, 'text/html; charset=utf-8')

            def do_POST(self):
                request = json.loads(self.rfile.read(int(self.headers.get('Content-Length') or 0)) or b'{}')
                with lock:
                    body = handle_json(session, request.get('fn'), json.dumps(request.get('args') or {}))
                self._send(body, 'application/json; charset=utf-8')

            def log_message(self, *args):
                pass

        server = ThreadingHTTPServer(('127.0.0.1', port), Handler)
        threading.Thread(target=server.serve_forever, daemon=True).start()
        return server.server_address[1]


Panel HTML (vista). La lógica no vive aquí: cada acción llama al motor Python mediante `google.colab.kernel.invokeFunction`.

In [ ]:
%%writefile ares_x/web/index.html
<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>ARES-X Control de Misión</title>
<style>
:root{
  --app-h:100vh;
  --bg:#070b12;--panel:#0d1420;--panel2:#111b2a;--line:#1d2a3d;--line2:#2a3b55;
  --text:#d7e2f0;--muted:#7f93ad;--dim:#56688a;
  --accent:#4cc3ff;--ok:#35d07f;--warn:#ffb547;--bad:#ff5d6c;--violet:#a98bff;--sci:#3ee6d6;
  --rock:#8b6b55;--font:"Inter","Segoe UI",system-ui,-apple-system,sans-serif;--mono:"JetBrains Mono","SFMono-Regular",Consolas,monospace;
}
*{box-sizing:border-box}
html,body{margin:0;height:100%;background:var(--bg);color:var(--text);font:12px/1.35 var(--font);overflow:hidden}
button,select,input{font:inherit;color:inherit}
#app{height:var(--app-h);display:grid;grid-template-rows:42px 1fr;min-height:560px}
header{display:flex;align-items:center;gap:10px;padding:0 12px;background:linear-gradient(90deg,#0b1626,#0a1220);border-bottom:1px solid var(--line)}
.brand{font-weight:800;letter-spacing:.08em;font-size:13px;white-space:nowrap}
.brand b{color:var(--accent)}
.chips{display:flex;gap:6px;flex-wrap:nowrap;overflow:hidden;flex:1;min-width:0}
.chip{padding:3px 8px;border:1px solid var(--line2);border-radius:999px;color:var(--muted);white-space:nowrap;font-size:11px}
.chip strong{color:var(--text);font-weight:600}
.state{font-weight:700;letter-spacing:.04em}
.st-EXPLORACION{color:var(--ok);border-color:#1f5a3e}.st-ALERTA,.st-REPLANIFICACION{color:var(--warn);border-color:#6b4d1d}
.st-REFUGIO{color:var(--accent);border-color:#1d4d6b}.st-MODO_SEGURO,.st-FRACASO{color:var(--bad);border-color:#6b1d27}.st-EXITO{color:#062;background:var(--ok);border-color:var(--ok)}
.st-CONFIGURACION{color:var(--muted)}
.body{display:grid;grid-template-columns:258px 1fr;min-height:0}
aside{border-right:1px solid var(--line);padding:8px 10px;display:flex;flex-direction:column;gap:7px;min-height:0;overflow:hidden;background:#0a101a}
h3{margin:0 0 4px;font-size:10.5px;letter-spacing:.12em;text-transform:uppercase;color:var(--muted);font-weight:700}
.grid2{display:grid;grid-template-columns:1fr 1fr;gap:5px}
.field{display:flex;flex-direction:column;gap:2px;min-width:0}
.field label{font-size:10px;color:var(--dim);text-transform:uppercase;letter-spacing:.06em}
select,input[type=number]{background:var(--panel2);border:1px solid var(--line2);border-radius:5px;padding:3px 5px;width:100%;min-width:0}
input[type=range]{width:100%;accent-color:var(--accent);margin:0}
.btn{background:var(--panel2);border:1px solid var(--line2);border-radius:6px;padding:4px 7px;cursor:pointer;white-space:nowrap}
.btn:hover{border-color:var(--accent)}.btn:active{transform:translateY(1px)}
.btn.primary{background:#0f3550;border-color:#1c6a9a;color:#cdeeff}
.btn.on{background:#173b23;border-color:var(--ok);color:#bff5d6}
.row{display:flex;gap:5px;align-items:center}
.row>.btn{flex:1}
.events{display:grid;grid-template-columns:1fr 1fr;gap:4px}
.events .btn{font-size:10.5px;padding:3px 4px;overflow:hidden;text-overflow:ellipsis}
.msg{min-height:30px;max-height:52px;overflow:hidden;font-size:11px;padding:5px 7px;border-radius:6px;background:#0b1420;border:1px solid var(--line);color:var(--muted)}
.msg.err{color:#ffd0d5;border-color:#6b1d27;background:#1a0b10}
main{display:grid;grid-template-rows:30px 1fr;min-width:0;min-height:0}
.tabs{display:flex;gap:2px;padding:0 8px;border-bottom:1px solid var(--line);align-items:flex-end}
.tab{padding:6px 12px;border:1px solid transparent;border-bottom:none;border-radius:7px 7px 0 0;color:var(--muted);cursor:pointer;background:none}
.tab.active{color:var(--text);background:var(--panel);border-color:var(--line)}
.view{display:none;min-height:0;padding:8px;gap:8px}
.view.active{display:grid}
.panel{background:var(--panel);border:1px solid var(--line);border-radius:9px;padding:8px 10px;min-height:0;min-width:0;overflow:hidden;display:flex;flex-direction:column}
.panel>h3{display:flex;justify-content:space-between;align-items:center;gap:6px}
.tag{font-size:9.5px;padding:1px 6px;border-radius:4px;border:1px solid var(--line2);color:var(--muted);letter-spacing:.04em;text-transform:none;font-weight:600}
#v-mission{grid-template-columns:minmax(0,1.45fr) minmax(0,1fr);grid-template-rows:minmax(0,1fr) minmax(0,.62fr)}
#mapPanel{grid-row:1}
.right{display:grid;grid-template-rows:auto minmax(0,1fr);gap:8px;min-height:0}
canvas{display:block;width:100%;height:100%}
.canvasBox{flex:1;min-height:0;position:relative}
.legend{display:flex;gap:9px;flex-wrap:wrap;color:var(--muted);font-size:10px;margin-top:4px}
.legend i{display:inline-block;width:9px;height:9px;border-radius:2px;margin-right:3px;vertical-align:-1px}
.tele{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:5px}
.kv{background:var(--panel2);border:1px solid var(--line);border-radius:6px;padding:4px 6px;min-width:0}
.kv .k{font-size:9.5px;color:var(--dim);text-transform:uppercase;letter-spacing:.05em;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.kv .v{font:600 13px var(--mono);white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.bar{height:4px;background:#1a2638;border-radius:2px;margin-top:3px;overflow:hidden}.bar>span{display:block;height:100%;background:var(--ok)}
.sensors{display:flex;gap:4px;margin-top:5px;flex-wrap:wrap}
.dot{font-size:10px;padding:1px 6px;border-radius:4px;border:1px solid #1f5a3e;color:var(--ok)}.dot.bad{border-color:#6b1d27;color:var(--bad)}.dot.warn{border-color:#6b4d1d;color:var(--warn)}
.why{display:grid;gap:5px;overflow:hidden}
.line{display:grid;grid-template-columns:62px 1fr;gap:7px;padding:5px 7px;border-radius:6px;background:var(--panel2);border-left:3px solid var(--line2)}
.line .who{font-weight:800;font-size:10.5px;letter-spacing:.08em}
.line .what{font-family:var(--mono);font-size:11px;overflow:hidden}
.line.rna{border-left-color:var(--accent)}.line.rna .who{color:var(--accent)}
.line.fz{border-left-color:var(--warn)}.line.fz .who{color:var(--warn)}
.line.ga{border-left-color:var(--violet)}.line.ga .who{color:var(--violet)}
.line.sup{border-left-color:var(--ok)}.line.sup .who{color:var(--ok)}
.line.sup.veto{border-left-color:var(--bad)}.line.sup.veto .who{color:var(--bad)}
.small{font-size:10.5px;color:var(--muted)}
.trace{flex:1;overflow:auto;font-family:var(--mono);font-size:10.5px;min-height:0}
.trace table{width:100%;border-collapse:collapse}
.trace td,.trace th{padding:2px 5px;border-bottom:1px solid #142033;text-align:left;white-space:nowrap}
.trace th{position:sticky;top:0;background:var(--panel);color:var(--dim);font-weight:600}
.trace td.x{white-space:normal;color:var(--muted);font-family:var(--font)}
.r-veto{color:var(--bad)}.r-warn{color:var(--warn)}.r-ok{color:var(--ok)}
.bottom{display:grid;grid-template-columns:minmax(0,1.45fr) minmax(0,1fr);gap:8px;grid-column:1/3;min-height:0}
.charts{display:grid;grid-template-rows:minmax(0,1fr) minmax(0,1fr);gap:4px;flex:1;min-height:0}
.charts canvas,.mini4 canvas{min-height:0;height:100%;overflow:hidden}
.mini4{grid-template-rows:minmax(0,1fr) minmax(0,1fr)!important}
.lab{grid-template-columns:repeat(3,minmax(0,1fr));grid-template-rows:minmax(0,1fr) minmax(0,1fr)}
.slider{display:grid;grid-template-columns:92px 1fr 52px;gap:6px;align-items:center;margin:3px 0}
.slider span{font-family:var(--mono);text-align:right}
.probs .p{display:grid;grid-template-columns:72px 1fr 52px;gap:6px;align-items:center;margin:4px 0}
.probs .track{height:12px;background:#1a2638;border-radius:3px;overflow:hidden}.probs .track span{display:block;height:100%}
table.mt{border-collapse:collapse;width:100%;font-family:var(--mono);font-size:11px}
table.mt td,table.mt th{border:1px solid var(--line);padding:3px 5px;text-align:center}
table.mt th{color:var(--dim);font-weight:600}
.scroll{overflow:auto;min-height:0;flex:1}
.rule{padding:3px 6px;border-radius:5px;margin:2px 0;background:var(--panel2);display:grid;grid-template-columns:26px 1fr 44px;gap:6px;font-size:11px;border:1px solid transparent}
.rule.act{border-color:#6b4d1d;background:#21180a}
.rule b{font-family:var(--mono);color:var(--warn)}
.big{font:700 22px var(--mono)}
.mini4{display:grid;grid-template-columns:1fr 1fr;grid-template-rows:1fr 1fr;gap:6px;flex:1;min-height:0}
.pill{display:inline-block;padding:1px 6px;border-radius:4px;font-size:10px;border:1px solid var(--line2)}
.pill.ok{color:var(--ok);border-color:#1f5a3e}.pill.bad{color:var(--bad);border-color:#6b1d27}
.overlay{position:absolute;inset:0;display:flex;align-items:center;justify-content:center;background:rgba(7,11,18,.7);font-size:13px;color:var(--muted);pointer-events:none}
@media (max-width:1100px){.body{grid-template-columns:220px 1fr}.tele{grid-template-columns:repeat(3,minmax(0,1fr))}}
</style>
</head>
<body>
<div id="app">
<header>
  <div class="brand"><b>ARES-X</b> | CENTRO DE CONTROL DE MISIÓN</div>
  <div class="chips">
    <span class="chip" id="cPlanet">—</span>
    <span class="chip state" id="cState">—</span>
    <span class="chip" id="cStep">—</span>
    <span class="chip" id="cPolicy">—</span>
    <span class="chip" id="cOutcome" style="display:none"></span>
  </div>
  <span class="chip" title="Todos los valores operacionales son hipótesis pedagógicas">supuesto de simulación</span>
</header>
<div class="body">
<aside>
  <div>
    <h3>Configuración</h3>
    <div class="grid2">
      <div class="field"><label>Planeta</label><select id="fPlanet"></select></div>
      <div class="field"><label>Modo</label><select id="fMode"><option value="rover">Rover (ruedas)</option><option value="sonda">Sonda atmosférica</option></select></div>
      <div class="field"><label>Semilla</label><input id="fSeed" type="number" min="0" step="1" value="42"></div>
      <div class="field"><label>Dificultad</label><select id="fDiff"><option value="low">Baja</option><option value="medium" selected>Media</option><option value="high">Alta</option></select></div>
    </div>
    <div class="field" style="margin-top:5px"><label>Batería inicial: <b id="lBat">100</b> %</label><input id="fBattery" type="range" min="0" max="100" step="1" value="100"></div>
    <div class="field" style="margin-top:3px"><label>Política</label><select id="fPolicy"></select></div>
    <div class="row" style="margin-top:6px"><button class="btn primary" id="bApply">Aplicar configuración</button></div>
  </div>
  <div>
    <h3>Simulación</h3>
    <div class="row"><button class="btn on" id="bRun">▶ Iniciar</button><button class="btn" id="bStep">Paso</button><button class="btn" id="bReset">Reiniciar</button></div>
    <div class="field" style="margin-top:5px"><label>Velocidad: <b id="lSpeed">1</b>× pasos por ciclo</label><input id="fSpeed" type="range" min="1" max="8" step="1" value="1"></div>
  </div>
  <div>
    <h3>Inyectar evento</h3>
    <div class="events" id="events"></div>
    <div class="field" style="margin-top:5px"><label>Editar terreno real (clic en el mapa)</label>
      <select id="fTool"><option value="">Sin edición</option><option value="obstaculo">Colocar obstáculo</option><option value="barranco">Colocar barranco</option><option value="limpiar">Limpiar celda</option></select></div>
  </div>
  <div>
    <h3>Exportar</h3>
    <div class="row"><button class="btn" data-export="csv">Traza CSV</button><button class="btn" data-export="json">Misión JSON</button></div>
  </div>
  <div class="msg" id="msg">Listo. Configure la misión y presione Iniciar.</div>
</aside>
<main>
  <div class="tabs">
    <button class="tab active" data-view="mission">Misión</button>
    <button class="tab" data-view="rna">Laboratorio RNA</button>
    <button class="tab" data-view="fuzzy">Laboratorio Difuso</button>
    <button class="tab" data-view="ga">Laboratorio Genético</button>
    <button class="tab" data-view="compare">Comparación de políticas</button>
  </div>

  <section class="view active" id="v-mission">
    <div class="panel" id="mapPanel">
      <h3>Mapa y trayectoria en tiempo real <label class="tag"><input type="checkbox" id="fReveal" style="vertical-align:-2px"> mostrar terreno oculto</label></h3>
      <div class="canvasBox"><canvas id="map"></canvas></div>
      <div class="legend">
        <span><i style="background:#ffd54a"></i>rover</span><span><i style="background:var(--accent)"></i>ruta ejecutada</span>
        <span><i style="background:var(--ok)"></i>ruta planificada</span><span><i style="background:var(--violet)"></i>candidatas AG</span>
        <span><i style="background:var(--rock)"></i>obstáculo</span><span><i style="background:var(--bad)"></i>barranco</span>
        <span><i style="background:var(--warn)"></i>riesgo RNA</span><span><i style="background:var(--sci)"></i>sitio científico</span><span><i style="background:#2f8f5b"></i>refugio</span>
      </div>
    </div>
    <div class="right">
      <div class="panel">
        <h3>Telemetría y eventos <span class="tag" id="tDay">día</span></h3>
        <div class="tele" id="tele"></div>
        <div class="sensors" id="sensors"></div>
      </div>
      <div class="panel">
        <h3>Decisión explicable <span class="tag" id="tReason">—</span></h3>
        <div class="why">
          <div class="line rna"><div class="who">RNA</div><div class="what" id="wRna">—</div></div>
          <div class="line fz"><div class="who">DIFUSA</div><div class="what" id="wFz">—</div></div>
          <div class="line ga"><div class="who">AG</div><div class="what" id="wGa">—</div></div>
          <div class="line sup" id="lSup"><div class="who">ÁRBITRO</div><div class="what" id="wSup">—</div></div>
        </div>
      </div>
    </div>
    <div class="bottom">
      <div class="panel">
        <h3>Traza temporal / historial de decisiones
          <select id="fFilter" style="width:auto;padding:1px 4px;font-size:10.5px"><option value="">Todo</option><option value="veto">Vetos y alertas</option><option value="ga">Replanificaciones</option><option value="ev">Eventos</option></select></h3>
        <div class="trace" id="trace"></div>
      </div>
      <div class="panel">
        <h3>Evolución <span class="tag">batería · velocidad difusa · riesgo RNA · fitness AG</span></h3>
        <div class="charts"><canvas id="cSeries"></canvas><canvas id="cFitness"></canvas></div>
      </div>
    </div>
  </section>

  <section class="view lab" id="v-rna">
    <div class="panel"><h3>Entrada de sensores <button class="btn" id="bRnaCur" style="font-size:10.5px;padding:2px 6px">Usar lectura actual</button></h3><div id="rnaSliders"></div>
      <div class="small" style="margin-top:6px">La etiqueta de verdad la genera el simulador (pertenencia a barranco/obstáculo), no las variables que recibe la red: no hay fuga de etiqueta.</div></div>
    <div class="panel"><h3>Predicción y confianza</h3><div class="probs" id="rnaProbs"></div><div id="rnaPred" style="margin-top:6px"></div></div>
    <div class="panel"><h3>Capas ocultas (ReLU 32 → 16)</h3><div id="rnaLayers"></div></div>
    <div class="panel"><h3>Métricas en test (mapas no vistos)</h3><div id="rnaMetrics"></div></div>
    <div class="panel"><h3>Matriz de confusión (test)</h3><div id="rnaConf"></div></div>
    <div class="panel"><h3>Partición por mapa</h3><div id="rnaSplit" class="small"></div></div>
  </section>

  <section class="view lab" id="v-fuzzy">
    <div class="panel"><h3>Entradas crisp <button class="btn" id="bFzCur" style="font-size:10.5px;padding:2px 6px">Usar lectura actual</button></h3><div id="fzSliders"></div>
      <div style="margin-top:8px"><span class="small">Velocidad (centroide)</span><div class="big" id="fzOut">—</div><div class="small" id="fzReason"></div></div></div>
    <div class="panel" style="grid-column:2/4"><h3>Funciones de membresía (triangular / trapezoidal) y punto de entrada</h3><div class="mini4"><canvas id="fz0"></canvas><canvas id="fz1"></canvas><canvas id="fz2"></canvas><canvas id="fz3"></canvas></div></div>
    <div class="panel" style="grid-column:1/3"><h3>Reglas Mamdani (min–max) <span class="tag" id="fzCount">—</span></h3><div class="scroll" id="fzRules"></div></div>
    <div class="panel"><h3>Agregación y desfusificación</h3><div class="canvasBox"><canvas id="fzAgg"></canvas></div></div>
  </section>

  <section class="view lab" id="v-ga">
    <div class="panel"><h3>Ejecución <select id="fRun" style="width:auto;flex:1;padding:1px 4px;font-size:10.5px"></select> gen. <span class="tag" id="gaGenLbl">—</span></h3><input type="range" id="fGen" min="0" max="0" value="0"><div id="gaOps" class="small" style="margin:5px 0"></div><div class="scroll" id="gaChrom"></div></div>
    <div class="panel"><h3>Mejor fitness por generación</h3><div class="canvasBox"><canvas id="gaChart"></canvas></div><div class="small" id="gaInfo"></div></div>
    <div class="panel"><h3>Pesos del fitness <span class="tag">normalizados</span></h3><div id="gaWeights"></div><div class="row" style="margin-top:6px"><button class="btn primary" id="bWeights">Aplicar pesos y reiniciar</button></div>
      <div class="small" style="margin-top:6px">fitness = w1·ciencia − w2·distancia − w3·energía − w4·riesgo − w5·tiempo − P·inviabilidad</div></div>
    <div class="panel" style="grid-column:1/4"><h3>Rutas candidatas: desglose de aptitud y vetos del supervisor</h3><div class="scroll" id="gaCand"></div></div>
  </section>

  <section class="view" id="v-compare" style="grid-template-rows:auto minmax(0,1fr) minmax(0,.8fr)">
    <div class="panel"><div class="row" style="flex-wrap:wrap">
      <div class="field" style="width:260px"><label>Escenario</label><select id="fScenario"></select></div>
      <div class="field" style="width:120px"><label>Semillas</label><select id="fSeeds"><option>3</option><option selected>5</option><option>10</option><option>20</option></select></div>
      <button class="btn primary" id="bCompare" style="flex:0;margin-top:13px">Ejecutar comparación</button>
      <button class="btn" data-export="comparison_csv" style="flex:0;margin-top:13px">Exportar CSV</button>
      <span class="small" style="margin-top:13px">Misma semilla y condiciones idénticas por política. 20 semillas toman ~1–2 min.</span></div></div>
    <div class="panel"><h3>Métricas por política <span class="tag">media ± desv. (mediana)</span></h3><div class="scroll" id="cmpTable"><div class="small">Sin resultados todavía.</div></div></div>
    <div class="panel"><h3>Éxito de misión y colisiones</h3><div class="canvasBox"><canvas id="cmpChart"></canvas></div></div>
  </section>
</main>
</div>
</div>
<script>
'use strict';
// ---------------------------------------------------------------- transport
async function api(fn, args){
  args = args || {};
  if (window.ARES_TRANSPORT) return JSON.parse(await window.ARES_TRANSPORT(fn, JSON.stringify(args)));
  const colab = window.google && google.colab && google.colab.kernel;
  if (colab && colab.invokeFunction){
    const r = await colab.invokeFunction('ares_api', [fn, JSON.stringify(args)], {});
    return r.data['application/json'];
  }
  const res = await fetch('api', {method:'POST', headers:{'Content-Type':'application/json'}, body: JSON.stringify({fn, args})});
  return res.json();
}
const $ = id => document.getElementById(id);
let S = null, META = null, running = false, busy = false, view = 'mission';
const css = n => getComputedStyle(document.documentElement).getPropertyValue(n).trim();
function msg(text, err){ const m=$('msg'); m.textContent=text; m.className='msg'+(err?' err':''); }
const fmt = (v, d=2) => v===null||v===undefined||Number.isNaN(v) ? '—' : (typeof v==='number' ? v.toFixed(d) : v);

async function call(fn, args, quiet){
  if (busy && fn==='step') return null;
  busy = true;
  try{
    const out = await api(fn, args);
    if (out && out.error){ msg(out.error, true); return null; }
    if (out && out.status){ S = out; render(); }
    if (!quiet && fn!=='step' && out && !out.error && out.status) msg('Hecho: ' + fn);
    return out;
  }catch(e){ msg('Error de comunicación con el motor Python: ' + e, true); running=false; syncRun(); return null; }
  finally{ busy = false; }
}

// ---------------------------------------------------------------- canvas utils
function ctx2d(c){ const r=c.getBoundingClientRect(); const dpr=window.devicePixelRatio||1;
  const w=Math.max(10,Math.floor(r.width)), h=Math.max(10,Math.floor(r.height));
  if (c.width!==w*dpr||c.height!==h*dpr){ c.width=w*dpr; c.height=h*dpr; }
  const g=c.getContext('2d'); g.setTransform(dpr,0,0,dpr,0,0); g.clearRect(0,0,w,h); return [g,w,h]; }
function lineChart(c, series, opts){
  const [g,w,h]=ctx2d(c); opts=opts||{}; const pad={l:30,r:8,t:14,b:14};
  const all=series.flatMap(s=>s.data.filter(v=>v!==null&&v!==undefined));
  let lo=opts.min!==undefined?opts.min:Math.min(...all,0), hi=opts.max!==undefined?opts.max:Math.max(...all,1);
  if (hi===lo) hi=lo+1;
  g.strokeStyle=css('--line'); g.lineWidth=1; g.fillStyle=css('--dim'); g.font='10px sans-serif';
  for (let i=0;i<=2;i++){ const y=pad.t+(h-pad.t-pad.b)*i/2; g.beginPath(); g.moveTo(pad.l,y); g.lineTo(w-pad.r,y); g.stroke();
    g.fillText((hi-(hi-lo)*i/2).toFixed(opts.digits??0),2,y+3); }
  const n=Math.max(...series.map(s=>s.data.length),2);
  series.forEach(s=>{ g.strokeStyle=s.color; g.lineWidth=1.6; g.beginPath(); let started=false;
    s.data.forEach((v,i)=>{ if(v===null||v===undefined) return; const x=pad.l+(w-pad.l-pad.r)*i/(n-1), y=pad.t+(h-pad.t-pad.b)*(1-(v-lo)/(hi-lo));
      started?g.lineTo(x,y):g.moveTo(x,y); started=true; }); g.stroke(); });
  let x=pad.l; series.forEach(s=>{ g.fillStyle=s.color; g.fillRect(x,2,8,8); g.fillStyle=css('--muted'); g.fillText(s.name,x+11,10); x+=g.measureText(s.name).width+24; });
  if (opts.title){ g.fillStyle=css('--dim'); g.textAlign='right'; g.fillText(opts.title,w-pad.r,10); g.textAlign='left'; }
}

// ---------------------------------------------------------------- map
let mapGeom=null;
function drawMap(){
  if (!S) return; const c=$('map'); const [g,w,h]=ctx2d(c); const M=S.map;
  const cs=Math.floor(Math.min(w/M.width, h/M.height)); const ox=Math.floor((w-cs*M.width)/2), oy=Math.floor((h-cs*M.height)/2);
  mapGeom={cs,ox,oy};
  const cx=(c)=>ox+c*cs+cs/2, cy=(r)=>oy+r*cs+cs/2;
  let emin=Infinity, emax=-Infinity; M.elevation.forEach(row=>row.forEach(v=>{ if(v>-5){emin=Math.min(emin,v);emax=Math.max(emax,v);} }));
  for (let r=0;r<M.height;r++) for (let c=0;c<M.width;c++){
    const e=M.elevation[r][c]; const t=Math.max(0,Math.min(1,(e-emin)/((emax-emin)||1)));
    g.fillStyle=`rgb(${20+t*26},${32+t*32},${46+t*36})`; g.fillRect(ox+c*cs,oy+r*cs,cs-1,cs-1);
  }
  const reveal=$('fReveal').checked;
  M.risk.forEach(([r,c,v])=>{ g.fillStyle=`rgba(255,181,71,${Math.min(.65,v*.7)})`; g.fillRect(ox+c*cs,oy+r*cs,cs-1,cs-1); });
  M.refuges.forEach(([r,c])=>{ g.fillStyle='#2f8f5b'; g.fillRect(ox+c*cs+2,oy+r*cs+2,cs-5,cs-5); g.fillStyle='#d9ffe9'; g.font=`bold ${cs*.42}px sans-serif`; g.textAlign='center'; g.fillText('R',cx(c),cy(r)+cs*.15); });
  M.resources.forEach(([r,c])=>{ g.strokeStyle='#5f7da3'; g.strokeRect(ox+c*cs+cs*.3,oy+r*cs+cs*.3,cs*.4,cs*.4); });
  const rock=(r,c,dashed)=>{ g.beginPath(); g.arc(cx(c),cy(r),cs*.36,0,7); if(dashed){ g.setLineDash([3,3]); g.strokeStyle=css('--rock'); g.lineWidth=1.5; g.stroke(); g.setLineDash([]);} else { g.fillStyle=css('--rock'); g.fill(); } };
  const cliff=(r,c,dashed)=>{ const x=ox+c*cs, y=oy+r*cs; g.save(); g.beginPath(); g.rect(x+1,y+1,cs-3,cs-3); g.clip();
    g.strokeStyle=dashed?'rgba(255,93,108,.55)':css('--bad'); g.lineWidth=dashed?1:2; if(dashed) g.setLineDash([3,3]);
    for(let k=-cs;k<cs*2;k+=6){ g.beginPath(); g.moveTo(x+k,y); g.lineTo(x+k+cs,y+cs); g.stroke(); } g.restore(); g.setLineDash([]); };
  M.obstacles.forEach(([r,c])=>rock(r,c,false)); M.cliffs.forEach(([r,c])=>cliff(r,c,false));
  if (reveal){ M.hidden_obstacles.forEach(([r,c])=>rock(r,c,true)); M.hidden_cliffs.forEach(([r,c])=>cliff(r,c,true)); }
  M.science.forEach(([r,c,done])=>{ g.beginPath(); g.moveTo(cx(c),cy(r)-cs*.32); g.lineTo(cx(c)+cs*.3,cy(r)); g.lineTo(cx(c),cy(r)+cs*.32); g.lineTo(cx(c)-cs*.3,cy(r)); g.closePath();
    if(done){ g.fillStyle=css('--ok'); g.fill(); } else { g.strokeStyle=css('--sci'); g.lineWidth=2; g.stroke(); } });
  const poly=(pts,color,width,dash)=>{ if(!pts||pts.length<2) return; g.strokeStyle=color; g.lineWidth=width; g.setLineDash(dash||[]); g.beginPath();
    pts.forEach(([r,c],i)=>i?g.lineTo(cx(c),cy(r)):g.moveTo(cx(c),cy(r))); g.stroke(); g.setLineDash([]); };
  (M.candidates||[]).forEach((route,i)=>{ if(i!==M.chosen) poly(route,'rgba(169,139,255,.45)',1.5,[2,4]); });
  poly(M.route,css('--ok'),2,[6,4]);
  poly(M.path,css('--accent'),2.6);
  (M.events||[]).forEach(e=>{ if(!e.position) return; const [r,c]=e.position; g.strokeStyle='#ffd54a'; g.lineWidth=1; g.strokeRect(ox+c*cs+1,oy+r*cs+1,cs-3,cs-3); });
  const [pr,pc]=S.status.position; g.save(); g.translate(cx(pc),cy(pr)); g.fillStyle='#ffd54a'; g.beginPath();
  g.moveTo(0,-cs*.38); g.lineTo(cs*.32,cs*.3); g.lineTo(-cs*.32,cs*.3); g.closePath(); g.fill(); g.strokeStyle='#000'; g.lineWidth=1; g.stroke(); g.restore();
  g.fillStyle=css('--dim'); g.font='9px sans-serif'; g.textAlign='center';
  for (let c=0;c<M.width;c+=3) g.fillText(c,cx(c),oy-2>8?oy-2:9);
  g.textAlign='left';
}
$('map').addEventListener('click', async ev=>{
  const tool=$('fTool').value; if(!tool||!mapGeom||!S) return;
  const r=$('map').getBoundingClientRect(); const c=Math.floor((ev.clientX-r.left-mapGeom.ox)/mapGeom.cs), row=Math.floor((ev.clientY-r.top-mapGeom.oy)/mapGeom.cs);
  if(row<0||c<0||row>=S.map.height||c>=S.map.width) return;
  const out=await call('place',{position:[row,c],kind:tool},true); if(out) msg(`Terreno editado en (${row},${c}): ${tool}. El rover debe descubrirlo con sus sensores.`);
});

// ---------------------------------------------------------------- mission view
function kv(k,v,extra){ return `<div class="kv"><div class="k">${k}</div><div class="v">${v}</div>${extra||''}</div>`; }
function render(){
  if (!S) return; const st=S.status;
  $('cPlanet').innerHTML=`<strong>${S.planet.name.toUpperCase()}</strong> · ${S.planet.mode==='rover'?'rover':'sonda'} · semilla ${S.config.seed}`;
  const se=$('cState'); se.textContent=st.state; se.className='chip state st-'+st.state;
  $('cStep').innerHTML=`SOL <strong>${String(Math.floor(st.step/60)+1).padStart(3,'0')}</strong> · paso <strong>${st.step}</strong>`;
  $('cPolicy').innerHTML=`política: <strong>${S.policy_label}</strong>`;
  const oc=$('cOutcome'); if(st.finished){ oc.style.display=''; oc.textContent=st.outcome; oc.className='chip state st-'+st.state; } else oc.style.display='none';
  if (st.finished && running){ running=false; syncRun(); msg('Misión finalizada: '+st.outcome, st.state!=='EXITO'); }
  $('tDay').textContent=(st.daylight?'día':'noche')+(st.storm?' · tormenta':'')+(st.link?'':' · sin enlace');
  const T=S.telemetry; const bcol=st.battery<15?css('--bad'):st.battery<35?css('--warn'):css('--ok');
  $('tele').innerHTML=[
    kv('Batería %',fmt(st.battery,1),`<div class="bar"><span style="width:${st.battery}%;background:${bcol}"></span></div>`),
    kv('Integridad %',fmt(st.health,0),`<div class="bar"><span style="width:${st.health}%;background:${st.health<40?css('--bad'):css('--ok')}"></span></div>`),
    kv('Riesgo RNA',S.rna?fmt(1-S.rna.probabilities.libre,2):'—'), kv('Ciencia',Math.round(st.science*100)+' %'),
    kv('T ext °C',fmt(T.temperature_external,1)), kv('T int °C',fmt(st.temperature_internal,1)), kv('Pendiente °',fmt(T.slope,1)), kv('Δ elev m',fmt(T.elevation_delta,1)),
    kv('Proximidad m',fmt(T.distance,2)), kv('Visibilidad',fmt(st.visibility,2)), kv('Solar W',fmt(T.solar_power,1)),
    kv('O₂ amb. %',fmt(T.oxygen_ambient,2)+' <span class="small">ref.</span>')].join('');
  const nv=T.nav_valid||{}; const fails=st.sensor_failures||[];
  $('sensors').innerHTML=[['proximidad','Proximidad'],['lidar','LIDAR']].map(([k,l])=>`<span class="dot ${nv[k]===false?'bad':''}">${l} ${nv[k]===false?'NaN':'OK'}</span>`).join('')+
    `<span class="dot ${T.edge_cliff?'bad':''}">Borde ${T.edge_cliff?'¡BARRANCO!':'OK'}</span>`+
    `<span class="dot ${st.link?'':'warn'}">Enlace ${st.link?'OK':'autónomo'}</span>`+(st.quake?'<span class="dot warn">Sismo</span>':'')+
    (S.rna_error?`<span class="dot bad">RNA sin lectura</span>`:'')+(fails.length?`<span class="dot bad">Falla: ${fails.join(', ')}</span>`:'');
  // explanation
  const R=S.rna; $('wRna').innerHTML = R ? `celda (${R.cell}) → <b>${R.label}</b> p=${fmt(R.confidence,2)} · libre ${fmt(R.probabilities.libre,2)} · obst ${fmt(R.probabilities.obstaculo,2)} · barr ${fmt(R.probabilities.barranco,2)}${R.low_confidence?' · <span class="r-warn">baja confianza</span>':''}`
    : (S.config.policy==='sin_rna'||S.config.policy==='control'||S.config.policy==='ag_puro' ? '<span class="small">desactivada en esta política</span>' : (S.rna_error?`<span class="r-veto">${S.rna_error}: fusión con sensor determinista</span>`:'—'));
  const F=S.fuzzy; $('wFz').innerHTML = F ? `dist ${fmt(F.inputs.distance,1)} m · bat ${fmt(F.inputs.battery,0)} % · pend ${fmt(F.inputs.slope,1)}° → velocidad <b>${fmt(F.speed,2)}</b> · ${F.rules.length} reglas (${F.rules.slice(0,3).map(r=>'R'+r.id+'='+fmt(r.strength,2)).join(', ')})`
    : (st.step===0 ? '—' : '<span class="small">desactivada: velocidad fija 0,50</span>');
  const G=S.ga; const ch=G.candidates&&G.chosen!==null&&G.chosen!==undefined?G.candidates[G.chosen]:null;
  $('wGa').innerHTML = G.mode==='ag' ? `${G.candidates.length} rutas · elegida #${G.chosen??'—'} fitness <b>${ch?fmt(ch.fitness,3):'—'}</b> · ${G.generations_run} gen${G.stopped_early?' (parada temprana)':''} · motivo ${G.reason}${G.rejected.length?` · <span class="r-veto">${G.rejected.length} vetada(s)</span>`:''}`
    : (G.mode ? `<span class="small">${G.message||G.mode}</span>` : '—');
  const D=S.decision; const sup=$('lSup');
  if (D){ $('wSup').innerHTML=`<b>${D.action.toUpperCase()}</b> · ${D.explanation}${D.vetoes.length?' · veto: '+D.vetoes.join('; '):''}`; $('tReason').textContent=D.reason;
    sup.className='line sup'+(['replanificar','detener'].includes(D.action)&&D.reason!=='R15_MISION_COMPLETA'?' veto':''); }
  else { $('wSup').textContent='Esperando el primer paso.'; $('tReason').textContent='—'; }
  renderTrace(); drawMap();
  const ser=S.series;
  lineChart($('cSeries'),[{name:'batería %',data:ser.battery,color:css('--ok')},{name:'velocidad×100',data:ser.speed.map(v=>v*100),color:css('--warn')},{name:'riesgo×100',data:ser.risk.map(v=>v===null?null:v*100),color:css('--bad')}],{min:0,max:100});
  lineChart($('cFitness'),[{name:'mejor fitness AG',data:(G.history||[]).map(h=>h.best),color:css('--violet')},{name:'mejor factible',data:(G.history||[]).map(h=>h.best_feasible),color:css('--accent')}],{digits:2,title:'por generación (última planificación)'});
  if (view!=='mission') renderView();
}
function renderTrace(){
  const f=$('fFilter').value; let rows=S.log.slice().reverse();
  if (f==='veto') rows=rows.filter(r=>/R0[0-8]|R1[0-3]/.test(r.reason));
  if (f==='ga') rows=rows.filter(r=>r.ga!==null&&r.ga!==undefined);
  if (f==='ev') rows=rows.filter(r=>r.events.length);
  const cls=r=>/R0[0-6]|R10/.test(r)?'r-veto':/R0[7-9]|R1[1-3]/.test(r)?'r-warn':'r-ok';
  $('trace').innerHTML='<table><tr><th>T+</th><th>estado</th><th>acción</th><th>motivo</th><th>bat</th><th>RNA</th><th>vel</th><th>detalle</th></tr>'+
    rows.map(r=>`<tr><td>${String(r.step).padStart(4,'0')}</td><td>${r.state}</td><td>${r.action}</td><td class="${cls(r.reason)}">${r.reason}</td><td>${fmt(r.battery,1)}</td><td>${r.rna?r.rna+' '+fmt(r.conf,2):'—'}</td><td>${fmt(r.speed,2)}</td><td class="x">${r.events.length?'<b>⚡ '+r.events.join(', ')+'</b> · ':''}${r.ga!==null&&r.ga!==undefined?'AG fitness '+fmt(r.ga,3)+' · ':''}${r.explanation}</td></tr>`).join('')+'</table>';
}
$('fFilter').addEventListener('change',renderTrace);
$('fReveal').addEventListener('change',drawMap);

// ---------------------------------------------------------------- RNA lab
const RNA_RANGES={distance:[.01,20,.01,'m'],slope:[0,45,.5,'°'],elevation_delta:[-15,10,.1,'m'],texture:[0,1,.01,''],visibility:[.35,1,.01,''],vibration:[0,5,.05,'']};
const RNA_LABELS={distance:'Distancia',slope:'Pendiente',elevation_delta:'Δ elevación',texture:'Textura',visibility:'Visibilidad',vibration:'Vibración'};
function buildRnaSliders(){ $('rnaSliders').innerHTML=Object.entries(RNA_RANGES).map(([k,[lo,hi,st]])=>`<div class="slider"><label>${RNA_LABELS[k]}</label><input type="range" id="rn_${k}" min="${lo}" max="${hi}" step="${st}" value="${k==='distance'?10:k==='visibility'?1:k==='texture'?.25:0}"><span id="rv_${k}"></span></div>`).join('');
  Object.keys(RNA_RANGES).forEach(k=>$('rn_'+k).addEventListener('input',rnaEval)); }
let rnaTimer=null;
function rnaEval(){ Object.keys(RNA_RANGES).forEach(k=>$('rv_'+k).textContent=(+$('rn_'+k).value).toFixed(2)); clearTimeout(rnaTimer); rnaTimer=setTimeout(async()=>{
  const args={}; Object.keys(RNA_RANGES).forEach(k=>args[k]=+$('rn_'+k).value);
  const out=await api('rna_eval',args); if(out.error){ msg(out.error,true); return; }
  const colors={libre:css('--ok'),obstaculo:css('--rock'),barranco:css('--bad')};
  $('rnaProbs').innerHTML=Object.entries(out.probabilities).map(([k,v])=>`<div class="p"><span>${k}</span><div class="track"><span style="width:${v*100}%;background:${colors[k]}"></span></div><span>${fmt(v,3)}</span></div>`).join('');
  $('rnaPred').innerHTML=`<div class="big">${out.label}</div><div class="small">confianza ${fmt(out.confidence,3)} ${out.low_confidence?'<span class="r-warn">· baja confianza → fusión con sensor determinista y modo conservador</span>':''}<br>riesgo para AG = 0,7·p(obstáculo)+p(barranco) = ${fmt(out.risk,3)}</div>
    <div class="small" style="margin-top:6px">entrada normalizada (z): [${out.normalized.map(v=>fmt(v,2)).join(', ')}]</div>`;
  $('rnaLayers').innerHTML=out.layers.map((l,i)=>`<div class="kv" style="margin-bottom:6px"><div class="k">Capa oculta ${i+1} · ${l.neurons} neuronas</div><div class="v" style="font-size:11.5px">media ${fmt(l.mean,3)} · máx ${fmt(l.max,3)} · activas ${Math.round(l.active_fraction*100)} %</div><div class="bar"><span style="width:${l.active_fraction*100}%;background:var(--accent)"></span></div></div>`).join('')+
    '<div class="small">Salida softmax de 3 clases → acción posterior: veto del supervisor si p ≥ 0,65 y la clase no es «libre».</div>';
},120); }
$('bRnaCur').addEventListener('click',()=>{ if(!S||!S.rna){ msg('No hay lectura RNA actual (ejecute pasos con una política que use RNA).',true); return; } Object.entries(S.rna.features).forEach(([k,v])=>{ $('rn_'+k).value=v; }); rnaEval(); });
function renderRnaStatic(){ if(!S) return; const t=S.nn.test;
  $('rnaMetrics').innerHTML=[kv('Exactitud',fmt(t.accuracy,3)),kv('Baseline mayoría',fmt(t.majority_baseline,3)),kv('Macro-F1',fmt(t.macro_f1,3)),kv('Recall barranco',fmt(t.barranco_recall,3)),kv('Error de calibración',fmt(t.calibration_error,3)),kv('Muestras test',t.samples),kv('Iteraciones',S.nn.iterations),kv('Entrenamiento s',fmt(S.nn.train_seconds,2))].map(x=>x).join('').replace(/^/,'<div class="tele" style="grid-template-columns:1fr 1fr">')+'</div>';
  const L=t.labels; $('rnaConf').innerHTML=`<table class="mt"><tr><th>real \\ pred</th>${L.map(l=>`<th>${l}</th>`).join('')}</tr>${t.confusion_matrix.map((row,i)=>`<tr><th>${L[i]}</th>${row.map((v,j)=>`<td style="background:${i===j?'rgba(53,208,127,.15)':v?'rgba(255,93,108,.12)':''}">${v}</td>`).join('')}</tr>`).join('')}</table><div class="small" style="margin-top:6px">Validación: exactitud ${fmt(S.nn.validation.accuracy,3)} · macro-F1 ${fmt(S.nn.validation.macro_f1,3)}</div>`;
  const sp=S.nn.split_maps; $('rnaSplit').innerHTML=`Los datos se separan <b>por mapa</b> (no por filas), evitando filas correlacionadas entre particiones.<br><br>Entrenamiento: mapas ${sp.train.join(', ')}<br>Validación: mapas ${sp.validation.join(', ')}<br>Test: mapas ${sp.test.join(', ')}<br><br>Muestras de entrenamiento: ${S.nn.training_samples} · MLP (32,16) ReLU · softmax · CPU scikit-learn.`;
}

// ---------------------------------------------------------------- fuzzy lab
const FZ=[['distance','Distancia a riesgo','m',0,20,.1],['battery','Batería','%',0,100,1],['slope','Pendiente','°',0,45,.5],['temperature','Temperatura interna','°C',-80,120,1]];
function buildFzSliders(){ $('fzSliders').innerHTML=FZ.map(([k,l,u,lo,hi,st])=>`<div class="slider"><label>${l}</label><input type="range" id="fz_${k}" min="${lo}" max="${hi}" step="${st}" value="${k==='distance'?10:k==='battery'?60:k==='slope'?5:12}"><span id="fv_${k}"></span></div>`).join('');
  FZ.forEach(([k])=>$('fz_'+k).addEventListener('input',fzEval)); }
let fzTimer=null, lastFz=null;
function fzEval(){ FZ.forEach(([k,,u])=>$('fv_'+k).textContent=(+$('fz_'+k).value).toFixed(1)+u); clearTimeout(fzTimer); fzTimer=setTimeout(async()=>{
  const out=await api('fuzzy_eval',{distance:+$('fz_distance').value,battery:+$('fz_battery').value,slope:+$('fz_slope').value,temperature:+$('fz_temperature').value});
  if(out.error){ msg(out.error,true); return; } lastFz=out; drawFz(); },90); }
const SETCOL=['#4cc3ff','#ffb547','#ff5d6c','#35d07f'];
const VARES={distance:'distancia',battery:'batería',slope:'pendiente',temperature:'temperatura'};
function drawFz(){ if(!META||!lastFz) return; const out=lastFz;
  $('fzOut').textContent=fmt(out.speed,3); $('fzReason').innerHTML=out.reason==='SIN_REGLAS_ACTIVAS'?`<span class="r-warn">${out.fallback}</span>`:`cautela ${fmt(out.caution,3)} · ${out.reason}`;
  FZ.forEach(([k,l],i)=>{ const set=META.fuzzy_sets[k]; const c=$('fz'+i); const [g,w,h]=ctx2d(c); const pad=16; const xs=set.x; const lo=xs[0], hi=xs[xs.length-1];
    g.fillStyle=css('--muted'); g.font='10px sans-serif'; g.fillText(l,4,10);
    Object.entries(set.sets).forEach(([name,ys],j)=>{ g.strokeStyle=SETCOL[j]; g.lineWidth=1.6; g.beginPath(); ys.forEach((y,n)=>{ const X=pad+(w-2*pad)*(xs[n]-lo)/(hi-lo), Y=h-pad-(h-2*pad-8)*y; n?g.lineTo(X,Y):g.moveTo(X,Y); }); g.stroke();
      const deg=(out.degrees[k]||{})[name]; g.fillStyle=SETCOL[j]; g.fillText(`${name} ${deg!==undefined?fmt(deg,2):''}`,pad+j*((w-2*pad)/3.2),h-3); });
    const v=+$('fz_'+k).value; const X=pad+(w-2*pad)*(v-lo)/(hi-lo); g.strokeStyle='#fff'; g.setLineDash([3,3]); g.beginPath(); g.moveTo(X,14); g.lineTo(X,h-pad); g.stroke(); g.setLineDash([]); });
  const active=new Map(out.rules.map(r=>[r.id,r.strength]));
  $('fzCount').textContent=`${out.rules.length} de ${out.all_rules.length} activas`;
  $('fzRules').innerHTML=out.all_rules.map(r=>`<div class="rule ${active.has(r.id)?'act':''}"><b>R${r.id}</b><span>SI ${r.antecedents.map(([v,s])=>`${VARES[v]||v} es <i>${s}</i>`).join(' Y ')} → velocidad <i>${r.consequent}</i><br><span class="small">${r.description}</span></span><b>${active.has(r.id)?fmt(active.get(r.id),2):'0'}</b></div>`).join('');
  const sp=META.fuzzy_sets.speed; const c=$('fzAgg'); const [g,w,h]=ctx2d(c); const pad=16;
  Object.entries(sp.sets).forEach(([name,ys],j)=>{ g.strokeStyle=SETCOL[j]+'88'; g.lineWidth=1; g.beginPath(); ys.forEach((y,n)=>{ const X=pad+(w-2*pad)*sp.x[n], Y=h-pad-(h-2*pad-10)*y; n?g.lineTo(X,Y):g.moveTo(X,Y); }); g.stroke(); g.fillStyle=SETCOL[j]; g.font='10px sans-serif'; g.fillText(name,pad+j*((w-2*pad)/4),12); });
  const cur=out.curve; if(cur.length){ g.fillStyle='rgba(255,181,71,.35)'; g.strokeStyle=css('--warn'); g.lineWidth=2; g.beginPath(); g.moveTo(pad,h-pad);
    cur.forEach((y,n)=>{ g.lineTo(pad+(w-2*pad)*n/(cur.length-1), h-pad-(h-2*pad-10)*y); }); g.lineTo(w-pad,h-pad); g.closePath(); g.fill(); g.stroke(); }
  const X=pad+(w-2*pad)*out.centroid; g.strokeStyle='#fff'; g.lineWidth=2; g.beginPath(); g.moveTo(X,14); g.lineTo(X,h-pad); g.stroke(); g.fillStyle='#fff'; g.fillText('centroide '+fmt(out.centroid,3),Math.min(X+4,w-90),26);
}
$('bFzCur').addEventListener('click',()=>{ if(!S||!S.fuzzy){ msg('No hay inferencia difusa actual en esta política.',true); return; } const I=S.fuzzy.inputs; $('fz_distance').value=I.distance; $('fz_battery').value=I.battery; $('fz_slope').value=I.slope; $('fz_temperature').value=I.temperature; fzEval(); });

// ---------------------------------------------------------------- GA lab
const WNAMES={science:'w1 ciencia',distance:'w2 distancia',energy:'w3 energía',predicted_risk:'w4 riesgo',time:'w5 tiempo'};
function buildWeights(){ const w=(S&&S.ga.weights)||{science:3,distance:1,energy:1,predicted_risk:2,time:1};
  $('gaWeights').innerHTML=Object.entries(WNAMES).map(([k,l])=>`<div class="slider"><label>${l}</label><input type="range" id="w_${k}" min="0" max="6" step=".1" value="${w[k]??1}"><span id="wv_${k}">${fmt(w[k]??1,1)}</span></div>`).join('');
  Object.keys(WNAMES).forEach(k=>$('w_'+k).addEventListener('input',()=>$('wv_'+k).textContent=(+$('w_'+k).value).toFixed(1))); }
$('bWeights').addEventListener('click',async()=>{ const weights={}; Object.keys(WNAMES).forEach(k=>weights[k]=+$('w_'+k).value); const out=await call('configure',{...formConfig(),weights}); if(out){ msg('Pesos aplicados; misión reiniciada con la misma semilla.'); } });
let gaRunSel=null;
function runArg(){ return gaRunSel===null?{}:{run:gaRunSel}; }
async function gaGen(){ const g=+$('fGen').value; const out=await api('ga_generation',{generation:g,...runArg()}); if(out.error){ $('gaChrom').innerHTML=`<div class="small">${out.error}</div>`; return; }
  $('gaGenLbl').textContent=`${out.generation} / ${out.total-1}`;
  $('gaOps').innerHTML=`Operadores hacia esta generación: torneos ${out.tournaments} · cruces ${out.crossovers} · genes mutados ${out.mutations} · élite 2 · mejor ${fmt(out.best,3)}`;
  $('gaChrom').innerHTML='<table class="mt"><tr><th>#</th><th>cromosoma (puntos de paso)</th><th>fitness</th><th>factible</th></tr>'+out.rows.map((r,i)=>`<tr><td>${i}</td><td style="text-align:left">${r.chromosome.map(p=>`(${p})`).join(' → ')}</td><td>${fmt(r.fitness,3)}</td><td>${r.feasible?'<span class="pill ok">sí</span>':`<span class="pill bad">${r.violations.join(',')}</span>`}</td></tr>`).join('')+'</table>';
}
$('fGen').addEventListener('input',()=>{ clearTimeout(window._gt); window._gt=setTimeout(gaGen,80); });
$('fRun').addEventListener('change',()=>{ gaRunSel=+$('fRun').value; $('fGen').value=5; renderGa(); });
async function renderGa(){ if(!S) return; const runs=S.ga_runs||[];
  $('fRun').innerHTML=runs.length?runs.map(r=>`<option value="${r.index}">#${r.index} · paso ${r.step} · ${r.reason}</option>`).join(''):'<option>sin ejecuciones del AG</option>';
  if(gaRunSel!==null && gaRunSel>=runs.length) gaRunSel=null;
  $('fRun').value=gaRunSel===null?Math.max(0,runs.length-1):gaRunSel;
  let G=S.ga; if(runs.length){ const out=await api('ga_run',runArg()); if(!out.error) G=out.ga; }
  const n=(G.history||[]).length;
  $('fGen').max=Math.max(0,n-1); if(+$('fGen').value>n-1) $('fGen').value=Math.max(0,n-1);
  lineChart($('gaChart'),[{name:'mejor',data:G.history.map(h=>h.best),color:css('--violet')},{name:'mejor factible',data:G.history.map(h=>h.best_feasible),color:css('--accent')}],{digits:2});
  $('gaInfo').innerHTML=G.mode==='ag'?`Población 30 · ${G.generations_run} generaciones${G.stopped_early?' (parada temprana: 10 sin mejora)':''} · torneo 3 · elitismo 2 · semilla controlada · ${G.message}`:`<span class="r-warn">${G.message||'La política actual no usa el AG.'}</span>`;
  const rej=new Map((G.rejected||[]).map(r=>[r.index,r]));
  $('gaCand').innerHTML=G.candidates.length?'<table class="mt"><tr><th>#</th><th>estado</th><th>fitness</th><th>ciencia</th><th>distancia</th><th>energía</th><th>riesgo</th><th>tiempo</th><th>penalización</th><th>orden científico</th><th>celdas</th></tr>'+
    G.candidates.map(c=>`<tr style="${c.index===G.chosen?'background:rgba(53,208,127,.12)':''}"><td>${c.index}</td><td>${c.index===G.chosen?'<span class="pill ok">ELEGIDA</span>':rej.has(c.index)?`<span class="pill bad">vetada: ${rej.get(c.index).reason}</span>`:'alternativa'}</td><td>${fmt(c.fitness,3)}</td>${['science','distance','energy','predicted_risk','time'].map(k=>`<td>${fmt(c.components[k],3)}</td>`).join('')}<td>${fmt(c.penalty,2)}</td><td style="text-align:left">${c.science_order.map(p=>`(${p})`).join(' → ')||'—'}</td><td>${c.distance}</td></tr>`).join('')+'</table>'
    :'<div class="small">Sin candidatas (la política actual no usa AG o aún no se planificó).</div>';
  gaGen();
}

// ---------------------------------------------------------------- comparison
$('bCompare').addEventListener('click',async()=>{ running=false; syncRun(); msg('Ejecutando comparación… (las 6 políticas con las mismas semillas)');
  $('bCompare').disabled=true; const out=await api('compare',{seeds:+$('fSeeds').value,scenario:$('fScenario').value}); $('bCompare').disabled=false;
  if(out.error){ msg(out.error,true); return; } if(S) S.comparison=out; renderCompare(); msg('Comparación lista: '+out.label+', '+out.seeds+' semillas por política.'); });
function renderCompare(){ const C=S&&S.comparison; if(!C) return;
  const cols=[['success','éxito',2],['collisions','colisiones',2],['safety_violations','violaciones',2],['cliffs_avoided','barrancos evitados',2],['hazards_avoided','peligros RNA',2],['battery_left','energía rest.',1],['distance','distancia',1],['science','ciencia',2],['replans','replan.',1],['decision_ms','ms/decisión',2]];
  $('cmpTable').innerHTML=`<div class="small" style="margin-bottom:4px">${C.label} · n=${C.seeds} semillas por política</div><table class="mt"><tr><th>política</th>${cols.map(c=>`<th>${c[1]}</th>`).join('')}</tr>`+
    C.summary.map(r=>`<tr><td style="text-align:left">${r.label}</td>${cols.map(([k,,d])=>`<td>${fmt(r[k].media,d)} ± ${fmt(r[k].desv,d)}<br><span class="small">(${fmt(r[k].mediana,d)})</span></td>`).join('')}</tr>`).join('')+'</table>';
  const c=$('cmpChart'); const [g,w,h]=ctx2d(c); const n=C.summary.length; const bw=(w-40)/n; const pad=18;
  C.summary.forEach((r,i)=>{ const x=30+i*bw; const s=r.success.media, col=r.collisions.media;
    g.fillStyle=css('--ok'); g.fillRect(x+bw*.15,h-pad-(h-2*pad)*s,bw*.32,(h-2*pad)*s);
    const cc=Math.min(1,col/3); g.fillStyle=css('--bad'); g.fillRect(x+bw*.52,h-pad-(h-2*pad)*cc,bw*.32,(h-2*pad)*cc);
    g.fillStyle=css('--muted'); g.font='10px sans-serif'; g.textAlign='center'; g.fillText(r.policy,x+bw/2,h-4); g.fillText(fmt(s*100,0)+'%',x+bw*.31,h-pad-(h-2*pad)*s-3); g.fillText(fmt(col,1),x+bw*.68,h-pad-(h-2*pad)*cc-3); });
  g.textAlign='left'; g.fillStyle=css('--ok'); g.fillRect(30,2,8,8); g.fillStyle=css('--muted'); g.fillText('éxito %',42,10); g.fillStyle=css('--bad'); g.fillRect(100,2,8,8); g.fillStyle=css('--muted'); g.fillText('colisiones (escala 0–3)',112,10);
}

// ---------------------------------------------------------------- controls
function formConfig(){ return {planet:$('fPlanet').value, mode:$('fMode').value, seed:Number($('fSeed').value), difficulty:$('fDiff').value, battery:Number($('fBattery').value), policy:$('fPolicy').value}; }
function syncRun(){ const b=$('bRun'); b.textContent=running?'⏸ Pausa':'▶ Iniciar'; b.className='btn '+(running?'':'on'); }
async function loop(){ if(!running) return; const n=+$('fSpeed').value; await call('step',{n},true); if(running && S && !S.status.finished) setTimeout(loop, 260); else { running=false; syncRun(); } }
$('bRun').addEventListener('click',()=>{ if(S&&S.status.finished){ msg('La misión terminó. Reinicie o cambie la configuración.'); return; } running=!running; syncRun(); if(running){ msg('Simulación en curso.'); loop(); } else msg('Pausada.'); });
$('bStep').addEventListener('click',()=>{ running=false; syncRun(); call('step',{n:1},true); });
$('bReset').addEventListener('click',async()=>{ running=false; syncRun(); if(await call('reset',{},true)) msg('Misión reiniciada con la misma semilla.'); });
$('bApply').addEventListener('click',async()=>{ running=false; syncRun(); const out=await call('configure',formConfig(),true); if(out) msg('Configuración válida aplicada. Nuevo mapa generado.'); });
$('fBattery').addEventListener('input',()=>$('lBat').textContent=$('fBattery').value);
$('fSpeed').addEventListener('input',()=>$('lSpeed').textContent=$('fSpeed').value);
$('fPlanet').addEventListener('change',async()=>{ const v=await api('validate',formConfig()); if(v.error){ msg(v.error,true); if(/sonda/.test(v.error)) { $('fMode').value='sonda'; msg(v.error+' Se propone modo sonda (aplicar para confirmar).',true);} } else msg('Configuración compatible. Presione Aplicar.'); });
$('fMode').addEventListener('change',async()=>{ const v=await api('validate',formConfig()); msg(v.error||'Configuración compatible. Presione Aplicar.',!!v.error); });
document.querySelectorAll('[data-export]').forEach(b=>b.addEventListener('click',async()=>{ const out=await api('export',{format:b.dataset.export}); if(out.error){ msg(out.error,true); return; }
  const blob=new Blob([out.content],{type:'text/plain;charset=utf-8'}); const a=document.createElement('a'); a.href=URL.createObjectURL(blob); a.download=out.filename; document.body.appendChild(a); a.click(); a.remove(); msg('Exportado '+out.filename); }));
document.querySelectorAll('.tab').forEach(t=>t.addEventListener('click',()=>{ document.querySelectorAll('.tab').forEach(x=>x.classList.toggle('active',x===t));
  view=t.dataset.view; document.querySelectorAll('.view').forEach(v=>v.classList.toggle('active',v.id==='v-'+view)); renderView(); if(view==='mission') render(); }));
function renderView(){ if(view==='rna'){ renderRnaStatic(); rnaEval(); } if(view==='fuzzy') fzEval(); if(view==='ga'){ buildWeights(); renderGa(); } if(view==='compare') renderCompare(); }
window.addEventListener('resize',()=>{ clearTimeout(window._rt); window._rt=setTimeout(()=>{ render(); if(view==='fuzzy') drawFz(); if(view==='compare') renderCompare(); },120); });

async function init(){
  META=await api('meta'); if(META.error){ msg(META.error,true); return; }
  $('fPlanet').innerHTML=META.planets.map(p=>`<option value="${p.name}">${p.name}${p.rocky?'':' (gigante)'}</option>`).join('');
  $('fPolicy').innerHTML=Object.entries(META.policies).map(([k,v])=>`<option value="${k}">${v}</option>`).join('');
  $('fScenario').innerHTML=Object.entries(META.scenarios).map(([k,v])=>`<option value="${k}">${v}</option>`).join('');
  $('events').innerHTML=Object.entries(META.events).map(([k,v])=>`<button class="btn" data-ev="${k}" title="${v}">${v}</button>`).join('')+
    '<button class="btn" data-ev="falla_sensor" data-sensor="todos" title="Fallan proximidad y LIDAR">Falla total navegación</button>';
  document.querySelectorAll('[data-ev]').forEach(b=>b.addEventListener('click',async()=>{ const args={kind:b.dataset.ev}; if(b.dataset.sensor) args.sensor=b.dataset.sensor;
    const out=await call('inject',args,true); if(out) msg('Evento inyectado: '+b.title+'. '+((out.status&&S.map.events.length)?'':'')); }));
  buildRnaSliders(); buildFzSliders();
  await call('state',{},true);
  $('fPlanet').value=S.config.planet; $('fMode').value=S.config.mode==='rover'?'rover':'sonda'; $('fSeed').value=S.config.seed; $('fDiff').value=S.config.difficulty; $('fBattery').value=S.config.battery; $('lBat').textContent=S.config.battery; $('fPolicy').value=S.config.policy;
}
if (window.ARES_BOOT) window.ARES_BOOT.then(init).catch(e=>msg('No se pudo iniciar el motor Python: '+e,true)); else init();
</script>
</body>
</html>


In [ ]:
import sys, importlib
sys.path.insert(0, 'ares_x/src')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from ares.api import Session
from ares.mission import HybridMission, shared_perception, POLICIES
from ares.fuzzy import FuzzyController
from ares.genetic import GeneticPlanner
from ares.experiments import run_experiments, summarize
from ares.dashboard import Dashboard
print('Motor ARES-X cargado.')

## 2. RNA — percepción (RF-04, CA-06)
Datos sintéticos etiquetados por el simulador (la verdad es la pertenencia de la celda a barranco/obstáculo, no la variable que recibe la red).
Partición **por mapa** en entrenamiento/validación/test para evitar fuga. MLP (32,16) ReLU con softmax de 3 clases.

In [ ]:
perception = shared_perception(0)
report = perception.report_
print(f"Entrenamiento: {report['training_samples']} muestras · {report['iterations']} iteraciones · {perception.train_seconds_:.2f} s")
print('Mapas por partición:', report['split_maps'])
for split in ('validation', 'test'):
    m = report[split]
    print(f"{split:>10}: exactitud {m['accuracy']:.3f} | baseline mayoría {m['majority_baseline']:.3f} | macro-F1 {m['macro_f1']:.3f} | recall barranco {m['barranco_recall']:.3f} | ECE {m['calibration_error']:.3f}")
fig, ax = plt.subplots(figsize=(4, 3.4))
cm = np.array(report['test']['confusion_matrix'])
ax.imshow(cm, cmap='Blues'); ax.set_xticks(range(3), report['test']['labels']); ax.set_yticks(range(3), report['test']['labels'])
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha='center', va='center')
ax.set_xlabel('predicho'); ax.set_ylabel('real'); ax.set_title('Matriz de confusión (test)'); plt.show()
sample = dict(distance=.05, slope=30, elevation_delta=-10, texture=.3, visibility=.9, vibration=1.5)
pred = perception.predict(sample)
print('Lectura con caída de 10 m →', pred['label'], {k: round(v, 3) for k, v in pred['probabilities'].items()})

## 3. Lógica difusa — control (RF-05, RF-06, CA-05)
Distancia (cerca/media/lejos), batería (crítica/baja/alta), pendiente (plana/media/fuerte) y temperatura; 10 reglas Mamdani min–max,
desfusificación por centroide. Sin reglas activas → salida documentada 0 (parar).

In [ ]:
fz = FuzzyController()
for rule in fz.rules:
    print(f"R{rule['id']:>2}: {rule['description']}")
for distance in (10, 1):
    out = fz.infer(distance, 60, 5, 12)
    active = ', '.join(f"R{r['id']}={r['strength']:.2f}" for r in out['activated_rules'])
    print(f"distancia {distance:>2} m → velocidad {out['speed']:.3f} · reglas activas: {active}")
fig, axes = plt.subplots(1, 2, figsize=(10, 2.8))
xs = np.linspace(0, 20, 201)
for label, spec in fz.membership_sets['distance'].items():
    axes[0].plot(xs, [fz.membership(x, spec) for x in xs], label=label)
axes[0].axvline(10, ls='--', c='k'); axes[0].axvline(1, ls=':', c='r'); axes[0].set_title('Distancia a riesgo (m)'); axes[0].legend()
for distance, style in ((10, '-'), (1, ':')):
    out = fz.infer(distance, 60, 5, 12)
    axes[1].plot(out['universe'], out['aggregated_curve'], style, label=f"{distance} m → centroide {out['centroid']:.2f}")
axes[1].set_title('Agregación y centroide (velocidad)'); axes[1].legend(); plt.show()

## 4. Algoritmo genético — planificación (RF-07, CA-04)
Cromosoma: K=3 puntos de paso decodificados a una ruta contigua con reparación. Población 30, ≥30 generaciones, torneo, cruce, mutación,
elitismo 2, semilla controlada y parada temprana tras 10 generaciones sin mejora. Familias integradas: **laberinto** (factibilidad de ruta)
y **viajero** (orden de visita de sitios científicos).

In [ ]:
mission = HybridMission(seed=42)
mission.step()
ga = mission.ga
print(f"Generaciones: {ga['generations_run']} · parada temprana: {ga['stopped_early']} · pesos: {ga['weights']}")
for cand in ga['candidates']:
    print(f"#{cand['index']} fitness {cand['fitness']:+.3f} factible={cand['feasible']} componentes={cand['components']} orden={cand['science_order']}")
gen5 = mission.ga_generation(5, run=0)
print(f"Generación 5: torneos {gen5['tournaments']}, cruces {gen5['crossovers']}, genes mutados {gen5['mutations']}")
for row in gen5['rows'][:3]:
    print('  ', row['chromosome'], 'fitness', row['fitness'])
plt.figure(figsize=(6, 2.6))
plt.plot([h['best'] for h in ga['history']], label='mejor'); plt.plot([h['best_feasible'] for h in ga['history']], '--', label='mejor factible')
plt.xlabel('generación'); plt.ylabel('fitness'); plt.legend(); plt.title('Convergencia del AG'); plt.show()

## 5. Centro de control de misión (una sola pantalla)
Panel A mapa y rutas · B telemetría · C decisión explicable por algoritmo · D traza filtrable/exportable · E controles y eventos.
Pestañas: Laboratorio RNA, Difuso, Genético y Comparación de políticas. Iniciar, pausar, paso a paso, reiniciar, velocidad,
inyectar eventos y editar el terreno real (clic en el mapa).

In [ ]:
dashboard = Dashboard(seed=42, planet='Marte', height=880)
dashboard.show()

## 6. Experimentos: 20 semillas, ablaciones y baselines (RF-14, RF-15)
Mismas semillas y condiciones para cada política: control (ruta más corta sin riesgos), AG puro con riesgos fijos, híbrido completo y
ablaciones sin RNA / sin difuso / sin AG. Se reporta media, mediana y dispersión.

In [ ]:
import time, json
results = {}
for scenario in ('estandar', 'eventos'):
    t0 = time.time()
    frame = run_experiments(seeds=20, scenario=scenario)
    results[scenario] = frame
    print(f"Escenario {scenario}: {len(frame)} misiones en {time.time() - t0:.1f} s")
    display(summarize(frame).round(3))
frame = pd.concat(results.values())
frame.to_csv('ares_x_experimentos.csv', index=False)
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, metric in zip(axes, ('success', 'collisions', 'battery_left')):
    data = results['eventos'].groupby('policy')[metric].apply(list)
    ax.boxplot([data[p] for p in POLICIES], tick_labels=list(POLICIES)) if metric != 'success' else ax.bar(list(POLICIES), [np.mean(data[p]) for p in POLICIES])
    ax.set_title(metric + ' (eventos)'); ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
# Exportación de una misión completa (CSV de pasos + JSON con configuración, motivos y trazas) (RF-12)
demo = HybridMission(seed=42)
demo.step(); demo.inject('barranco'); demo.run()
open('ares_x_mision_semilla42.csv', 'w', encoding='utf-8').write(demo.export_csv())
json.dump(demo.export_json(), open('ares_x_mision_semilla42.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print(demo.summary())
try:
    from google.colab import files  # descarga opcional en Colab
    print('Use files.download("ares_x_mision_semilla42.json") para descargar.')
except ImportError:
    pass

## 7. Pruebas automáticas
Casos de borde (sección 12) y casos de aceptación CA-01…CA-08.

In [ ]:
%%writefile ares_x/tests/test_h1_physics.py
import sys
from pathlib import Path

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from ares import PlanetProfile, World, SensorSuite, Mission


def test_empty_battery():
    mission = Mission(World(seed=3), battery=0)
    start = mission.position
    for _ in range(3):
        mission.step((0, 1))
    assert mission.position == start
    assert mission.battery == 0
    assert mission.state.value == 'FRACASO'


@pytest.mark.parametrize('distance', [float('nan'), float('inf'), -float('inf'), 0, None])
def test_invalid_distance(distance):
    assert not SensorSuite.validate({'distance': distance})


def test_new_cliff_stops():
    mission = Mission(World(seed=2))
    mission.inject_event('barranco', position=(0, 1))
    mission.step((0, 1))
    assert mission.position == (0, 0)
    assert mission.state.value == 'REPLANIFICACION'
    assert (0, 1) in mission.world.cliffs


def test_seed_trace():
    def run():
        mission = Mission(World(seed=42))
        for target in [(0, 1), (1, 0), (0, 0)]:
            mission.step(target)
        return mission.logger.trace
    assert run() == run()


def test_jupiter_wheels():
    with pytest.raises(ValueError, match='sonda'):
        PlanetProfile.for_body('Júpiter', mode='ruedas')


def test_no_science():
    mission = Mission(World(seed=4, science_count=0))
    assert mission.science_progress == 1.0
    mission.step()
    assert mission.state.value == 'EXITO'


@pytest.mark.parametrize('body', ['Mercurio', 'Venus', 'Tierra', 'Marte', 'Júpiter', 'Saturno', 'Urano', 'Neptuno'])
def test_planet_modes(body):
    assert PlanetProfile.for_body(body, mode='sonda').mode == 'probe'
    if body in ['Júpiter', 'Saturno', 'Urano', 'Neptuno']:
        with pytest.raises(ValueError, match='suelo sólido'):
            PlanetProfile.for_body(body)
    else:
        assert PlanetProfile.for_body(body).rocky


def test_map_generation_and_manual_obstacle():
    import numpy as np
    a, b = World(seed=19), World(seed=19)
    assert (a.height, a.width) == (12, 16)
    assert np.array_equal(a.elevation, b.elevation)
    assert a.science_sites == b.science_sites
    assert not (a.obstacles & a.science_sites)
    a.add_obstacle((1, 1))
    assert not a.traversable((1, 1))
    with pytest.raises(ValueError):
        a.add_obstacle((-1, 0))


@pytest.mark.parametrize('failure', ['nan', 'saturation', 'missing'])
def test_sensor_failure_stops_and_logs(failure):
    mission = Mission(World(seed=3))
    mission.world.obstacles.discard((0, 1))
    mission.inject_event('falla_sensor', failure=failure)
    row = mission.step((0, 1))
    assert mission.position == (0, 0)
    assert row['state'] == 'MODO_SEGURO'
    assert row['sensors']['distance'] is None


def test_link_loss_local_operation_and_csv():
    import csv
    import io
    mission = Mission(World(seed=3))
    mission.world.obstacles.discard((0, 1))
    mission.inject_event('perdida_enlace')
    row = mission.step((0, 1))
    assert mission.position == (0, 1)
    assert not row['sensors']['link_available']
    rows = list(csv.DictReader(io.StringIO(mission.logger.export_csv())))
    assert len(rows) == 1
    assert rows[0]['action'] == 'avanzar'
    row['battery'] = -100
    assert mission.logger.trace[0]['battery'] >= 0


def test_events_and_narrative():
    from ares import EVENTS
    mission = Mission(World(seed=5), profile=PlanetProfile.for_body('Júpiter', 'probe'))
    for kind in EVENTS:
        mission.inject_event(kind, position=(1, 1))
    assert len(mission.pending_events) == len(EVENTS)
    assert mission.world.cliffs == {(1, 1)}
    assert mission.health == 80
    assert not mission.daylight
    assert mission.visibility == .2
    assert 'no confirma vida' in next(e['message'] for e in mission.pending_events if e['kind'] == 'biosenal_anomala')
    assert 'ficticio' in next(e['message'] for e in mission.pending_events if e['kind'] == 'alienigena')


def test_invalid_configuration():
    for kwargs in [{'battery': float('nan')}, {'battery': -1}, {'step_duration': 0}]:
        with pytest.raises(ValueError):
            Mission(World(), **kwargs)
    for kwargs in [{'width': 0}, {'difficulty': 'impossible'}, {'science_count': -1}]:
        with pytest.raises(ValueError):
            World(**kwargs)


def test_cliff_altimetry_and_insufficient_energy():
    mission = Mission(World(seed=1), battery=.5)
    mission.world.obstacles.discard((0, 1))
    mission.world.elevation[0, 1] = mission.world.elevation[0, 0] - 10
    assert mission.step((0, 1))['reason'] == 'BARRANCO_CONFIRMADO'
    mission.world.cliffs.clear()
    mission.world.elevation[0, 1] = 0
    assert mission.step((0, 1))['reason'] == 'ENERGIA_INSUFICIENTE'
    assert mission.battery == .5


def test_navigation_invalid_altimetry():
    mission = Mission(World(seed=1))
    mission.world.obstacles.discard((0, 1))
    mission.inject_event('falla_sensor', sensor='elevation_delta')
    assert mission.step((0, 1))['state'] == 'MODO_SEGURO'
    assert mission.position == (0, 0)


def test_thermal_limit():
    mission = Mission(World(seed=1), profile=PlanetProfile.for_body('Venus'), thermal_limit=100)
    assert mission.step()['reason'] == 'LIMITE_TERMICO'


@pytest.mark.parametrize('body', ['Júpiter', 'Saturno', 'Urano', 'Neptuno'])
def test_all_giants_require_probe(body):
    with pytest.raises(ValueError, match='suelo sólido.*sonda'):
        PlanetProfile.for_body(body, 'ruedas')
    assert PlanetProfile.for_body(body, 'sonda').mode == 'probe'


def test_invalid_altimeter_stops_before_movement():
    world = World(seed=1)
    world.obstacles.clear()
    sensors = SensorSuite(noise=0)
    sensors.fail('elevation_delta', 'nan')
    mission = Mission(world, sensors=sensors)
    row = mission.step((0, 1))
    assert row['action'] == 'detener'
    assert mission.position == (0, 0)
    assert row['state'] == 'MODO_SEGURO'


def test_empty_mission_does_not_override_cliff_stop():
    mission = Mission(World(science_count=0))
    mission.inject_event('barranco', position=(0, 1))
    assert mission.step((0, 1))['state'] == 'REPLANIFICACION'


def test_low_energy_allows_reachable_return():
    world = World(seed=0)
    world.obstacles.clear()
    mission = Mission(world, battery=5)
    mission.position = (0, 1)
    row = mission.step((0, 0))
    assert row['action'] == 'avanzar'
    assert mission.position == (0, 0)
    assert mission.battery == 4


def test_dust_reduces_solar_power():
    mission = Mission(World())
    before = mission.step()['sensors']['solar_power']
    mission.inject_event('tormenta_polvo')
    assert mission.step()['sensors']['solar_power'] < before


def test_invalid_target_is_safe():
    mission = Mission(World())
    row = mission.step((float('nan'), 1))
    assert row['action'] == 'detener'
    assert row['state'] == 'REPLANIFICACION'


def test_csv_and_snapshots():
    import csv
    import io
    from ares import MissionLogger
    logger = MissionLogger()
    readings = {'distance': float('nan')}
    row = logger.record(sensors=readings, reason='ESPERA')
    readings['distance'] = 10
    row['reason'] = 'changed'
    assert logger.trace[0]['sensors']['distance'] is None
    assert list(csv.DictReader(io.StringIO(logger.export_csv())))[0]['reason'] == 'ESPERA'


def test_energy_threshold_preserves_reachable_return():
    world = World(width=30, height=1)
    world.obstacles.clear()
    world.elevation[:] = 0
    mission = Mission(world, battery=20)
    for column in range(1, 11):
        mission.step((0, column))
    assert mission.battery == 10
    row = mission.step((0, 11))
    assert row['action'] == 'detener'
    assert mission.position == (0, 10)
    assert mission.battery == 10
    assert mission.step((0, 9))['action'] == 'avanzar'


In [ ]:
%%writefile ares_x/tests/test_h2_fuzzy.py
"""Standalone Mamdani inference and diagnostic contract."""
import sys
from pathlib import Path

import numpy as np
import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from ares.fuzzy import FuzzyController, triangular, trapezoidal


def test_sets_and_documented_rules():
    controller = FuzzyController()
    assert len(controller.rules) >= 8
    assert all(len(sets) >= 3 for sets in controller.membership_sets.values())
    assert all(rule['description'] for rule in controller.rules)


def test_distance_changes_crisp_and_centroid():
    controller = FuzzyController()
    near = controller.infer(distance=1, battery=80, slope=0, temperature=20)
    medium = controller.infer(distance=10, battery=80, slope=0, temperature=20)
    assert near['speed'] < medium['speed']
    assert near['caution'] > medium['caution']
    assert len(medium['activated_rules']) > 1
    x = np.asarray(medium['universe'])
    curve = np.asarray(medium['aggregated_curve'])
    assert medium['centroid'] == pytest.approx(np.trapezoid(x * curve, x) / np.trapezoid(curve, x))
    assert medium['speed'] == medium['centroid']


def test_no_rule_fallback():
    result = FuzzyController(rules=[]).infer(10, 80, 0, 20)
    assert result['speed'] == 0
    assert result['reason'] == 'SIN_REGLAS_ACTIVAS'
    assert not any(result['aggregated_curve'])


@pytest.mark.parametrize('values', [(float('nan'),80,0,20), (float('inf'),80,0,20),
                                   (-1,80,0,20), (10,101,0,20), (10,80,46,20)])
def test_invalid_inputs_safe(values):
    result = FuzzyController().infer(*values)
    assert result['speed'] == 0
    assert result['reason'] == 'ENTRADA_INVALIDA'


def test_membership_shoulders_and_triangle():
    assert trapezoidal(0, 0, 0, 2, 4) == 1
    assert trapezoidal(4, 0, 2, 4, 4) == 1
    assert triangular(2, 0, 2, 4) == 1
    assert triangular(1, 0, 2, 4) == .5


def test_temperature_and_boundaries():
    controller = FuzzyController()
    normal = controller.infer(20, 100, 0, 20)
    hot = controller.infer(20, 100, 0, 450)
    assert hot['speed'] < normal['speed']
    for distance in (0, 20):
        result = controller.infer(distance, 0, 45, -200)
        assert 0 <= result['speed'] <= 1
        assert 0 <= result['caution'] <= 1


In [ ]:
%%writefile ares_x/tests/test_h2_genetic.py
"""Public genetic-planner behavior and observable evolutionary operations."""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))

import numpy as np
import pytest

from ares.genetic import GeneticPlanner
from ares.world import World


def flat(width=6, height=5):
    world = World(width=width, height=height, science_count=0)
    world.obstacles.clear()
    world.cliffs.clear()
    world.science_sites = {(0, 3), (3, 3)}
    return world


def test_population_candidates_history_and_determinism():
    world = flat()
    first = GeneticPlanner(seed=7).plan(world, (0, 0), (4, 5))
    second = GeneticPlanner(seed=7).plan(world, (0, 0), (4, 5))
    assert first == second
    assert first['feasible'] and len(first['candidates']) >= 3
    assert len({tuple(c['route']) for c in first['candidates']}) >= 3
    assert len(first['history'][0]['chromosomes']) >= 30
    for candidate in first['candidates']:
        assert set(candidate['components']) == {'science', 'distance', 'energy', 'predicted_risk', 'time'}
        assert all(0 <= value <= 1 for value in candidate['components'].values())
        assert all(abs(a[0]-b[0]) + abs(a[1]-b[1]) == 1 for a, b in zip(candidate['route'], candidate['route'][1:]))
    assert any(h['crossovers'] > 0 and h['mutations'] > 0 for h in first['history'][1:])
    assert any(h['tournaments'] > 0 for h in first['history'][1:])


def test_elitism_preserves_best_chromosome():
    result = GeneticPlanner(seed=4, early_stop=False).plan(flat(), (0, 0), (4, 5))
    history = result['history']
    assert len(history) == 31
    for previous, current in zip(history, history[1:]):
        assert previous['best_chromosome'] in current['chromosomes']
        assert current['best_fitness'] >= previous['best_fitness']


def test_early_stop_retains_best_feasible():
    result = GeneticPlanner(seed=4, mutation_rate=0, crossover_rate=0).plan(flat(), (0, 0), (4, 5))
    assert result['stopped_early'] and result['generations_run'] == 10
    assert result['feasible']
    assert result['fitness'] == max(h['best_feasible_fitness'] for h in result['history'])


def test_invalid_mutation_is_repaired_and_penalty_is_inspectable():
    world = flat()
    planner = GeneticPlanner(seed=1, waypoints=3)
    invalid = [(900, -40), (float('nan'), 0), None]
    repaired = planner.repair(invalid, world)
    assert len(repaired) == 3 and all(world.traversable(p) for p in repaired)
    mutated = planner.mutate(invalid, world)
    assert all(world.traversable(p) for p in mutated)
    candidate = planner.evaluate(invalid, world, (0, 0), (4, 5))
    assert candidate['repaired'] and candidate['feasible']


@pytest.mark.parametrize('kind', ['obstacle', 'cliff'])
def test_maze_avoids_hard_barriers(kind):
    world = flat()
    barrier = {(r, 2) for r in range(4)}
    getattr(world, 'obstacles' if kind == 'obstacle' else 'cliffs').update(barrier)
    result = GeneticPlanner(seed=2).plan(world, (0, 0), (0, 5))
    assert result['feasible'] and not (set(result['route']) & barrier)
    assert (4, 2) in result['route']


@pytest.mark.parametrize('kwargs', [{'battery': 0}, {'battery': 1}, {'temperature': 81}, {'thermal_limit': 10}])
def test_energy_thermal_and_safe_return(kwargs):
    result = GeneticPlanner(seed=3).plan(flat(), (0, 0), (4, 5), **kwargs)
    assert not result['feasible'] and result['route'] == []
    assert 'sin solución' in result['message'].lower()
    assert all(c['penalty'] > 0 for c in result['candidates'])


def test_safe_return_is_included_not_only_outbound_energy():
    world = flat()
    planner = GeneticPlanner(seed=1, waypoints=1)
    candidate = planner.evaluate([(0, 0)], world, (0, 0), (0, 5), battery=7)
    assert candidate['energy_required'] == 10
    assert not candidate['feasible'] and 'battery' in candidate['violations']


def test_unreachable_and_bounds_are_explicit():
    world = flat()
    world.obstacles.update({(r, 2) for r in range(5)})
    planner = GeneticPlanner(seed=1)
    result = planner.plan(world, (0, 0), (0, 5))
    assert not result['feasible'] and 'sin solución' in result['message'].lower()
    assert not planner.plan(world, (-1, 0), (0, 5))['feasible']


def test_risk_weights_and_scientific_visit_order():
    world = flat()
    planner = GeneticPlanner(waypoints=2, weights={'science': 5, 'distance': 1, 'energy': 1, 'predicted_risk': 10, 'time': 1})
    world.science_sites = {(3, 0), (0, 3)}
    risk = np.zeros((5, 6))
    risk[0, 3] = 1
    candidate = planner.evaluate([(3, 0), (0, 3)], world, (0, 0), (0, 5), estimated_risk=risk)
    assert candidate['science_order'] == [(3, 0), (0, 3)]
    assert candidate['components']['science'] == 1
    without_risk = planner.evaluate([(3, 0), (0, 3)], world, (0, 0), (0, 5))
    assert candidate['fitness'] < without_risk['fitness']
    world.science_sites.clear()
    assert planner.evaluate([(0, 0)] * 2, world, (0, 0), (0, 5))['components']['science'] == 0


@pytest.mark.parametrize('kwargs', [{'population_size': 29}, {'generations': 29}, {'mutation_rate': float('nan')}, {'weights': {'science': -1}}])
def test_invalid_configuration(kwargs):
    with pytest.raises(ValueError):
        GeneticPlanner(**kwargs)


def test_disconnected_waypoint_is_repaired_without_losing_reachable_goal():
    world = flat()
    world.obstacles.update({(r, 2) for r in range(5)})
    candidate = GeneticPlanner(waypoints=1).evaluate([(0, 5)], world, (0, 0), (4, 0))
    assert candidate['feasible'] and candidate['repaired']
    assert candidate['chromosome'] == [(0, 0)]
    assert candidate['route'][-1] == (4, 0)


def test_physical_cliff_edges_rejected_outbound_and_on_safe_return():
    world = flat(width=3, height=1)
    world.science_sites.clear()
    world.elevation[:] = [[0, -10, 0]]
    result = GeneticPlanner(seed=2).plan(world, (0, 0), (0, 2))
    assert not result['feasible'] and result['route'] == []
    assert 'sin solución' in result['message'].lower()
    assert all(not c['feasible'] and c['penalty'] > 0 for c in result['candidates'])

    # Uphill travel is permitted, but its return edge is a forbidden descent.
    world.elevation[:] = [[0, 10, 0]]
    candidate = GeneticPlanner(waypoints=1).evaluate([(0, 0)], world, (0, 0), (0, 1))
    assert candidate['route'] == [(0, 0), (0, 1)]
    assert not candidate['feasible'] and candidate['energy_required'] == float('inf')
    assert 'battery' in candidate['violations']


In [ ]:
%%writefile ares_x/tests/test_h2_neural.py
"""Leakage, reproducibility and inspectable neural perception contracts."""
import numpy as np
import pytest
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))

from ares.neural import FEATURES, LABELS, NeuralPerception, generate_training_data


@pytest.fixture(scope='module')
def fitted():
    model = NeuralPerception(seed=17)
    model.train(map_count=12, samples_per_map=100)
    return model


def test_synthetic_truth_and_group_separation(fitted):
    data = generate_training_data(seed=17, map_count=12, samples_per_map=100)
    assert set(data['y']) == set(LABELS)
    assert data['X'].shape == (1200, len(FEATURES))
    assert np.isfinite(data['X']).all()
    assert 'cliff' not in FEATURES
    assert len(np.unique(data['X'][:, 0])) > 100
    splits = fitted.split_indices_
    groups = [set(data['groups'][splits[key]]) for key in ('train', 'validation', 'test')]
    assert not groups[0] & groups[1] and not groups[0] & groups[2] and not groups[1] & groups[2]
    assert set.union(*groups) == set(data['groups'])
    np.testing.assert_allclose(fitted.scaler.mean_, data['X'][splits['train']].mean(axis=0))
    assert fitted.scaler.n_samples_seen_ == len(splits['train'])
    assert not np.allclose(fitted.scaler.mean_, data['X'].mean(axis=0))
    assert fitted.model.hidden_layer_sizes == (32, 16)
    assert fitted.model.activation == 'relu'
    assert fitted.model.out_activation_ == 'softmax'
    assert fitted.model.early_stopping is False


def test_probability_trace_confidence_and_actual_network(fitted):
    row = dict(distance=9., slope=4., elevation_delta=1., texture=.2, visibility=.8, vibration=.1)
    result = fitted.predict(row)
    probabilities = list(result['probabilities'].values())
    assert list(result['probabilities']) == list(LABELS)
    assert np.isfinite(probabilities).all() and sum(probabilities) == pytest.approx(1.)
    assert result['confidence'] == max(probabilities)
    assert result['low_confidence'] == (result['confidence'] < fitted.confidence_threshold)
    assert 0 <= result['risk'] <= 1
    transformed = fitted.scaler.transform([[row[key] for key in FEATURES]])
    np.testing.assert_allclose(result['normalized_input'], transformed[0])
    direct = fitted.model.predict_proba(transformed)[0]
    for label, probability in zip(fitted.model.classes_, direct):
        assert result['probabilities'][label] == pytest.approx(probability)
    assert [layer['neurons'] for layer in result['activation_summaries']] == [32, 16]
    assert all(0 <= layer['active_fraction'] <= 1 for layer in result['activation_summaries'])
    old_threshold = fitted.confidence_threshold
    fitted.confidence_threshold = 1.
    try:
        assert fitted.predict(row)['low_confidence']
    finally:
        fitted.confidence_threshold = old_threshold


def test_metrics_are_heldout_and_have_calibration(fitted):
    for split in ('validation', 'test'):
        metrics = fitted.report_[split]
        assert np.asarray(metrics['confusion_matrix']).shape == (3, 3)
        assert np.asarray(metrics['confusion_matrix']).sum() == metrics['samples']
        for key in ('accuracy', 'majority_baseline', 'macro_f1', 'barranco_recall', 'calibration_error'):
            assert 0 <= metrics[key] <= 1
        assert metrics['macro_f1'] > .65
        assert metrics['accuracy'] > metrics['majority_baseline']
    assert fitted.operational_label == 'supuesto de simulación'


def test_deterministic_data_training_and_predictions(fitted):
    other = NeuralPerception(seed=17)
    other.train(map_count=12, samples_per_map=100)
    row = [8., 3., -1., .4, .8, .05]
    assert other.predict(row) == fitted.predict(row)
    assert other.report_ == fitted.report_


@pytest.mark.parametrize('value', [None, [], [1] * 5, [np.nan] * 6,
    [np.inf] * 6, [0., 3., 0., .5, 1., 0.], [21., 3., 0., .5, 1., 0.],
    [1e308] * 6, {}, {'distance': 4.}, ['x'] * 6])
def test_invalid_input_is_rejected(fitted, value):
    with pytest.raises(ValueError, match='Entrada|sensor'):
        fitted.predict(value)


def test_untrained_and_invalid_configuration():
    with pytest.raises(ValueError, match='entren'):
        NeuralPerception().predict([8., 3., -1., .4, .8, .05])
    for threshold in (np.nan, 0, 1.1, 'x'):
        with pytest.raises(ValueError, match='confianza'):
            NeuralPerception(confidence_threshold=threshold)
    with pytest.raises(ValueError, match='mapas'):
        generate_training_data(map_count=2)


def test_invalid_training_data_rejected():
    data = generate_training_data(seed=3, map_count=8, samples_per_map=40)
    data['X'][0, 0] = np.nan
    with pytest.raises(ValueError, match='Entrada|datos'):
        NeuralPerception().train(data)


In [ ]:
%%writefile ares_x/tests/test_h3_hybrid.py
"""H3: hybrid integration, acceptance cases CA-01..CA-08, causality and edge cases."""
import json
import math
import sys
from pathlib import Path

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))

from ares.api import Session, handle_json  # noqa: E402
from ares.mission import HybridMission, validate_config  # noqa: E402


def first_move_mission(**kw):
    mission = HybridMission(seed=42, **kw)
    mission.step()
    return mission


# ------------------------------------------------------------------ CA-01
def test_ca01_mars_reproducible_at_least_ten_steps_consistent_telemetry():
    mission = HybridMission(planet='Marte', seed=42)
    records = [mission.step() for _ in range(12)]
    assert [r['step'] for r in records] == list(range(1, 13))
    for prev, cur in zip(records, records[1:]):
        assert 0 <= cur['battery'] <= 100
        for a, b in zip([prev['position']] + cur['moved'], cur['moved']):
            assert abs(a[0] - b[0]) + abs(a[1] - b[1]) == 1  # contiguous moves only
    assert len(mission.path) > 10


def test_same_seed_and_configuration_gives_identical_trace():
    a = HybridMission(seed=7).run()
    b = HybridMission(seed=7).run()
    ta = HybridMission(seed=7)
    tb = HybridMission(seed=7)
    ta.run(), tb.run()
    assert a['steps'] == b['steps']
    assert json.dumps(ta.logger.trace, sort_keys=True) == json.dumps(tb.logger.trace, sort_keys=True)


# ------------------------------------------------------------------ CA-02
def test_ca02_cliff_in_front_stops_before_danger_and_replans():
    mission = first_move_mission()
    runs_before = len(mission.ga_runs)
    entry = mission.inject('barranco')
    cliff = tuple(entry['position'])
    record = mission.step()
    assert record['reason'] == 'R04_VETO_BARRANCO_CONFIRMADO'
    assert tuple(record['position']) != cliff
    assert record['rna'] is not None and 'barranco' in record['rna']['probabilities']
    assert record['fuzzy']['rules'], 'fuzzy rules must be visible'
    assert len(mission.ga_runs) == runs_before + 1
    assert cliff not in [tuple(c) for c in mission.route]
    mission.run()
    assert cliff not in mission.path
    assert mission.metrics['safety_violations'] == 0


# ------------------------------------------------------------------ CA-03
def test_ca03_low_battery_prioritizes_refuge_or_explains():
    mission = HybridMission(seed=42, battery=12)
    mission.run()
    reasons = {r['reason'] for r in mission.logger.trace}
    assert mission.finished
    assert min(r['battery'] for r in mission.logger.trace) >= 0
    assert reasons & {'R07_RETORNO_ENERGIA', 'R06_REFUGIO_INALCANZABLE', 'R15_MISION_COMPLETA'}
    assert mission.metrics['safety_violations'] == 0


def test_battery_one_percent_far_refuge_declared_unreachable():
    mission = HybridMission(seed=42, battery=100)
    for _ in range(12):
        mission.step()
    mission.battery = 1.
    record = mission.step()
    assert record['reason'] in ('R06_REFUGIO_INALCANZABLE', 'R07_RETORNO_ENERGIA', 'R09_RECARGA', 'R15_MISION_COMPLETA')
    if record['reason'] == 'R06_REFUGIO_INALCANZABLE':
        assert mission.finished and mission.state == 'MODO_SEGURO'


def test_battery_zero_never_moves_and_never_negative():
    mission = HybridMission(seed=3, battery=0)
    record = mission.step()
    assert record['moved'] == [] and record['battery'] == 0
    assert mission.state == 'FRACASO'


# ------------------------------------------------------------------ CA-04
def test_ca04_generation_five_shows_three_chromosomes_and_operators():
    session = Session(seed=42)
    session.dispatch('step', {'n': 1})
    out = session.dispatch('ga_generation', {'generation': 5, 'run': 0})
    assert 'error' not in out
    assert len(out['rows']) >= 3
    assert all('fitness' in row for row in out['rows'])
    assert out['crossovers'] + out['mutations'] > 0


# ------------------------------------------------------------------ CA-05
def test_ca05_distance_10_to_1_changes_fuzzy_output():
    session = Session()
    far = session.dispatch('fuzzy_eval', {'distance': 10, 'battery': 60, 'slope': 5})
    near = session.dispatch('fuzzy_eval', {'distance': 1, 'battery': 60, 'slope': 5})
    assert near['speed'] < far['speed']
    assert {r['id'] for r in near['rules']} != {r['id'] for r in far['rules']}
    assert near['degrees']['distance']['cerca'] == 1.0


# ------------------------------------------------------------------ CA-06
def test_ca06_changing_rna_features_updates_probabilities():
    session = Session()
    base = dict(distance=10, slope=2, elevation_delta=0, texture=.25, visibility=1, vibration=0)
    free = session.dispatch('rna_eval', base)
    drop = session.dispatch('rna_eval', dict(base, distance=.05, elevation_delta=-10, slope=30, vibration=1.5))
    assert free['label'] == 'libre'
    assert drop['probabilities']['barranco'] > free['probabilities']['barranco']
    state = session.dispatch('state')
    assert 0 <= state['nn']['test']['barranco_recall'] <= 1
    assert state['nn']['test']['confusion_matrix']


# ------------------------------------------------------------------ CA-07
def test_ca07_jupiter_wheels_blocked_with_explanation():
    session = Session()
    out = session.dispatch('configure', {'planet': 'Júpiter', 'mode': 'rover'})
    assert 'error' in out and 'sonda' in out['error']
    ok = session.dispatch('configure', {'planet': 'Júpiter', 'mode': 'sonda'})
    assert ok['planet']['mode'] == 'probe'
    session.dispatch('step', {'n': 5})


# ------------------------------------------------------------------ CA-08
def test_ca08_same_seed_full_vs_ablation_and_export():
    session = Session(seed=11)
    out = session.dispatch('compare', {'seeds': 2, 'scenario': 'eventos', 'policies': ['hibrido', 'sin_rna']})
    assert {row['policy'] for row in out['summary']} == {'hibrido', 'sin_rna'}
    for row in out['summary']:
        assert set(row['success']) == {'media', 'mediana', 'desv'}
    session.dispatch('step', {'n': 5})
    csv_text = session.dispatch('export', {'format': 'csv'})['content']
    assert csv_text.count('\n') >= 6
    exported = json.loads(session.dispatch('export', {'format': 'json'})['content'])
    assert exported['config']['seed'] == 11 and len(exported['trace']) == 5


# -------------------------------------------------------------- causality
def test_causality_rna_output_changes_decision(monkeypatch):
    normal = first_move_mission()
    assert normal.step()['reason'] == 'R14_AVANCE'
    forced = first_move_mission()
    real_predict = forced.perception.predict

    def fake(features):
        out = real_predict(features)
        out.update(label='barranco', confidence=.95, probabilities={'libre': .02, 'obstaculo': .03, 'barranco': .95})
        return out
    monkeypatch.setattr(forced.perception, 'predict', fake)
    assert forced.step()['reason'] == 'R05_RNA_PELIGRO'


def test_causality_fuzzy_output_changes_movement(monkeypatch):
    normal = first_move_mission()
    assert normal.step()['moved']
    forced = first_move_mission()
    real_infer = forced.fuzzy.infer
    monkeypatch.setattr(forced.fuzzy, 'infer', lambda *a: {**real_infer(*a), 'speed': .02})
    record = forced.step()
    assert record['reason'] == 'R12_DIFUSO_PARAR' and record['moved'] == []


def test_causality_ga_output_changes_route():
    default = HybridMission(seed=42)
    default.step()
    no_science = HybridMission(seed=42, weights={'science': 0, 'distance': 1, 'energy': 1, 'predicted_risk': 2, 'time': 1})
    no_science.step()
    assert default.planned_history[0] != no_science.planned_history[0]


# ---------------------------------------------------------------- edges
@pytest.mark.parametrize('bad', [{'battery': math.nan}, {'battery': 150}, {'seed': -1}, {'policy': 'magia'},
                                 {'weights': {'science': math.inf}}])
def test_impossible_configuration_rejected_with_readable_message(bad):
    with pytest.raises(ValueError):
        validate_config(bad)
    assert 'error' in json.loads(handle_json(Session(), 'configure', json.dumps(bad).replace('NaN', '"NaN"')))


def test_all_navigation_sensors_failing_never_moves_blind():
    mission = first_move_mission()
    position = mission.position
    mission.inject('falla_sensor', sensor='todos')
    record = mission.step()
    assert record['reason'] == 'R02_SENSORES_NAVEGACION_FALLIDOS'
    assert tuple(record['position']) == position


def test_single_sensor_failure_degrades_without_exception():
    mission = first_move_mission()
    mission.inject('falla_sensor', sensor='distance')
    record = mission.step()
    assert record['reason'] in ('R13_AVANCE_DEGRADADO', 'R12_DIFUSO_PARAR', 'R11_REPLANIFICAR')
    assert record['rna'] is None and record['rna_error']


def test_simultaneous_meteorite_and_low_battery_resolved_deterministically():
    outcomes = []
    for _ in range(2):
        mission = first_move_mission()
        mission.inject('meteorito')
        mission.inject('bateria_baja')
        outcomes.append([mission.step()['reason'] for _ in range(4)])
    assert outcomes[0] == outcomes[1]


def test_map_without_science_sites_closes_without_division_by_zero():
    mission = HybridMission(seed=5, science_count=0)
    summary = mission.run()
    assert summary['science'] == 1.0 and mission.finished


def test_no_route_reports_no_solution():
    mission = HybridMission(seed=42)
    for cell in ((0, 1), (1, 0), (1, 1)):
        mission.world.obstacles.add(cell)
        mission.belief.obstacles.add(cell)
    mission.run()
    assert mission.finished
    assert any(r['reason'] == 'R10_SIN_RUTA' for r in mission.logger.trace)
    assert mission.metrics['distance'] == 0


def test_ga_route_crossing_confirmed_cliff_is_vetoed():
    mission = HybridMission(seed=42)
    accepted, rejected = mission.supervisor.filter_routes(
        [{'route': [(0, 0), (0, 1)], 'feasible': True}, {'route': [(0, 0), (1, 0)], 'feasible': True}],
        known_cliffs={(0, 1)}, known_obstacles=set())
    assert accepted == [1] and rejected[0]['reason'] == 'cruza peligro confirmado'


def test_events_cover_all_kinds_without_exception():
    for kind in ('bateria_baja', 'dia', 'noche', 'sismo', 'meteorito', 'tormenta_polvo', 'corrosion',
                 'biosenal_anomala', 'alienigena', 'falla_sensor', 'perdida_enlace', 'viento', 'barranco'):
        mission = first_move_mission()
        mission.inject(kind)
        mission.step()
        mission.step()


def test_biosignal_never_claims_life():
    mission = first_move_mission()
    entry = mission.inject('biosenal_anomala')
    assert 'no confirma vida' in entry['message']


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest'])
result = subprocess.run([sys.executable, '-m', 'pytest', '-q', '--color=no', '-p', 'no:cacheprovider', 'ares_x/tests'], capture_output=True, text=True)
print(result.stdout[-2500:] or result.stderr[-2500:])

## 8. Conclusiones, límites y supuestos
- **Integración causal:** las pruebas de causalidad muestran que alterar solo la salida de la RNA, del difuso o del AG cambia la decisión o el
  movimiento; el supervisor nunca decide con una "función secreta".
- **Seguridad primero:** prohibiciones físicas (energía, sensores, barranco confirmado) preceden a las propuestas de IA; cada decisión tiene
  un código de motivo reproducible (R00–R15).
- **No omnisciencia:** el rover planifica sobre un mapa creído; barrancos y parte de los obstáculos están ocultos y se descubren con sensores.
- **Límites:** física simplificada en cuadrícula 16×12; consumos, temperaturas internas, umbrales y daños son *supuestos de simulación*,
  no especificaciones de ingeniería. La señal biológica es anómala y simulada: nunca confirma vida.
- **Gigantes gaseosos:** Júpiter, Saturno, Urano y Neptuno bloquean el modo ruedas y proponen la sonda atmosférica virtual.